# Notebook 09 — Le code du projet et ses concepts

*Projet Nikkei 225 · Applied ML & Maths/Stats*

Ce notebook rassemble **tout le code Python du projet** (`src/`, `app/` et `notebooks/06_annexe_news_prix/`), dans l'ordre où il sert. Chaque fichier est précédé de ce qu'il fait, des concepts qu'il mobilise et de la séance du cours qui les introduit (**S** = Applied ML, **L** = Maths & Stats).

> **Comment le lire.** Chaque cellule de code contient un fichier du dépôt, tel quel. C'est un notebook **de lecture : ne pas exécuter les cellules** (les modules s'importent entre eux et certains lancent de longs calculs). Pour lancer un module : `python -m src.<module>` depuis la racine du projet.

| | Partie | Contenu |
|:-:|:--|:--|
| **0** | [Vue d'ensemble](#partie-0) | Le projet en une page, le lexique des concepts |
| **1** | [Configuration et données](#partie-1) | Paramètres, table de modélisation, audit des fuites |
| **2** | [Modéliser, décider, tester](#partie-2) | Pipeline, Optuna, logistique à la main, calibration, test |
| **3** | [Expliquer et prédire](#partie-3) | Signal du mois, contributions SHAP |
| **4** | [Après le test](#partie-4) | Version Bloomberg 1999-2026, améliorations sous protocole figé |
| **5** | [Explorations](#partie-5) | Bancs d'essai, volatilité, signal de risque |
| **6** | [Annexe news × prix](#partie-6) | Backtests, modèle de Markov caché |
| **7** | [La plateforme](#partie-7) | API, tableau de bord, site |

<a id="partie-0"></a>
## Partie 0 · Vue d'ensemble

### Le projet en une page

**La question.** Chaque fin de mois, faut-il *acheter* le Nikkei 225, *rester neutre* ou le *vendre* pour le mois suivant ? Et faut-il couvrir le risque de change yen/euro ?

**La chaîne, de bout en bout.**

| Étape | Ce qui se passe | Où |
|:-:|:--|:--|
| 1 | **Collecte** : prix, taux, change, enquêtes (Tankan), news GDELT notées par FinBERT, puis Bloomberg comme référence. Chaque série est décalée à sa *date de publication*. | notebooks 00a, 00b |
| 2 | **Table de modélisation** : une ligne par mois, les cibles du mois suivant, un audit des fuites. | `src/data.py` |
| 3 | **Découpage chronologique** : entraînement jusqu'à fin 2021, validation 2022-2023, test 2024-2026 ouvert une seule fois. | `config.yaml` |
| 4 | **Trois modèles** (logistique, forêt aléatoire, gradient boosting) dans une même pipeline, réglés par Optuna avec une validation croisée temporelle. | `src/pipeline.py`, `src/tune.py` |
| 5 | **Calibration de Platt et règle de décision** par espérance de gain, figées avant le test. | `src/decision.py` |
| 6 | **Test unique**, puis explicabilité : tranches, permutation, SHAP. | `src/evaluate.py`, `src/predict.py`, nb 04 |
| 7 | **Après coup** : version 1999-2026 avec Bloomberg, améliorations sous protocole figé, prévision de la volatilité plutôt que du sens. | `src/v2_bloomberg.py`, `src/ameliorations.py`, `src/volatilite.py` |

### Lexique des concepts

#### Cadrer et préparer les données · S1

- **Fuite d'information (*leakage*).** Utiliser, pour prédire le mois $t$, une donnée qui n'était pas connue à la fin du mois $t$. Exemple du projet : le Tankan est daté par Bloomberg à la fin du trimestre mais publié le mois suivant ; pris à sa date Bloomberg, il aurait fui. D'où le décalage de chaque série à sa date de publication, et un **audit automatique** qui arrête la construction si une variable regarde le futur.
- **Découpage chronologique.** En série temporelle, on n'apprend que sur le passé et on évalue sur le futur ; un mélange aléatoire ferait apprendre l'avenir. Trois blocs : entraînement, validation (pour choisir), test (pour juger, une seule fois).
- **Cible.** `y_baisse` $= 1$ si le Nikkei baisse le mois suivant (la classe positive des métriques est la baisse) ; `y_non_couvert` $= 1$ si ne pas couvrir le yen était meilleur.
- **Variables construites.** Momentum (rendement passé sur 3 et 12 mois), volatilité EWMA (moyenne exponentielle des rendements au carré, qui pèse davantage les jours récents), variation du taux japonais à 10 ans en points de base, portage (*carry* : écart de taux entre l'euro et le yen), attention et tonalité des news.
- **Standardisation.** $z = (x - \mu)/\sigma$, avec $\mu$ et $\sigma$ calculés sur l'entraînement seulement, pour que les variables soient comparables et que la pénalité traite chaque coefficient de la même façon.

#### Modèles · S2 à S5, L1 à L3

- **Régression logistique** (S2, L5). $p = \sigma(w^\top x + b)$ avec $\sigma(u) = \dfrac{1}{1 + e^{-u}}$. Elle sort une probabilité ; $w^\top x + b$ est la *log-cote* (log-odds).
- **Log-loss** (S2, L5). $-\big[y \log p + (1-y)\log(1-p)\big]$, moyennée sur les mois. C'est l'opposé de la log-vraisemblance : la minimiser revient à maximiser la vraisemblance. Elle punit fort une probabilité confiante et fausse.
- **Descente de gradient** (L1). On part de $w = 0$ et on avance de $-\eta\,\nabla J$ à chaque pas. Pour la logistique, $\nabla J = \tfrac{1}{m} X^\top (p - y) + 2\lambda w$. Le coût est convexe : un seul minimum, atteint si le pas vérifie $\eta < 2/L$ ($L$ : constante de Lipschitz du gradient). Codé à la main dans `src/logreg_scratch.py`.
- **Régularisation** (S3, L7). On ajoute $\lambda\lVert w\rVert_2^2$ (L2, *ridge*) ou $\lambda\lVert w\rVert_1$ (L1, *lasso*) au coût. Elle rétrécit les coefficients et limite le surapprentissage. Vue bayésienne : L2 = un *prior* gaussien centré sur 0.
- **Moindres carrés** (L2). Régression linéaire : $w = (X^\top X)^{-1} X^\top y$ (équations normales). $R^2 = 1 - \mathrm{SCR}/\mathrm{SCT}$ ; un $R^2$ négatif en validation veut dire « pire que la moyenne ».
- **Arbre de décision** (S4, L3B). Coupe l'espace en rectangles en choisissant à chaque nœud la coupe qui réduit le plus l'impureté (Gini ou entropie).
- **Forêt aléatoire** (S5). Beaucoup d'arbres, chacun sur un rééchantillon et un sous-ensemble de variables, dont on fait la moyenne. Réduit la variance.
- **Gradient boosting** (S5). Des arbres peu profonds ajoutés un par un, chacun corrigeant les erreurs des précédents. Réduit le biais ; se surajuste vite sans régularisation.
- **k-means** (L3C). Regroupe les mois en $k$ groupes autour de centres ; ici deux régimes de marché (calme, agité), dont la distance sert de variable.

#### Réglage et validation · S3, S5, S6

- **Pipeline scikit-learn.** Enchaîne standardisation, régimes et modèle dans un seul objet, ajusté uniquement sur les données d'apprentissage de chaque pli : pas de fuite par le prétraitement.
- **TimeSeriesSplit.** Validation croisée temporelle : chaque pli apprend sur un début de période et valide sur la suite.
- **Optuna (TPE).** Cherche les hyperparamètres ($\lambda$, profondeur, taux d'apprentissage…) en proposant de nouveaux essais là où les précédents étaient bons. Chaque essai est journalisé.

#### Classes, seuil, calibration · S7, S8

- **Déséquilibre et PR-AUC** (S7). Quand une classe est rare, l'exactitude trompe ; la courbe précision-rappel et son aire jugent mieux la classe positive. La référence d'une PR-AUC est la fréquence de la classe.
- **Calibration** (S8). Un modèle est calibré si, parmi les mois où il annonce 60 %, la baisse arrive environ 6 fois sur 10. **Platt** ajuste une petite logistique sur les scores : $p = \sigma(a\,s + c)$. **Score de Brier** $= \tfrac{1}{n}\sum (p - y)^2$ ; **ECE** = écart moyen entre probabilité annoncée et fréquence observée, par paquets.
- **Règle de décision** (S8). Agir seulement si l'espérance de gain dépasse le coût : on achète si la probabilité de hausse est assez haute, on vend si elle est assez basse, sinon on reste neutre. Les seuils sont choisis sur la validation puis **figés** avant le test.

#### Explicabilité · S9

- **Tranches (*slices*).** La performance par sous-période ou par régime : un score global peut cacher une période où le modèle se trompe toujours.
- **Importance par permutation.** On garde le modèle figé, on mélange une variable (ou un groupe de variables corrélées) entre les lignes, et on mesure combien la log-loss augmente. Grande hausse = variable utile ; valeur négative = le modèle ferait mieux sans elle.
- **SHAP.** Décompose une prédiction : $f(x) = \varphi_0 + \sum_j \varphi_j$, sur l'échelle des log-cotes. Pour un modèle linéaire, $\varphi_j = w_j\,(z_j - \mathbb{E}[z_j])$. Les valeurs de Shapley répartissent l'écart à la moyenne entre variables, en moyenne sur tous les ordres d'arrivée. SHAP décrit le modèle, pas l'économie : aucune causalité.
- **Dépendance partielle (PDP).** La prédiction moyenne quand on fait varier une seule variable.

#### Évaluer honnêtement après coup

- **Walk-forward.** Réentraîner chaque mois sur tous les mois passés, prédire le suivant : c'est ce qu'on aurait fait en vrai.
- **Test de Diebold-Mariano.** Compare les pertes de deux prévisions mois par mois et dit si l'écart est significatif, en tenant compte de l'autocorrélation.
- **Correction de Holm.** Quand on teste beaucoup de variantes, quelques-unes passent par hasard ; Holm resserre les seuils de significativité.
- **Bootstrap par blocs.** Rééchantillonne des blocs de mois consécutifs pour obtenir un intervalle de confiance du ratio de Sharpe (rendement moyen / volatilité) sans casser la dépendance temporelle.
- **Protocole figé (SHA-256).** Les variantes et critères sont écrits avant le calcul, avec une empreinte ; le code refuse de tourner si le protocole a changé. C'est la défense contre le surapprentissage par essais répétés.
- **GARCH, HMM.** GARCH prévoit la volatilité de demain à partir des chocs et de la volatilité d'hier ; un modèle de Markov caché (HMM) suppose des régimes cachés (calme, agité) qui se succèdent avec des probabilités de transition.

#### Données externes

- **GDELT et FinBERT.** GDELT recense les articles de presse du monde entier ; FinBERT, un modèle de langue entraîné sur des textes financiers, note leur tonalité (positive, négative, neutre).
- **Bloomberg.** Référence pour la seconde version : il sert à contrôler les séries (Tankan, IPC, VIX) et leurs dates de publication. Les exports bruts restent hors de Git (licence).

---
<a id="partie-1"></a>
## Partie 1 · Configuration et données

### 1.1 · Configuration

Un seul endroit pour les dates du découpage, les listes de variables et la graine aléatoire : `config.yaml` (première version, 2010-2026) et `config_bloomberg.yaml` (seconde version, 1999-2026). `src/config.py` choisit l'un ou l'autre selon la variable d'environnement `NIKKEI_CONFIG`, et note le commit courant dans chaque résultat pour la traçabilité.

> **Concepts** · configuration unique, reproductibilité (graine, commit).

In [ ]:
# ─── src/config.py · 26 lignes ───

"""Chemins et configuration communs à tous les modules."""
import os
from pathlib import Path
import subprocess

import yaml

ROOT = Path(__file__).resolve().parents[1]
# NIKKEI_CONFIG choisit une autre configuration (config_bloomberg.yaml : seconde version, après le test)
CONFIG = ROOT / os.environ.get("NIKKEI_CONFIG", "config.yaml")
CFG = yaml.safe_load(open(CONFIG))
SEED = CFG["seed"]
RUNS = ROOT / CFG.get("runs", "runs")
MODELS = RUNS / "models"
FIG = ROOT / "figures"

FEATURES = [c for groupe in ["news", "price", "macro", "fx"] for c in CFG["features"][groupe]]
REGIME = CFG["features"]["regime"]
COLONNES = FEATURES + [c for c in REGIME if c not in FEATURES]
CIBLES = {"nikkei": "y_baisse", "couverture": "y_non_couvert"}


def commit_courant() -> str:
    """Hash court du commit Git en cours (pour le journal des essais)."""
    sortie = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True)
    return sortie.stdout.strip() or "inconnu"

**Les deux configurations**

| | début | fin entraînement | fin validation | fin test |
|:--|:-:|:-:|:-:|:-:|
| `config.yaml` (V1) | 2010-02-01 | 2021-12-31 | 2023-12-31 | 2026-09-30 |
| `config_bloomberg.yaml` (V2) | 1999-01-01 | 2021-12-31 | 2023-12-31 | 2026-09-30 |

| famille | variables (identiques dans les deux versions) |
|:--|:--|
| news | `news_attention_6m`, `news_sent_w`, `news_dispo` |
| prix | `n225_mom_12m`, `n225_mom_3m`, `n225_vol_ewma` |
| macro | `d_jgb10y_bp`, `tankan_large_manuf` |
| change | `carry_1m`, `eurjpy_rvol_1m` |

### 1.2 · Données : de la table mensuelle à la table de modélisation

Le fichier lit la table mensuelle construite par les notebooks de collecte, ajoute les cibles du mois suivant, construit les variables de news, attribue à chaque mois son bloc (entraînement, validation, test) et lance l'**audit des fuites**, qui arrête tout si un contrôle échoue :

- la cible est bien le rendement du mois suivant, et aucune cible n'est parmi les variables ;
- le Tankan ne change qu'aux mois de publication (janvier, avril, juillet, octobre) ;
- les variables de news recalculées sur des données coupées en 2020 sont identiques (elles ne regardent pas le futur) ;
- aucune variable n'est presque parfaitement corrélée (Spearman) à une cible.

> **Concepts** · fuite d'information, décalage de publication, cible décalée d'un mois (`shift(-1)`), découpage chronologique (S1).

In [ ]:
# ─── src/data.py · 106 lignes ───

"""Étape A : de la table mensuelle à la table de modélisation (notebook 01).

Entrée : data/2_construites/dataset_monthly.csv, construit par les notebooks de collecte 00a et 00b
(téléchargement, nettoyage, calendrier de Tokyo, décalages de publication, agrégation mensuelle).
Sortie : data/3_modele/dataset_model.csv, avec les cibles, les variables construites et le bloc du split.
"""
import numpy as np
import pandas as pd

from .config import CFG, COLONNES, FEATURES, REGIME, ROOT

GARDER_EN_PLUS = ["y_baisse", "y_non_couvert", "target_up_next", "target_rn_next", "target_fx_next",
                  "target_reur_unhedged_next", "target_reur_hedged_next", "target_short_eur_next",
                  "eur_overnight", "boj_meeting_next_month", "n225_close", "bloc"]


def charger_mensuel() -> pd.DataFrame:
    return pd.read_csv(ROOT / CFG["data"]["monthly"], parse_dates=["date"]).set_index("date")


def ajouter_cibles(df: pd.DataFrame) -> pd.DataFrame:
    """y_baisse = 1 si le Nikkei baisse le mois suivant ; y_non_couvert = 1 si ne pas couvrir était mieux."""
    df = df.copy()
    inconnu = df["target_rn_next"].isna()
    df["y_baisse"] = (df["target_rn_next"] <= 0).astype(float)
    df["y_non_couvert"] = (df["carry_1m"] - df["target_fx_next"] <= 0).astype(float)
    df.loc[inconnu, ["y_baisse", "y_non_couvert"]] = np.nan
    return df


def variables_news(df: pd.DataFrame) -> pd.DataFrame:
    """Volume anormal d'articles, sentiment pondéré et indicateur de présence des news.

    Chaque ligne n'utilise que son propre passé : le calcul peut se faire avant le split.
    """
    n_tot = (df["news_japan_n_1m"] + df["news_boj_n_1m"]).where(lambda n: n > 0)
    log_n = np.log(n_tot)
    out = pd.DataFrame(index=df.index)
    out["news_attention_6m"] = log_n - log_n.shift(1).rolling(6, min_periods=3).mean()
    out["news_sent_w"] = (df["news_japan_n_1m"] * df["news_japan_sent_1m"]
                          + df["news_boj_n_1m"] * df["news_boj_sent_1m"]) / n_tot
    out["news_dispo"] = out["news_attention_6m"].notna().astype(int)
    return out


def ajouter_blocs(df: pd.DataFrame) -> pd.DataFrame:
    s = {k: pd.Timestamp(v) for k, v in CFG["split"].items()}
    df = df.copy()
    df["bloc"] = np.select(
        [df.index < s["start"], df.index <= s["train_end"], df.index <= s["validation_end"], df.index <= s["test_end"]],
        ["hors période", "entraînement", "validation", "test"], default="à prédire")
    return df


def audit_fuites(df: pd.DataFrame) -> None:
    """Contrôles du notebook 01 : lèvent une erreur si une fuite est détectée."""
    ecart_cible = (df["target_rn_next"] - df["n225_ret_1m"].shift(-1)).abs().max()
    assert ecart_cible < 1e-9, "la cible n'est pas le rendement du mois suivant"
    ecart_couv = ((df["target_reur_hedged_next"] - df["target_reur_unhedged_next"])
                  - (df["carry_1m"] - df["target_fx_next"])).abs().max()
    assert ecart_couv < 1e-5, "l'identité de la couverture ne tient pas"
    cibles = {c for c in df.columns if c.startswith("target_")} | {"y_baisse", "y_non_couvert"}
    assert not cibles & set(FEATURES + REGIME), "une cible est parmi les variables"
    mois_tankan = set(df.index[df["tankan_large_manuf"].diff().fillna(0) != 0].month)
    assert mois_tankan <= {1, 4, 7, 10}, "le Tankan change avant sa publication"
    coupe = variables_news(df.loc[:"2020-06-30"])
    assert np.allclose(variables_news(df).loc[:"2020-06-30"].to_numpy(), coupe.to_numpy(), equal_nan=True), \
        "les variables de news regardent le futur"
    ecart_prix = (df["target_rn_next"] - (df["n225_close"].shift(-1) / df["n225_close"] - 1)).abs().max()
    assert ecart_prix < 1e-5, "la cible ne correspond pas aux prix de t et t+1"
    # Contrôle sur les valeurs, pas seulement sur les noms : aucune variable ne doit recopier une cible
    connus = df["target_rn_next"].notna()
    for col in FEATURES + REGIME:
        for cible in ["target_rn_next", "target_fx_next", "y_baisse"]:
            paire = df.loc[connus, [col, cible]].dropna()
            if paire[col].nunique() > 2:
                rho = abs(paire[col].corr(paire[cible], method="spearman"))
                assert rho < 0.9, f"{col} est presque identique à {cible} (corrélation de rang {rho:.2f}) : fuite probable"
    if "bloc" in df:
        assert (df["bloc"] == "à prédire").sum() == 1, "il doit y avoir exactement une ligne à prédire"


def construire_table() -> pd.DataFrame:
    df = ajouter_cibles(charger_mensuel())
    df[["news_attention_6m", "news_sent_w", "news_dispo"]] = variables_news(df)
    df = ajouter_blocs(df)
    audit_fuites(df)
    garder = COLONNES + [c for c in GARDER_EN_PLUS if c not in COLONNES]
    return df.loc[df["bloc"] != "hors période", garder]


def charger_table() -> pd.DataFrame:
    return pd.read_csv(ROOT / CFG["data"]["model_table"], parse_dates=["date"]).set_index("date")


def main() -> None:
    table = construire_table()
    sortie = ROOT / CFG["data"]["model_table"]
    sortie.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(sortie)
    print(f"{sortie.relative_to(ROOT)} : {table.shape[0]} mois x {table.shape[1]} colonnes ; "
          f"blocs {table['bloc'].value_counts().to_dict()} ; audit des fuites réussi")


if __name__ == "__main__":
    main()

---
<a id="partie-2"></a>
## Partie 2 · Modéliser, décider, tester

### 2.1 · La pipeline et les espaces de recherche

`construire_pipeline` assemble imputation et standardisation, ajout des distances aux régimes k-means (calme, agité), puis le modèle. `proposer` décrit pour Optuna l'espace de chaque modèle : force de la régularisation de la logistique, nombre et profondeur des arbres, taux d'apprentissage du boosting.

> **Concepts** · pipeline sans fuite (le prétraitement s'ajuste dans chaque pli), standardisation, k-means (L3C), logistique régularisée (S2, S3), forêt et boosting (S4, S5).

In [ ]:
# ─── src/pipeline.py · 50 lignes ───

"""Étape B : la pipeline scikit-learn et les espaces de recherche des modèles (notebook 02)."""
from lightgbm import LGBMClassifier
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .config import FEATURES, REGIME, SEED

K_REGIMES = 2   # coude du k-means sur l'entraînement (notebook 02, L3B)


def construire_pipeline(modele, k: int = K_REGIMES) -> Pipeline:
    """Imputation, distances aux régimes k-means, standardisation, puis le modèle.

    Tout ce qui apprend un paramètre est dans la pipeline, donc ajusté sur le fold d'entraînement seulement.
    """
    regime = Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler()),
                       ("kmeans", KMeans(n_clusters=k, n_init=10, random_state=SEED))])
    prep = ColumnTransformer([("vars", SimpleImputer(strategy="median"), FEATURES),
                              ("regime", regime, REGIME)])
    return Pipeline([("prep", prep), ("scale", StandardScaler()), ("model", modele)])


def proposer(trial, nom: str):
    """Modèle proposé par un essai Optuna (ou par optuna.trial.FixedTrial pour des paramètres fixés)."""
    poids = trial.suggest_categorical("class_weight", [None, "balanced"])
    if nom == "logit":
        # saga ne pénalise pas l'intercept b (liblinear le pénalise)
        return LogisticRegression(C=trial.suggest_float("C", 1e-3, 10, log=True),
                                  penalty=trial.suggest_categorical("penalty", ["l1", "l2"]),
                                  solver="saga", class_weight=poids, max_iter=5000, random_state=SEED)
    if nom == "rf":
        return RandomForestClassifier(n_estimators=trial.suggest_int("n_estimators", 100, 500, step=100),
                                      max_depth=trial.suggest_int("max_depth", 2, 4),
                                      min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 15),
                                      max_features=trial.suggest_categorical("max_features", ["sqrt", 0.5]),
                                      class_weight=poids, random_state=SEED, n_jobs=1)
    if nom == "lgbm":
        return LGBMClassifier(n_estimators=trial.suggest_int("n_estimators", 50, 300, step=25),
                              learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                              num_leaves=trial.suggest_int("num_leaves", 4, 8),
                              min_child_samples=trial.suggest_int("min_child_samples", 5, 15),
                              reg_lambda=trial.suggest_float("reg_lambda", 0.0, 5.0),
                              class_weight=poids, random_state=SEED, n_jobs=1, verbose=-1)
    raise ValueError(f"modèle inconnu : {nom}")

### 2.2 · Réglage Optuna

Pour chaque modèle, Optuna propose des hyperparamètres ; chaque proposition est jugée par la log-loss moyenne sur les plis d'un `TimeSeriesSplit` de l'entraînement. Tous les essais sont écrits dans un journal ; le meilleur de chaque modèle est réajusté et sauvegardé.

> **Concepts** · validation croisée temporelle, recherche bayésienne TPE, log-loss comme critère (S5, S6).

In [ ]:
# ─── src/tune.py · 104 lignes ───

"""Réglage Optuna avec TimeSeriesSplit, journal des essais, modèles retenus (notebook 02).

Usage : python -m src.tune [--essais 50] [--sortie runs]
"""
import argparse
import json
import time
import warnings
from datetime import datetime

import joblib
import numpy as np
import optuna
import pandas as pd
from optuna.samplers import TPESampler
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss
from sklearn.model_selection import TimeSeriesSplit

from .config import CIBLES, COLONNES, RUNS, SEED, commit_courant
from .data import charger_table
from .pipeline import construire_pipeline, proposer

PLAN = [("nikkei", "logit"), ("nikkei", "rf"), ("nikkei", "lgbm"), ("couverture", "logit")]
FOLDS_CV = TimeSeriesSplit(n_splits=4, test_size=12)


def evaluer_cv(pipe, X, y, folds):
    """Log-loss, Brier et PR-AUC moyens sur les folds, et prédictions hors fold."""
    pertes, briers, praucs = [], [], []
    oof = pd.Series(np.nan, index=y.index)
    for a, b in folds:
        modele = clone(pipe).fit(X.iloc[a], y.iloc[a])
        p = modele.predict_proba(X.iloc[b])[:, 1]
        pertes.append(log_loss(y.iloc[b], p, labels=[0, 1]))
        briers.append(brier_score_loss(y.iloc[b], p))
        praucs.append(average_precision_score(y.iloc[b], p))
        oof.iloc[b] = p
    return {"logloss": float(np.mean(pertes)), "logloss_std": float(np.std(pertes)),
            "brier": float(np.mean(briers)), "pr_auc": float(np.mean(praucs)),
            "logloss_folds": [round(float(x), 4) for x in pertes]}, oof


def main(n_essais: int = 50, sortie=RUNS) -> None:
    from pathlib import Path
    sortie = Path(sortie)
    modeles = sortie / "models"
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", message=".*feature names.*")
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    modeles.mkdir(parents=True, exist_ok=True)

    table = charger_table()
    train, valid = table[table["bloc"] == "entraînement"], table[table["bloc"] == "validation"]
    X_tr, X_va = train[COLONNES], valid[COLONNES]
    folds = list(FOLDS_CV.split(X_tr))
    date, commit = datetime.now().strftime("%Y-%m-%d %H:%M"), commit_courant()

    journal, predictions = [], []
    for cible, nom in PLAN:
        y_tr, y_va = train[CIBLES[cible]].astype(int), valid[CIBLES[cible]].astype(int)

        def objectif(trial):
            res, _ = evaluer_cv(construire_pipeline(proposer(trial, nom)), X_tr, y_tr, folds)
            for cle in ["logloss_std", "brier", "pr_auc", "logloss_folds"]:
                trial.set_user_attr(cle, res[cle])
            return res["logloss"]

        debut = time.time()
        etude = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED))
        etude.optimize(objectif, n_trials=n_essais)
        meilleur = etude.best_trial

        fixe = lambda: construire_pipeline(proposer(optuna.trial.FixedTrial(etude.best_params), nom))
        pipe = fixe().fit(X_tr, y_tr)
        joblib.dump(pipe, modeles / f"{cible}_{nom}.joblib")
        _, oof = evaluer_cv(fixe(), X_tr, y_tr, folds)
        predictions.append(pd.DataFrame({"cible": cible, "modele": nom, "bloc": "hors fold", "y": y_tr, "p": oof}).dropna())
        predictions.append(pd.DataFrame({"cible": cible, "modele": nom, "bloc": "validation", "y": y_va,
                                         "p": pipe.predict_proba(X_va)[:, 1]}, index=valid.index))

        for t in etude.trials:
            retenu = t.number == meilleur.number
            journal.append({"date": date, "commit": commit, "cible": cible, "modele": nom, "essai": t.number,
                            "params": json.dumps(t.params), "cv_logloss": round(t.value, 5),
                            "cv_logloss_std": round(t.user_attrs["logloss_std"], 5),
                            "cv_brier": round(t.user_attrs["brier"], 5), "cv_pr_auc": round(t.user_attrs["pr_auc"], 5),
                            "logloss_par_fold": json.dumps(t.user_attrs["logloss_folds"]), "retenu": retenu,
                            "artefact_modele": f"runs/models/{cible}_{nom}.joblib" if retenu else "",
                            "figure": "figures/02_optuna_historique.png"})
        print(f"{cible:10s} {nom:5s} : log-loss CV {meilleur.value:.4f} (essai {meilleur.number}) en {time.time() - debut:.0f} s")

    pd.DataFrame(journal).to_csv(sortie / "trials_log.csv", index=False)
    pd.concat(predictions).rename_axis("date").to_csv(sortie / "predictions_02.csv")
    print(f"{sortie}/trials_log.csv : {len(journal)} essais ; modèles dans {modeles}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--essais", type=int, default=50)
    parser.add_argument("--sortie", default=str(RUNS), help="dossier où écrire le journal, les prédictions et les modèles")
    args = parser.parse_args()
    main(args.essais, args.sortie)

### 2.3 · La logistique à la main (annexe de Maths)

Le fichier réécrit la régression logistique pénalisée sans scikit-learn : la sigmoïde, le coût $J_\lambda$, son gradient, la constante de Lipschitz qui borne le pas, et la descente de gradient. Le notebook 05 vérifie qu'on retrouve les coefficients de scikit-learn.

> **Concepts** · vraisemblance et log-loss (L5), gradient et descente (L1), pénalité L2 et prior gaussien (L7), convexité : un seul minimum (L7, L8).

In [ ]:
# ─── src/logreg_scratch.py · 42 lignes ───

"""Régression logistique L2 codée à la main, par descente de gradient (annexe de Maths, notebook 05).

J_λ(w, b) = -(1/m) Σ [y log p + (1 - y) log(1 - p)] + λ ||w||²,   p = σ(w·x + b)
∇_w J_λ = (1/m) Xᵀ(p - y) + 2λ w,   ∂J_λ/∂b = (1/m) Σ (p - y)   (b n'est pas pénalisé)
"""
import numpy as np


def sigmoide(u):
    return 1 / (1 + np.exp(-u))


def cout(w, b, X, y, lam=0.0):
    p = np.clip(sigmoide(X @ w + b), 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)) + lam * (w @ w)


def gradient(w, b, X, y, lam=0.0):
    r = sigmoide(X @ w + b) - y
    return X.T @ r / len(y) + 2 * lam * w, r.mean()


def constante_lipschitz(X, lam):
    """Borne L ≤ ¼ λmax(X̃ᵀX̃/m) + 2λ, avec X̃ = [X, 1] ; la descente converge pour α < 2/L."""
    Xt = np.c_[X, np.ones(len(X))]
    return 0.25 * np.linalg.eigvalsh(Xt.T @ Xt / len(X)).max() + 2 * lam


def descente(X, y, alpha, lam, T, w=None, b=0.0):
    """Descente de gradient ; renvoie les poids, l'intercept et l'historique du coût."""
    w = np.zeros(X.shape[1]) if w is None else np.asarray(w, dtype=float).copy()
    historique = np.empty(T)
    for t in range(T):
        gw, gb = gradient(w, b, X, y, lam)
        w, b = w - alpha * gw, b - alpha * gb
        historique[t] = cout(w, b, X, y, lam)
    return w, b, historique


def ajuster(X, y, lam, T=20000):
    """Ajuste avec le pas classique α = 1/L."""
    return descente(X, y, 1 / constante_lipschitz(X, lam), lam, T)

### 2.4 · Calibration, règle de décision, décision figée

Le fichier calibre les scores par Platt sur la validation, calcule les rendements des actions (acheter, neutre, vendre ; couvert ou non), choisit les seuils qui maximisent l'espérance de gain, et **fige** le tout dans `runs/decision_finale.json`. L'option `--verifier` recalcule et compare au fichier sans rien écrire : la décision ne peut pas être modifiée en douce après le test.

> **Concepts** · calibration de Platt, espérance de gain et coût de transaction, seuils, engagement avant le test (S8).

In [ ]:
# ─── src/decision.py · 116 lignes ───

"""Calibration de Platt, règle de décision par espérance de gain, et décision figée (notebook 03).

La décision du projet est figée dans runs/decision_finale.json (commit 75712cf, avant le test).
Usage :
  python -m src.decision --verifier   recalcule tout et compare à la décision figée, sans rien écrire
  python -m src.decision --ecrire     réécrit le fichier (refusé s'il existe déjà, sauf --forcer)
"""
import argparse
import json
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from .config import RUNS, commit_courant
from .data import charger_table

MODELES = ["logit", "rf", "lgbm"]
MARGE = 0.005          # gain espéré minimal au-dessus du cash pour prendre position, fixé à l'avance
FICHIER = RUNS / "decision_finale.json"


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def apprendre_platt(p_hors_fold, y_hors_fold) -> dict:
    """Platt : sigmoïde du log-odds du modèle, apprise sur les prédictions hors fold."""
    cal = LogisticRegression(C=1e6, max_iter=1000).fit(logit(p_hors_fold).reshape(-1, 1), y_hors_fold)
    return {"a": float(cal.coef_[0, 0]), "c": float(cal.intercept_[0])}


def appliquer_platt(p, platt: dict):
    return 1 / (1 + np.exp(-(platt["a"] * logit(p) + platt["c"])))


def couts_entrainement(train: pd.DataFrame) -> dict:
    """Rendements moyens en excès du cash d'un achat couvert et d'une vente, selon hausse ou baisse."""
    cash = train["eur_overnight"] / 100 / 12
    achat = train["target_reur_hedged_next"] - cash
    vente = train["target_short_eur_next"] - cash
    hausse = train["y_baisse"] == 0
    return {"B_u": float(achat[hausse].mean()), "B_d": float(achat[~hausse].mean()),
            "S_u": float(vente[hausse].mean()), "S_d": float(vente[~hausse].mean())}


def seuils(couts: dict, marge: float) -> dict:
    return {"theta_a": (marge - couts["B_d"]) / (couts["B_u"] - couts["B_d"]),
            "theta_v": (couts["S_d"] - marge) / (couts["S_d"] - couts["S_u"])}


def actions(p_hausse: pd.Series, couts: dict, marge: float) -> pd.Series:
    """Acheter ou vendre si le gain espéré au-dessus du cash dépasse la marge, sinon neutre."""
    gain_achat = p_hausse * couts["B_u"] + (1 - p_hausse) * couts["B_d"]
    gain_vente = p_hausse * couts["S_u"] + (1 - p_hausse) * couts["S_d"]
    meilleur = np.where(gain_achat >= gain_vente, "acheter", "vendre")
    return pd.Series(np.where(np.maximum(gain_achat, gain_vente) >= marge, meilleur, "neutre"), index=p_hausse.index)


def rendements(act: pd.Series, bloc: pd.DataFrame) -> pd.Series:
    """Rendement en euros du mois suivant : achat couvert, cash ou vente."""
    cash = bloc["eur_overnight"] / 100 / 12
    return pd.Series(np.select([act == "acheter", act == "vendre"],
                               [bloc["target_reur_hedged_next"], bloc["target_short_eur_next"]], cash), index=bloc.index)


def calculer_decision() -> dict:
    table = charger_table()
    train = table[table["bloc"] == "entraînement"]
    pred = pd.read_csv(RUNS / "predictions_02.csv", parse_dates=["date"])
    hors_fold = pred[(pred["cible"] == "nikkei") & (pred["bloc"] == "hors fold")]
    platt = {}
    for m in MODELES:
        d = hors_fold[hors_fold["modele"] == m]
        platt[m] = apprendre_platt(d["p"].to_numpy(), d["y"].astype(int).to_numpy())
    couts = couts_entrainement(train)
    return {"modele_nikkei": "logit", "platt": platt, "couts_entrainement_exces_du_cash": couts,
            "marge_m": MARGE, "seuils": seuils(couts, MARGE), "couverture": "toujours couvert (h = 1)"}


def charger_decision() -> dict:
    return json.loads(FICHIER.read_text())


def verifier() -> None:
    """Recalcule la décision et la compare au fichier figé."""
    fige, recalcul = charger_decision(), calculer_decision()
    ecarts = {"platt " + m: max(abs(fige["platt"][m][k] - recalcul["platt"][m][k]) for k in "ac") for m in MODELES}
    ecarts["coûts"] = max(abs(fige["couts_entrainement_exces_du_cash"][k] - v)
                          for k, v in recalcul["couts_entrainement_exces_du_cash"].items())
    ecarts["seuils"] = max(abs(fige["seuils"][k] - v) for k, v in recalcul["seuils"].items())
    for nom, e in ecarts.items():
        print(f"{nom:14s} écart max {e:.1e}")
    assert max(ecarts.values()) < 1e-6, "le recalcul ne retrouve pas la décision figée"
    print("décision figée retrouvée à l'identique")


def ecrire(forcer: bool = False) -> None:
    if FICHIER.exists() and not forcer:
        raise SystemExit(f"{FICHIER.name} existe et a été figé avant le test : rien n'est réécrit (utiliser --forcer)")
    decision = calculer_decision() | {"date": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(),
                                      "fichiers_modeles": {m: f"runs/models/nikkei_{m}.joblib" for m in MODELES}}
    FICHIER.write_text(json.dumps(decision, indent=2, ensure_ascii=False))
    print(f"écrit : {FICHIER}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    groupe = parser.add_mutually_exclusive_group(required=True)
    groupe.add_argument("--verifier", action="store_true")
    groupe.add_argument("--ecrire", action="store_true")
    parser.add_argument("--forcer", action="store_true")
    args = parser.parse_args()
    verifier() if args.verifier else ecrire(args.forcer)

**Ce que contient `runs/decision_finale.json`**

| | |
|:--|:--|
| Date de la décision | 2026-11-11 18:08 (commit `2aa4b40`) |
| Modèle retenu | régression logistique |
| Calibration de Platt | $a = 0{,}915$, $c = 0{,}010$ |
| Seuil d'achat $	heta_a$ | 58,0 % de probabilité de hausse |
| Seuil de vente $	heta_v$ | 45,6 % |
| Marge $m$ (fixée à l'avance) | 0,5 % |
| Couverture | toujours couvert ($h = 1$) |
| Log-loss en validation | modèle 0,6993 · constante 0,6805 → plancher non battu |
| Engagement avant le test | évaluer une seule fois sur le test ; si le modèle y perd aussi contre la constante, recommander « toujours acheter, couvert » |

### 2.5 · Le test, ouvert une seule fois

Le fichier applique la décision figée aux mois 2024-2026 et mesure log-loss, Brier, PR-AUC et rendement en euros, contre des références (probabilité constante, toujours acheter). Le résultat est dans `runs/evaluation_test.json` ; le fichier ne sert plus qu'à vérifier qu'on retrouve les mêmes chiffres.

> **Concepts** · test unique, références naïves, métriques probabilistes et financières (S1, S7, S8).

In [ ]:
# ─── src/evaluate.py · 53 lignes ───

"""Évaluation sur le test 2024-2026 avec la décision figée (notebook 03b).

Le test a été évalué une seule fois, au commit 6efd12c (runs/evaluation_test.json).
Usage : python -m src.evaluate --verifier   recalcule et compare au fichier, sans rien écrire
"""
import argparse
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss

from .config import COLONNES, ROOT, RUNS
from .data import charger_table
from .decision import actions, appliquer_platt, charger_decision, rendements

FICHIER = RUNS / "evaluation_test.json"


def evaluer() -> dict:
    table, dec = charger_table(), charger_decision()
    train, test = table[table["bloc"] == "entraînement"], table[table["bloc"] == "test"]
    y = test["y_baisse"].astype(int)
    pipe = joblib.load(ROOT / dec["fichiers_modeles"][dec["modele_nikkei"]])
    p_baisse = appliquer_platt(pipe.predict_proba(test[COLONNES])[:, 1], dec["platt"][dec["modele_nikkei"]])
    p_const = np.full(len(y), train["y_baisse"].mean())
    act = actions(pd.Series(1 - p_baisse, index=test.index), dec["couts_entrainement_exces_du_cash"], dec["marge_m"])
    cumul = lambda r: float(np.prod(1 + np.asarray(r)) - 1)
    return {"log_loss_modele": log_loss(y, p_baisse), "log_loss_constante": log_loss(y, p_const),
            "brier_modele": brier_score_loss(y, p_baisse), "pr_auc_modele": average_precision_score(y, p_baisse),
            "actions": act.value_counts().to_dict(), "cumul_regle": cumul(rendements(act, test)),
            "cumul_achat_couvert": cumul(test["target_reur_hedged_next"])}


def verifier() -> None:
    fige, r = json.loads(FICHIER.read_text()), evaluer()
    attendu = {"log_loss_modele": fige["metriques"]["logistique"]["log-loss"],
               "log_loss_constante": fige["metriques"]["constante (fréquence de l'entraînement)"]["log-loss"],
               "cumul_regle": fige["performance_euros"]["règle du modèle"]["cumul"],
               "cumul_achat_couvert": fige["performance_euros"]["toujours acheter, couvert"]["cumul"]}
    for cle, valeur in attendu.items():
        print(f"{cle:22s} figé {valeur:.4f} | recalculé {r[cle]:.4f}")
        assert abs(valeur - r[cle]) < 1e-3, f"{cle} ne correspond pas"
    assert r["actions"] == fige["actions"], "les actions ne correspondent pas"
    print("évaluation du test retrouvée à l'identique (rien n'a été réécrit)")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--verifier", action="store_true", required=True)
    parser.parse_args()
    verifier()

---
<a id="partie-3"></a>
## Partie 3 · Expliquer et prédire

### 3.1 · Signal, historique et explication SHAP

Le fichier recharge le modèle figé, calcule la probabilité du dernier mois connu, l'action de la règle et la **contribution SHAP** de chaque variable (pour la logistique : coefficient × écart à la moyenne de l'entraînement, sur l'échelle des log-cotes). C'est ce que lisent l'API, le tableau de bord et le site.

> **Concepts** · SHAP linéaire, additivité $f(x) = \varphi_0 + \sum_j \varphi_j$, lecture en cascade (*waterfall*) (S9). L'importance par permutation et les tranches sont dans le notebook 04.

In [ ]:
# ─── src/predict.py · 93 lignes ───

"""Signal du mois, historique des décisions et explication SHAP, avec la décision figée.

Utilisé par la plateforme (API et tableau de bord). Usage : python -m src.predict [--mois 2026-10]
"""
import argparse
from functools import lru_cache

import joblib
import numpy as np
import pandas as pd

from .config import COLONNES, FEATURES, ROOT
from .data import charger_table
from .decision import actions, appliquer_platt, charger_decision, rendements

RECOMMANDATION = "toujours acheter, couvert (le modèle ne bat pas la constante sur le test)"


@lru_cache(maxsize=1)
def _modele():
    dec = charger_decision()
    return dec, joblib.load(ROOT / dec["fichiers_modeles"][dec["modele_nikkei"]])


def _ligne(table: pd.DataFrame, mois: str | None) -> pd.DataFrame:
    ligne = table[table["bloc"] == "à prédire"] if mois is None else table[table.index.strftime("%Y-%m") == mois]
    if ligne.empty:
        raise KeyError(f"aucune donnée pour le mois {mois}")
    return ligne


def probabilites_hausse(lignes: pd.DataFrame) -> pd.Series:
    """Probabilité calibrée de hausse le mois suivant : 1 - Platt(P(baisse))."""
    dec, pipe = _modele()
    p_baisse = appliquer_platt(pipe.predict_proba(lignes[COLONNES])[:, 1], dec["platt"][dec["modele_nikkei"]])
    return pd.Series(1 - p_baisse, index=lignes.index)


def signal(mois: str | None = None, table: pd.DataFrame | None = None) -> dict:
    """Signal pour le mois suivant, à partir des données de fin de `mois` (par défaut, la ligne à prédire)."""
    table = charger_table() if table is None else table
    dec, _ = _modele()
    ligne = _ligne(table, mois)
    p_hausse = float(probabilites_hausse(ligne).iloc[0])
    action = actions(pd.Series([p_hausse], index=ligne.index), dec["couts_entrainement_exces_du_cash"], dec["marge_m"]).iloc[0]
    resultat = ligne["target_rn_next"].iloc[0]
    return {"mois_des_donnees": ligne.index[0].strftime("%Y-%m"), "bloc": ligne["bloc"].iloc[0],
            "p_hausse": round(p_hausse, 4), "score": round(2 * p_hausse - 1, 4),
            "action_du_modele": action, "couverture": "couvert" if action == "acheter" else "sans objet",
            "seuils": {k: round(v, 4) for k, v in dec["seuils"].items()},
            "recommandation_finale": RECOMMANDATION,
            "rendement_realise_mois_suivant": None if np.isnan(resultat) else round(float(resultat), 4)}


def historique(table: pd.DataFrame | None = None, blocs=("validation", "test")) -> pd.DataFrame:
    """Probabilité, action et rendements en euros mois par mois, pour les blocs demandés."""
    table = charger_table() if table is None else table
    dec, _ = _modele()
    d = table[table["bloc"].isin(blocs)]
    p = probabilites_hausse(d)
    act = actions(p, dec["couts_entrainement_exces_du_cash"], dec["marge_m"])
    return pd.DataFrame({"bloc": d["bloc"], "p_hausse": p, "action": act,
                         "hausse_realisee": (d["y_baisse"] == 0).astype(int),
                         "rendement_regle": rendements(act, d),
                         "rendement_achat_couvert": d["target_reur_hedged_next"],
                         "rendement_achat_non_couvert": d["target_reur_unhedged_next"],
                         "rendement_vente": d["target_short_eur_next"],
                         "rendement_cash": d["eur_overnight"] / 100 / 12})


def contributions(mois: str | None = None, table: pd.DataFrame | None = None) -> pd.DataFrame:
    """SHAP du modèle linéaire : φ_j = w_j (z_j - moyenne d'entraînement de z_j), en log-odds de la baisse."""
    table = charger_table() if table is None else table
    _, pipe = _modele()
    prep, w = pipe[:-1], pipe.named_steps["model"].coef_[0]
    z_moyen = prep.transform(table.loc[table["bloc"] == "entraînement", COLONNES]).mean(axis=0)
    ligne = _ligne(table, mois)
    z = prep.transform(ligne[COLONNES])[0]
    regime = pipe.named_steps["prep"].named_transformers_["regime"]
    vix_des_centres = regime.named_steps["scale"].inverse_transform(regime.named_steps["kmeans"].cluster_centers_)[:, -1]
    nom_centre = ["distance au régime agité" if v == vix_des_centres.max() else "distance au régime calme"
                  for v in vix_des_centres]   # le régime agité est celui dont le centre a le VIX le plus élevé
    noms = FEATURES + nom_centre
    valeurs = pipe.named_steps["prep"].transform(ligne[COLONNES])[0]
    return (pd.DataFrame({"variable": noms, "valeur": valeurs, "contribution": w * (z - z_moyen)})
            .sort_values("contribution", key=np.abs, ascending=False).reset_index(drop=True))


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--mois", default=None, help="mois des données, au format AAAA-MM")
    for cle, valeur in signal(parser.parse_args().mois).items():
        print(f"{cle:32s} {valeur}")

---
<a id="partie-4"></a>
## Partie 4 · Après le test

### 4.1 · Seconde version : 1999-2026, Bloomberg en référence

Après l'ouverture du test, le fichier rejoue tout le protocole sur une histoire plus longue :

1. **Contrôle Bloomberg** : compare le Tankan, l'IPC et le VIX de la table aux exports Bloomberg. Le Tankan coïncide une fois décalé d'un mois (sa publication) ; pris à sa date Bloomberg, il aurait fui. L'export du VIX, converti en euros par erreur, est écarté.
2. **Réglage** Optuna sur 1999-2021, calibration, seuils, puis évaluation et SHAP sur le test.

Tout s'écrit dans `runs/v2_bloomberg/` : la décision et l'évaluation de la première version ne sont pas touchées. C'est cette table (`data/3_modele/dataset_model2.csv`) et ces contrôles que le site affiche dans la salle « Le dataset » et le chapitre « Après coup ».

> **Concepts** · contrôle de qualité des données, dates de publication, réentraînement sur une histoire plus longue.

In [ ]:
# ─── src/v2_bloomberg.py · 226 lignes ───

"""Seconde version, après l'ouverture du test : données 1999-2026 et séries Bloomberg en référence.

Même protocole que la première version, rejoué de bout en bout :
  1. contrôle des séries Bloomberg (Tankan, IPC, VIX) contre la table mensuelle, et des dates de publication ;
  2. réglage Optuna sur l'entraînement 1999-2021 (TimeSeriesSplit), comme le notebook 02 ;
  3. calibration de Platt et choix du modèle sur la validation 2022-2023, règle du notebook 03 ;
  4. une évaluation sur le test 2024-2026, au format du notebook 03b.

Le test 2024-2026 a déjà été ouvert par la première version : ce résultat est une exploration après coup, pas un
test à l'aveugle. La décision et l'évaluation figées de runs/ ne sont pas touchées : tout s'écrit dans runs/v2_bloomberg/.

Usage : NIKKEI_CONFIG=config_bloomberg.yaml python -m src.v2_bloomberg [--essais 50] [--sans-reglage]
"""
import argparse
import json
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss

from .config import CFG, COLONNES, CONFIG, ROOT, RUNS, commit_courant
from .data import charger_mensuel, charger_table
from .decision import MARGE, actions, appliquer_platt, apprendre_platt, couts_entrainement, rendements, seuils

MODELES = ["logit", "rf", "lgbm"]
NOMS = {"logit": "logistique", "rf": "Random Forest", "lgbm": "LightGBM", "moyenne": "moyenne des trois"}


# ---------- 1. les séries Bloomberg ----------
def _serie_bloomberg(fichier: str) -> pd.Series:
    x = pd.read_excel(ROOT / CFG["data"]["bloomberg"] / fichier, header=3)
    x.columns = ["date", "valeur"]
    return x.assign(date=pd.to_datetime(x["date"])).set_index("date")["valeur"].astype(float)


def controle_bloomberg() -> dict:
    """Compare les séries Bloomberg à la table mensuelle, en tenant compte de leur date de publication."""
    mensuel = charger_mensuel()
    mois = mensuel.index.to_period("M")
    rapport = {}

    # Tankan : Bloomberg date l'enquête à la fin du trimestre ; elle n'est connue qu'au mois suivant
    # (début avril, juillet, octobre ; mi-décembre pour l'enquête de décembre, utilisée en janvier par prudence).
    tankan = _serie_bloomberg("Trimestre_JNTSMFG_INDEX_HP.xlsx")
    publie = pd.Series(tankan.to_numpy(), index=tankan.index.to_period("M") + 1)
    table = pd.Series(mensuel["tankan_large_manuf"].to_numpy(), index=mois)
    communs = publie.index.intersection(table.index)
    egaux = int((publie[communs] == table[communs]).sum())
    naif = pd.Series(tankan.to_numpy(), index=tankan.index.to_period("M"))
    communs_naif = naif.index.intersection(table.index)
    rapport["tankan"] = {
        "trimestres_compares": int(len(communs)), "valeurs_identiques": egaux,
        "regle": "valeur de l'enquête du trimestre utilisée à partir du mois suivant sa date Bloomberg",
        "erreur_si_date_bloomberg_prise_telle_quelle": f"{int((naif[communs_naif] != table[communs_naif]).sum())} trimestres "
                                                       "auraient utilisé l'enquête avant sa publication"}

    # IPC : publié le mois suivant ; la table le recalcule à partir de l'indice, avec un mois de décalage
    ipc = _serie_bloomberg("Monthly_JNCPIYOY_INDEX_HP.xlsx")
    ipc_m = pd.Series(ipc.to_numpy(), index=ipc.index.to_period("M"))
    table_ipc = pd.Series(mensuel["cpi_all_yoy"].to_numpy(), index=mois).dropna()
    corr = {f"decalage_{k}": float(ipc_m.shift(k, freq="M").reindex(table_ipc.index).corr(table_ipc)) for k in range(3)}
    ecart = (ipc_m.shift(1, freq="M").reindex(table_ipc.index) - table_ipc).abs().dropna()
    rapport["ipc"] = {"mois_compares": int(len(ecart)), "correlations": corr,
                      "ecart_moyen_points": float(ecart.mean()), "ecart_max_points": float(ecart.max()),
                      "lecture": "même série que Bloomberg une fois décalée d'un mois (sa publication), recalculée à partir de l'indice ; "
                                 "l'IPC n'est pas une variable du modèle"}

    # VIX : l'export Bloomberg a été converti en euros, ce qui n'a pas de sens pour un indice de volatilité
    vix = _serie_bloomberg("Monthly_VIX_HP.xlsx")
    vix_m = pd.Series(vix.to_numpy(), index=vix.index.to_period("M"))
    table_vix = pd.Series(mensuel["vix"].to_numpy(), index=mois)
    ratio = (vix_m / table_vix.reindex(vix_m.index)).dropna()
    rapport["vix"] = {"mois_compares": int(len(ratio)), "ratio_median_bloomberg_sur_vix": float(ratio.median()),
                      "ratio_min": float(ratio.min()), "ratio_max": float(ratio.max()),
                      "decision": "non utilisé : export « EUR, as exported », converti au taux de change ; le VIX en points (Cboe) de la table est gardé"}

    # Calendrier Bloomberg des publications (export limité à 5 000 lignes : 1999-2006)
    cal = pd.read_excel(ROOT / CFG["data"]["bloomberg"] / "calendrier_chiffres_1999_2006.xlsx")
    cal["quand"] = pd.to_datetime(cal["Date Time"], errors="coerce", format="mixed")
    publications = {}
    for ticker, nom in [("JNTSMFG Index", "tankan"), ("JNCPIYOY Index", "ipc")]:
        c = cal[cal["Ticker"] == ticker].dropna(subset=["quand"])
        publications[nom] = {"publications": int(len(c)), "premiere": str(c["quand"].min().date()), "derniere": str(c["quand"].max().date()),
                             "jour_du_mois_median": float(c["quand"].dt.day.median()),
                             "publiees_en_decembre": int((c["quand"].dt.month == 12).sum())}
    rapport["calendrier"] = {"lignes": int(len(cal)), "couvre": f"{cal['quand'].min().date()} → {cal['quand'].max().date()}",
                             "limite": "export tronqué à 5 000 lignes : à refaire par périodes pour couvrir 2007-2026",
                             "publications": publications}
    return rapport


# ---------- 3. la décision, règle du notebook 03 ----------
def decider() -> dict:
    table = charger_table()
    train, valid = table[table["bloc"] == "entraînement"], table[table["bloc"] == "validation"]
    pred = pd.read_csv(RUNS / "predictions_02.csv", parse_dates=["date"])
    nik = pred[pred["cible"] == "nikkei"]
    platt, calibre_va = {}, {}
    for m in MODELES:
        hf = nik[(nik["modele"] == m) & (nik["bloc"] == "hors fold")]
        platt[m] = apprendre_platt(hf["p"].to_numpy(), hf["y"].astype(int).to_numpy())
        va = nik[(nik["modele"] == m) & (nik["bloc"] == "validation")].set_index("date")["p"]
        calibre_va[m] = pd.Series(appliquer_platt(va.to_numpy(), platt[m]), index=va.index)
    calibre_va = pd.DataFrame(calibre_va)
    calibre_va["moyenne"] = calibre_va[MODELES].mean(axis=1)
    y_va = valid["y_baisse"].astype(int).reindex(calibre_va.index)
    pr = {m: float(average_precision_score(y_va, calibre_va[m])) for m in calibre_va}
    ll = {m: float(log_loss(y_va, calibre_va[m], labels=[0, 1])) for m in calibre_va}
    candidats = [m for m in ["rf", "lgbm", "moyenne"] if pr[m] > pr["logit"] and ll[m] < ll["logit"]]
    choix = "logit" if not candidats else min(candidats, key=lambda m: ll[m])
    ll_const = float(log_loss(y_va, np.full(len(y_va), train["y_baisse"].mean()), labels=[0, 1]))
    couts = couts_entrainement(train)
    return {"date": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(), "configuration": CONFIG.name,
            "modele_nikkei": choix, "fichiers_modeles": {m: f"{RUNS.relative_to(ROOT)}/models/nikkei_{m}.joblib" for m in MODELES},
            "sortie_du_modele": "P(baisse) ; p_hausse = 1 - P(baisse) calibrée", "platt": platt,
            "couts_entrainement_exces_du_cash": couts, "marge_m": MARGE, "seuils": seuils(couts, MARGE),
            "couverture": "toujours couvert (h = 1)",
            "regle_choix_modele": "la logistique sauf si un modèle plus complexe la bat en PR-AUC et en log-loss calibrée sur la validation",
            "validation": {"pr_auc": pr, "log_loss_calibree": ll, "log_loss_constante": ll_const,
                           "plancher_battu": bool(ll[choix] < ll_const)},
            "mois_entrainement": int(len(train)), "premier_mois": str(train.index.min().date())}


# ---------- 4. l'évaluation, format du notebook 03b ----------
def _fiabilite(y, p, groupes=5):
    g = pd.qcut(pd.Series(p).rank(method="first"), q=groupes, labels=False)
    d = pd.DataFrame({"y": np.asarray(y), "p": np.asarray(p), "g": g.values})
    t = d.groupby("g").agg(conf=("p", "mean"), acc=("y", "mean"), n=("y", "size"))
    return float((t["n"] / t["n"].sum() * (t["acc"] - t["conf"]).abs()).sum())


def _scores(y, p) -> dict:
    return {"log-loss": float(log_loss(y, p, labels=[0, 1])), "Brier": float(brier_score_loss(y, p)), "ECE": _fiabilite(y, p),
            "PR-AUC": float(average_precision_score(y, p)), "proba moyenne": float(np.mean(p)), "fréquence observée": float(np.mean(y))}


def _perf(r) -> dict:
    w = np.cumprod(1 + np.asarray(r))
    return {"cumul": float(w[-1] - 1), "annualisé": float(w[-1] ** (12 / len(w)) - 1),
            "perte maximale": float(np.min(w / np.maximum.accumulate(w)) - 1)}


def _probas(dec: dict, bloc: pd.DataFrame) -> pd.DataFrame:
    p = {m: appliquer_platt(joblib.load(ROOT / dec["fichiers_modeles"][m]).predict_proba(bloc[COLONNES])[:, 1], dec["platt"][m])
         for m in MODELES}
    p = pd.DataFrame(p, index=bloc.index)
    p["moyenne"] = p[MODELES].mean(axis=1)
    return p


def _historique(dec: dict, bloc: pd.DataFrame, p_baisse: pd.Series) -> list:
    p_hausse = 1 - p_baisse
    act = actions(p_hausse, dec["couts_entrainement_exces_du_cash"], dec["marge_m"])
    r = rendements(act, bloc)
    cash = bloc["eur_overnight"] / 100 / 12
    return [{"date": d.strftime("%Y-%m"), "p_hausse": round(float(p_hausse[d]), 6), "action": act[d],
             "hausse_realisee": int(bloc.loc[d, "y_baisse"] == 0), "rendement_regle": round(float(r[d]), 6),
             "rendement_achat_couvert": round(float(bloc.loc[d, "target_reur_hedged_next"]), 6),
             "rendement_achat_non_couvert": round(float(bloc.loc[d, "target_reur_unhedged_next"]), 6),
             "rendement_vente": round(float(bloc.loc[d, "target_short_eur_next"]), 6), "rendement_cash": round(float(cash[d]), 6)}
            for d in bloc.index]


def evaluer(dec: dict) -> dict:
    table = charger_table()
    train, valid, test = (table[table["bloc"] == b] for b in ["entraînement", "validation", "test"])
    y = test["y_baisse"].astype(int)
    p = _probas(dec, test)
    p_const = float(train["y_baisse"].mean())
    metriques = {NOMS[m]: _scores(y, p[m]) for m in MODELES}
    if dec["modele_nikkei"] == "moyenne":
        metriques[NOMS["moyenne"]] = _scores(y, p["moyenne"])
    metriques["constante (fréquence de l'entraînement)"] = _scores(y, np.full(len(y), p_const))
    p_choisi = p[dec["modele_nikkei"]]
    hist = _historique(dec, test, p_choisi)
    act = pd.Series([h["action"] for h in hist], index=test.index)
    r_regle = pd.Series([h["rendement_regle"] for h in hist], index=test.index)
    cash = test["eur_overnight"] / 100 / 12
    expo = act.map({"acheter": 1, "neutre": 0, "vendre": -1}).mean()
    perf = {"règle du modèle": _perf(r_regle), "toujours acheter, couvert": _perf(test["target_reur_hedged_next"]),
            "toujours acheter, non couvert": _perf(test["target_reur_unhedged_next"]), "toujours neutre (cash)": _perf(cash),
            f"même exposition moyenne ({expo:.0%}), sans timing": _perf(cash + expo * (test["target_reur_hedged_next"] - cash))}
    ll_m = lambda q, yy: -(yy * np.log(q) + (1 - yy) * np.log(1 - q))
    ecarts = ll_m(np.clip(p_choisi.to_numpy(), 1e-6, 1 - 1e-6), y.to_numpy()) - ll_m(p_const, y.to_numpy())
    ll_choisi, ll_const = metriques[NOMS[dec["modele_nikkei"]]]["log-loss"], metriques["constante (fréquence de l'entraînement)"]["log-loss"]
    return {"date": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(), "configuration": CONFIG.name,
            "avertissement": "Seconde version, après l'ouverture du test 2024-2026 par la première : exploration après coup, pas un test à l'aveugle.",
            "mois_entrainement": int(len(train)), "mois_test": int(len(test)), "baisses_test": int(y.sum()),
            "modele": NOMS[dec["modele_nikkei"]], "metriques": metriques, "actions": act.value_counts().to_dict(),
            "performance_euros": perf, "plancher_battu": bool(ll_choisi < ll_const),
            "ecart_log_loss": {"moyenne": float(ecarts.mean()), "ecart_type": float(ecarts.std(ddof=1)),
                               "erreur_type": float(ecarts.std(ddof=1) / np.sqrt(len(ecarts)))},
            "historique_test": hist,
            "historique_validation": _historique(dec, valid, _probas(dec, valid)[dec["modele_nikkei"]])}


def main(n_essais: int, regler: bool) -> None:
    assert RUNS.name == "v2_bloomberg", "lancer avec NIKKEI_CONFIG=config_bloomberg.yaml : la première version ne doit pas être réécrite"
    RUNS.mkdir(parents=True, exist_ok=True)
    rapport = controle_bloomberg()
    (RUNS / "controle_bloomberg.json").write_text(json.dumps(rapport, indent=2, ensure_ascii=False))
    print(f"Tankan : {rapport['tankan']['valeurs_identiques']}/{rapport['tankan']['trimestres_compares']} trimestres identiques ; "
          f"VIX Bloomberg / VIX : {rapport['vix']['ratio_median_bloomberg_sur_vix']:.3f} (non utilisé)")
    if regler:
        from .tune import main as regler_optuna
        regler_optuna(n_essais, RUNS)
    dec = decider()
    (RUNS / "decision_finale.json").write_text(json.dumps(dec, indent=2, ensure_ascii=False))
    print(f"décision : {NOMS[dec['modele_nikkei']]} ; validation {dec['validation']['log_loss_calibree'][dec['modele_nikkei']]:.3f} "
          f"contre {dec['validation']['log_loss_constante']:.3f} pour la constante ; seuils {dec['seuils']}")
    ev = evaluer(dec)
    (RUNS / "evaluation_test.json").write_text(json.dumps(ev, indent=2, ensure_ascii=False))
    m, constante = ev["metriques"], "constante (fréquence de l'entraînement)"
    print(f"test : log-loss {m[ev['modele']]['log-loss']:.3f} contre {m[constante]['log-loss']:.3f} "
          f"pour la constante ; écart {ev['ecart_log_loss']['moyenne']:+.3f} ± {ev['ecart_log_loss']['erreur_type']:.3f} ; "
          f"règle {ev['performance_euros']['règle du modèle']['cumul']:+.1%} ; actions {ev['actions']}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--essais", type=int, default=50)
    parser.add_argument("--sans-reglage", action="store_true", help="réutiliser les modèles déjà réglés de runs/v2_bloomberg/")
    args = parser.parse_args()
    main(args.essais, not args.sans_reglage)

**Contrôle Bloomberg (`runs/v2_bloomberg/controle_bloomberg.json`)**

| série | comparaison | verdict |
|:--|:--|:--|
| Tankan | 112 trimestres, 112 valeurs identiques | utilisé à partir du mois suivant sa date Bloomberg ; pris tel quel, 99 trimestres auraient utilisé l'enquête avant sa publication |
| IPC | 202 mois, écart moyen 0,05 pt (max 0,42) | même série une fois décalée d'un mois ; l'IPC n'est pas une variable du modèle |
| VIX | 334 mois, ratio médian Bloomberg / VIX = 0,86 (0,60 à 1,24) | export Bloomberg écarté (converti en EUR) ; le VIX en points (Cboe) est gardé |
| Calendrier | 5 002 lignes, 1999-01-13 → 2006-12-27 | export tronqué à 5 000 lignes, à refaire par périodes pour 2007-2026 |

### 4.2 · Améliorations sans surapprentissage (V3)

Le fichier fige d'abord un protocole (variantes, réglages, périodes, critères) avec son empreinte SHA-256, puis réentraîne chaque variante mois après mois sur 2010-2023. Une variante n'est retenue que si elle bat la référence au test de Diebold-Mariano corrigé par Holm **et** sur les deux sous-périodes.

**Résultat : 0 levier retenu sur 17.**

> **Concepts** · walk-forward, Diebold-Mariano, Holm, bootstrap par blocs du Sharpe, protocole figé.

In [ ]:
# ─── src/ameliorations.py · 375 lignes ───

"""Troisième étape, après coup : appliquer les leviers d'amélioration sans surapprendre.

Le protocole (variantes, réglages, périodes, critères) est écrit dans runs/v3_ameliorations/protocole.json
AVANT tout calcul de résultat, avec son empreinte SHA-256. L'évaluation refuse de tourner si le protocole
du code ne correspond plus à celui qui a été figé.

Principes contre le surapprentissage :
  - chaque variante est réentraînée chaque mois sur tous les mois connus depuis 1999 (fenêtre croissante) ;
  - tous les réglages sont fixés à l'avance, aucun réglage automatique (pas d'Optuna) ;
  - la période de jugement est 2010-2023 ; une variante n'est retenue que si elle bat la référence
    après correction de Holm ET sur les deux sous-périodes 2010-2016 et 2017-2023 ;
  - 2024-2026 a déjà été ouvert par la V1 et la V2 : affiché à titre indicatif, jamais utilisé pour choisir.

Usage :
  python -m src.ameliorations --figer     écrit le protocole (refusé s'il existe déjà)
  python -m src.ameliorations             calcule tout, après avoir vérifié le protocole figé
"""
import argparse
import hashlib
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import yaml
from joblib import Parallel, delayed
from scipy.stats import norm
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .config import COLONNES, ROOT, SEED, commit_courant
from .decision import MARGE, actions, couts_entrainement
from .pipeline import construire_pipeline

V2 = yaml.safe_load(open(ROOT / "config_bloomberg.yaml"))
DOSSIER = ROOT / "runs" / "v3_ameliorations"
FICHIER_PROTOCOLE = DOSSIER / "protocole.json"

# Six variables choisies pour une raison économique connue avant les données, pas pour leur score :
ECONOMIQUES = ["n225_mom_12m",      # momentum 12 mois (Moskowitz, Ooi et Pedersen, 2012)
               "jgb_slope_10y2y",   # pente des taux japonais 10 ans - 2 ans (Fama et French, 1989)
               "us_slope_10y3m",    # pente des taux américains, indicateur avancé du cycle mondial
               "d_cli_jpn",         # variation de l'indicateur avancé OCDE du Japon
               "vix",               # appétit pour le risque mondial
               "n225_vol_ewma"]     # volatilité récente du Nikkei (arbitrage rendement-risque)
AJOUTEES = ["jgb_slope_10y2y", "us_slope_10y3m", "d_cli_jpn"]       # prises dans dataset_monthly2.csv

C_PROJET, C_FORT, ALPHA_RIDGE = 0.03, 0.003, 100.0
VOL_CIBLE, FRAIS, MELANGE = 0.15, 0.0005, 0.5
TIRAGES, BLOC, SEUIL = 5000, 12, 0.05

PERIODES = {"2010-2023 (jugement)": ("2010-01-01", "2023-12-31"),
            "2010-2016": ("2010-01-01", "2016-12-31"),
            "2017-2023": ("2017-01-01", "2023-12-31"),
            "2024-2026 (déjà vu, indicatif)": ("2024-01-01", "2026-09-30")}
JUGEMENT, SOUS_PERIODES, DEJA_VU = "2010-2023 (jugement)", ["2010-2016", "2017-2023"], "2024-2026 (déjà vu, indicatif)"

PROBABILITES = {
    "constante": "fréquence des baisses sur tout le passé connu (la référence)",
    "logistique du projet": f"13 variables du projet + régimes k-means, L2, C = {C_PROJET} (valeur retenue par la V1)",
    "logistique très pénalisée": f"mêmes variables, C = {C_FORT} : dix fois plus tirée vers la constante",
    "logistique du projet, mélangée à la constante": f"moyenne {MELANGE:.0%} logistique du projet, {1 - MELANGE:.0%} constante",
    "logistique économique": f"6 variables économiques, sans news ni régimes, L2, C = {C_PROJET}",
    "logistique économique, mélangée à la constante": f"moyenne {MELANGE:.0%} logistique économique, {1 - MELANGE:.0%} constante",
    "ridge sur le rendement": f"régression ridge (alpha = {ALPHA_RIDGE:g}) du rendement suivant sur les 6 variables économiques, "
                              "puis P(baisse) = Φ(-rendement prévu / volatilité récente mensuelle)",
}
LEVIERS = {
    "achat couvert, toujours": "la référence : acheter le Nikkei chaque mois, change couvert",
    "achat non couvert": "acheter sans couvrir le yen",
    "couverture à moitié": "acheter, couvrir la moitié du yen",
    "couvert seulement si le carry est positif": "couvrir quand la couverture rapporte (carry > 0), sinon ne pas couvrir",
    "achat couvert, dosé par la volatilité": f"part investie = min(1, {VOL_CIBLE:.0%} / volatilité récente), le reste au cash",
}
REFERENCE_EUROS = "achat couvert, toujours"

PROTOCOLE = {
    "question": "Quels leviers améliorent la prévision du mois suivant et la règle en euros, sans surapprendre ?",
    "donnees": "data/3_modele/dataset_model2.csv (1999-2026, Bloomberg en référence) + " + ", ".join(AJOUTEES)
               + " de data/2_construites/dataset_monthly2.csv",
    "reentrainement": "chaque mois, sur tous les mois connus depuis janvier 1999 (fenêtre croissante)",
    "reglages": "fixés ici, avant tout calcul ; aucun réglage automatique",
    "variables_economiques": ECONOMIQUES,
    "periodes": PERIODES,
    "periode_de_jugement": JUGEMENT,
    "probabilites": PROBABILITES,
    "critere_probabilites": "log-loss mensuelle comparée à la constante, mois par mois (test de Diebold-Mariano unilatéral) ; "
                            f"correction de Holm sur les variantes ; retenue si p corrigée < {SEUIL} et log-loss plus basse "
                            "que la constante sur 2010-2016 et sur 2017-2023",
    "regle_de_decision": f"celle du projet : gain espéré au-dessus du cash comparé à la marge {MARGE}, coûts appris sur le passé connu",
    "leviers_euros": LEVIERS | {f"règle : {v}": "acheter, neutre ou vendre selon la probabilité de cette variante" for v in PROBABILITES},
    "reference_euros": REFERENCE_EUROS,
    "critere_euros": f"ratio de Sharpe annualisé (rendement au-dessus du cash / écart-type) ; bootstrap par blocs de {BLOC} mois, "
                     f"{TIRAGES} tirages, graine {SEED} ; Holm sur les leviers ; retenu si p corrigée < {SEUIL} et Sharpe plus haut "
                     "que la référence sur 2010-2016 et sur 2017-2023",
    "frais": f"{FRAIS:.2%} par 100 % de position achetée ou vendue, pour toutes les stratégies",
    "assemblage_final": "les leviers retenus sont combinés (une seule politique de couverture : celle au meilleur Sharpe 2010-2023) ; "
                        "si aucun n'est retenu, on garde la référence : probabilité constante et achat couvert",
    "deja_vu": "2024-2026 a été ouvert par la V1 et la V2 : affiché à titre indicatif, jamais utilisé pour choisir",
    "ce_que_l_auteur_savait": "les explorations après coup sur 2016-2023 (runs/exploration_apres_coup.json) montraient déjà qu'aucun "
                              "modèle ne battait nettement la constante, et C = 0,03 est la valeur réglée par la V1 sur son "
                              "entraînement (qui recouvre 2010-2021) ; les autres choix sont des valeurs standards de la littérature",
}


def empreinte(protocole: dict) -> str:
    return hashlib.sha256(json.dumps(protocole, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def figer() -> None:
    if FICHIER_PROTOCOLE.exists():
        raise SystemExit(f"{FICHIER_PROTOCOLE.relative_to(ROOT)} existe déjà : un protocole figé ne se réécrit pas")
    DOSSIER.mkdir(parents=True, exist_ok=True)
    FICHIER_PROTOCOLE.write_text(json.dumps({"fige_le": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "commit_du_code": commit_courant(),
                                             "sha256": empreinte(PROTOCOLE), "protocole": PROTOCOLE}, indent=2, ensure_ascii=False))
    print(f"protocole figé : {FICHIER_PROTOCOLE.relative_to(ROOT)} (sha256 {empreinte(PROTOCOLE)[:12]}…)")


def verifier_protocole() -> dict:
    if not FICHIER_PROTOCOLE.exists():
        raise SystemExit("aucun protocole figé : lancer d'abord python -m src.ameliorations --figer")
    fige = json.loads(FICHIER_PROTOCOLE.read_text())
    if fige["sha256"] != empreinte(PROTOCOLE) or fige["protocole"] != json.loads(json.dumps(PROTOCOLE, ensure_ascii=False)):
        raise SystemExit("le protocole du code a changé depuis qu'il a été figé : évaluation refusée")
    return fige


# ---------- 1. les données ----------
def charger() -> pd.DataFrame:
    t = pd.read_csv(ROOT / V2["data"]["model_table"], parse_dates=["date"]).set_index("date")
    m = pd.read_csv(ROOT / V2["data"]["monthly"], parse_dates=["date"]).set_index("date")
    for c in AJOUTEES:
        t[c] = m[c].reindex(t.index)
    assert t[ECONOMIQUES].notna().all().all(), "une variable économique manque"
    return t


# ---------- 2. les probabilités, mois après mois ----------
def _simple(modele) -> Pipeline:
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", modele)])


def marcher(nom: str, table: pd.DataFrame, debut: int) -> tuple[pd.Series, pd.DataFrame]:
    """Probabilité de baisse du mois suivant, chaque mois t appris sur les mois strictement antérieurs."""
    warnings.filterwarnings("ignore")
    p, coefs = {}, {}
    for t in range(debut, len(table)):
        passe, ligne, date = table.iloc[:t], table.iloc[[t]], table.index[t]
        y = passe["y_baisse"].astype(int)
        if nom == "constante":
            q = y.mean()
        elif nom == "ridge sur le rendement":
            pipe = _simple(Ridge(alpha=ALPHA_RIDGE)).fit(passe[ECONOMIQUES], passe["target_rn_next"])
            mu, sigma = pipe.predict(ligne[ECONOMIQUES])[0], ligne["n225_vol_ewma"].iloc[0] / np.sqrt(12)
            q = norm.cdf(-mu / sigma)
            coefs[date] = dict(zip(ECONOMIQUES, pipe[-1].coef_)) | {"constante": pipe[-1].intercept_}
        elif nom == "logistique économique":
            pipe = _simple(LogisticRegression(C=C_PROJET, max_iter=5000)).fit(passe[ECONOMIQUES], y)
            q = pipe.predict_proba(ligne[ECONOMIQUES])[0, 1]
            coefs[date] = dict(zip(ECONOMIQUES, pipe[-1].coef_[0])) | {"constante": pipe[-1].intercept_[0]}
        else:
            C = C_PROJET if nom == "logistique du projet" else C_FORT
            pipe = construire_pipeline(LogisticRegression(C=C, max_iter=5000)).fit(passe[COLONNES], y)
            q = pipe.predict_proba(ligne[COLONNES])[0, 1]
        p[date] = float(np.clip(q, 1e-4, 1 - 1e-4))
    return pd.Series(p, name=nom), pd.DataFrame(coefs).T.assign(modele=nom)


def probabilites(table: pd.DataFrame, debut: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    appris = [v for v in PROBABILITES if "mélangée" not in v]
    sorties = Parallel(n_jobs=-1)(delayed(marcher)(v, table, debut) for v in appris)
    p = pd.concat([s for s, _ in sorties], axis=1)
    for base in ["logistique du projet", "logistique économique"]:
        p[f"{base}, mélangée à la constante"] = MELANGE * p[base] + (1 - MELANGE) * p["constante"]
    coefs = pd.concat([c for _, c in sorties if len(c)]).rename_axis("date")
    return p[list(PROBABILITES)], coefs


# ---------- 3. les stratégies en euros ----------
def actions_regle(p_baisse: pd.Series, table: pd.DataFrame) -> pd.Series:
    """La règle du projet, mois par mois, avec les coûts appris sur le passé connu à ce mois."""
    sortie = {}
    for date, p in p_baisse.items():
        couts = couts_entrainement(table.iloc[:table.index.get_loc(date)])
        sortie[date] = actions(pd.Series([1 - p], index=[date]), couts, MARGE).iloc[0]
    return pd.Series(sortie)


def strategie(table: pd.DataFrame, sens: pd.Series, couvert: pd.Series, poids: pd.Series) -> pd.Series:
    """Rendement en euros du mois suivant ; sens dans {1, 0, -1}, part couverte et part investie dans [0, 1]."""
    cash = table["eur_overnight"] / 100 / 12
    achat = couvert * table["target_reur_hedged_next"] + (1 - couvert) * table["target_reur_unhedged_next"]
    brut = pd.Series(np.select([sens > 0, sens < 0], [achat, table["target_short_eur_next"]], cash), index=table.index)
    position = sens * poids
    return poids * brut + (1 - poids) * cash - FRAIS * position.diff().abs().fillna(0)


def leviers(table: pd.DataFrame, regles: dict) -> pd.DataFrame:
    un = pd.Series(1.0, index=table.index)
    carry_positif = (table["carry_1m"] > 0).astype(float)
    dose = (VOL_CIBLE / table["n225_vol_ewma"]).clip(upper=1.0)
    r = {"achat couvert, toujours": strategie(table, un, un, un),
         "achat non couvert": strategie(table, un, 0 * un, un),
         "couverture à moitié": strategie(table, un, 0.5 * un, un),
         "couvert seulement si le carry est positif": strategie(table, un, carry_positif, un),
         "achat couvert, dosé par la volatilité": strategie(table, un, un, dose)}
    for nom, act in regles.items():
        r[f"règle : {nom}"] = strategie(table, act.map({"acheter": 1, "neutre": 0, "vendre": -1}).astype(float), un, un)
    return pd.DataFrame(r)


# ---------- 4. les tests ----------
def perte(y, p) -> np.ndarray:
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


def holm(p: dict) -> dict:
    """p-valeurs corrigées de Holm (contrôle du risque de se tromper au moins une fois)."""
    ordre, m, sortie, courant = sorted(p, key=p.get), len(p), {}, 0.0
    for i, cle in enumerate(ordre):
        courant = max(courant, min(1.0, (m - i) * p[cle]))
        sortie[cle] = courant
    return sortie


def sharpe(exces) -> float:
    exces = np.asarray(exces, dtype=float)
    s = exces.std(ddof=1)
    return float(exces.mean() / s * np.sqrt(12)) if s > 0 else 0.0


def bootstrap_sharpe(a, b) -> dict:
    """Écart de Sharpe a - b, bootstrap circulaire par blocs (mêmes mois tirés pour a et b)."""
    a, b, n = np.asarray(a, dtype=float), np.asarray(b, dtype=float), len(a)
    rng = np.random.default_rng(SEED)
    debuts = rng.integers(0, n, size=(TIRAGES, int(np.ceil(n / BLOC))))
    idx = ((debuts[:, :, None] + np.arange(BLOC)) % n).reshape(TIRAGES, -1)[:, :n]

    def s(x):
        e = x.std(axis=1, ddof=1)
        return np.where(e > 0, x.mean(axis=1) / np.where(e > 0, e, 1), 0.0) * np.sqrt(12)

    ecarts = s(a[idx]) - s(b[idx])
    return {"ecart": sharpe(a) - sharpe(b), "ic_90": [float(np.quantile(ecarts, 0.05)), float(np.quantile(ecarts, 0.95))],
            "p_unilaterale": float(np.mean(ecarts <= 0))}


def bilan_proba(y, p, ref) -> dict:
    ecart = perte(y, p) - perte(y, ref)
    se = ecart.std(ddof=1) / np.sqrt(len(ecart))
    return {"mois": int(len(y)), "log_loss": float(perte(y, p).mean()), "ecart_vs_constante": float(ecart.mean()),
            "erreur_type": float(se), "p_unilaterale": float(norm.cdf(ecart.mean() / se)) if se > 0 else 1.0,
            "brier": float(brier_score_loss(y, p)), "auc": float(roc_auc_score(y, p)) if len(set(p)) > 1 else 0.5,
            "pr_auc": float(average_precision_score(y, p)), "proba_moyenne": float(np.mean(p)), "frequence_observee": float(np.mean(y))}


def bilan_euros(r: pd.Series, cash: pd.Series) -> dict:
    valeur = (1 + r).cumprod()
    return {"mois": int(len(r)), "cumul": float(valeur.iloc[-1] - 1), "annualise": float(valeur.iloc[-1] ** (12 / len(r)) - 1),
            "volatilite": float(r.std(ddof=1) * np.sqrt(12)), "sharpe": sharpe(r - cash),
            "perte_maximale": float((valeur / valeur.cummax() - 1).min()), "pire_mois": float(r.min())}


# ---------- 5. tout assembler ----------
def main() -> None:
    fige = verifier_protocole()
    table = charger()
    debut = table.index.get_loc(table.index[table.index >= PERIODES[JUGEMENT][0]][0])
    p, coefs = probabilites(table, debut)
    connus = table.loc[p.index, "y_baisse"].notna()
    regles = {v: actions_regle(p[v], table) for v in PROBABILITES}
    r = leviers(table.loc[p.index], regles)
    cash = table.loc[p.index, "eur_overnight"] / 100 / 12
    mois = {k: p.index[connus & (p.index >= a) & (p.index <= b)] for k, (a, b) in PERIODES.items()}

    # Probabilités
    res_p = {}
    for per, idx in mois.items():
        y = table.loc[idx, "y_baisse"].astype(int).to_numpy()
        res_p[per] = {v: bilan_proba(y, p.loc[idx, v].to_numpy(), p.loc[idx, "constante"].to_numpy()) for v in PROBABILITES}
    candidats = [v for v in PROBABILITES if v != "constante"]
    p_holm = holm({v: res_p[JUGEMENT][v]["p_unilaterale"] for v in candidats})
    tests_p = {v: {"ecart_vs_constante": res_p[JUGEMENT][v]["ecart_vs_constante"], "erreur_type": res_p[JUGEMENT][v]["erreur_type"],
                   "p_unilaterale": res_p[JUGEMENT][v]["p_unilaterale"], "p_holm": p_holm[v],
                   "mieux_sur": {s: bool(res_p[s][v]["ecart_vs_constante"] < 0) for s in SOUS_PERIODES}} for v in candidats}
    for v, t in tests_p.items():
        t["retenue"] = bool(t["p_holm"] < SEUIL and all(t["mieux_sur"].values()))

    # Euros
    res_e = {per: {s: bilan_euros(r.loc[idx, s], cash[idx]) for s in r.columns} for per, idx in mois.items()}
    ref = r[REFERENCE_EUROS] - cash
    autres = [s for s in r.columns if s != REFERENCE_EUROS]
    boot = {s: bootstrap_sharpe((r[s] - cash)[mois[JUGEMENT]], ref[mois[JUGEMENT]]) for s in autres}
    e_holm = holm({s: boot[s]["p_unilaterale"] for s in autres})
    tests_e = {s: boot[s] | {"p_holm": e_holm[s],
                             "mieux_sur": {sp: bool(res_e[sp][s]["sharpe"] > res_e[sp][REFERENCE_EUROS]["sharpe"]) for sp in SOUS_PERIODES}}
               for s in autres}
    for s, t in tests_e.items():
        t["retenu"] = bool(t["p_holm"] < SEUIL and all(t["mieux_sur"].values()))

    # Assemblage final : seulement ce qui a passé les deux critères
    proba_retenues = [v for v in candidats if tests_p[v]["retenue"]]
    proba = min(proba_retenues, key=lambda v: res_p[JUGEMENT][v]["log_loss"]) if proba_retenues else "constante"
    couvertures = [s for s in ["achat non couvert", "couverture à moitié", "couvert seulement si le carry est positif"] if tests_e[s]["retenu"]]
    couverture = max(couvertures, key=lambda s: res_e[JUGEMENT][s]["sharpe"]) if couvertures else REFERENCE_EUROS
    dosage = tests_e["achat couvert, dosé par la volatilité"]["retenu"]
    regle = tests_e.get(f"règle : {proba}", {}).get("retenu", False) and proba != "constante"
    t = table.loc[p.index]
    part_couverte = {"achat non couvert": 0.0 * t["carry_1m"], "couverture à moitié": 0.5 + 0 * t["carry_1m"],
                     "couvert seulement si le carry est positif": (t["carry_1m"] > 0).astype(float)}.get(couverture, 1 + 0 * t["carry_1m"])
    poids = (VOL_CIBLE / t["n225_vol_ewma"]).clip(upper=1.0) if dosage else 1 + 0 * t["carry_1m"]
    sens = regles[proba].map({"acheter": 1, "neutre": 0, "vendre": -1}).astype(float) if regle else 1 + 0 * t["carry_1m"]
    final = strategie(t, sens, part_couverte, poids)
    r["assemblage final"] = final
    for per, idx in mois.items():
        res_e[per]["assemblage final"] = bilan_euros(final[idx], cash[idx])

    derniere = p.index[-1]
    prochain = {"donnees_arretees_au": str(derniere.date()), "mois_prevu": (derniere.to_period("M") + 1).strftime("%Y-%m"),
                "probabilites_de_baisse": {v: float(p.loc[derniere, v]) for v in PROBABILITES},
                "assemblage": {"probabilite_utilisee": proba, "p_baisse": float(p.loc[derniere, proba]),
                               "sens": {1.0: "acheter", 0.0: "neutre", -1.0: "vendre"}[float(sens[derniere])],
                               "part_couverte": float(part_couverte[derniere]), "part_investie": float(poids[derniere]),
                               "volatilite_recente": float(t.loc[derniere, "n225_vol_ewma"]), "carry_1m": float(t.loc[derniere, "carry_1m"])}}

    derniers_coefs = {m: coefs[coefs["modele"] == m].drop(columns="modele").iloc[-1].dropna().to_dict()
                      for m in coefs["modele"].unique()}
    sortie = {
        "avertissement": "Troisième étape, après l'ouverture du test 2024-2026 par la V1 et la V2. Le protocole a été figé avant le calcul "
                         "(protocole.json) ; 2024-2026 est montré à titre indicatif seulement.",
        "date": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(),
        "protocole": {"fichier": str(FICHIER_PROTOCOLE.relative_to(ROOT)), "sha256": fige["sha256"], "fige_le": fige["fige_le"]},
        "premiere_prevision": str(p.index[0].date()), "mois_par_periode": {k: int(len(v)) for k, v in mois.items()},
        "probabilites": res_p, "tests_probabilites": tests_p,
        "euros": res_e, "tests_euros": tests_e,
        "retenus": {"probabilites": proba_retenues, "euros": [s for s in autres if tests_e[s]["retenu"]]},
        "assemblage_final": {"probabilite": proba, "couverture": couverture, "dosage_par_la_volatilite": bool(dosage),
                             "regle_de_decision": bool(regle)},
        "mois_prochain": prochain,
        "coefficients_derniers": derniers_coefs,
    }
    DOSSIER.mkdir(parents=True, exist_ok=True)
    (DOSSIER / "resultats.json").write_text(json.dumps(sortie, indent=2, ensure_ascii=False))
    p.join(pd.DataFrame({f"action · {v}": a for v, a in regles.items()})).join(table[["y_baisse", "bloc"]]) \
        .round(6).rename_axis("date").to_csv(DOSSIER / "probabilites.csv")
    r.round(6).join(cash.rename("cash")).rename_axis("date").to_csv(DOSSIER / "rendements.csv")
    coefs.round(6).to_csv(DOSSIER / "coefficients.csv")

    print(f"protocole vérifié (sha256 {fige['sha256'][:12]}…, figé le {fige['fige_le']})")
    print(f"\nProbabilités, {JUGEMENT} — log-loss et écart à la constante (p de Holm) :")
    for v in PROBABILITES:
        b = res_p[JUGEMENT][v]
        extra = "" if v == "constante" else f"  {b['ecart_vs_constante']:+.4f} ± {b['erreur_type']:.4f}  p Holm {tests_p[v]['p_holm']:.2f}" \
                                            f"  {'RETENUE' if tests_p[v]['retenue'] else ''}"
        print(f"  {v:48s} {b['log_loss']:.4f}{extra}")
    print(f"\nEuros, {JUGEMENT} — Sharpe, perte maximale, cumul (p de Holm) :")
    for s in r.columns:
        b = res_e[JUGEMENT][s]
        extra = f"  p Holm {tests_e[s]['p_holm']:.2f} {'RETENU' if tests_e[s]['retenu'] else ''}" if s in tests_e else ""
        print(f"  {s:60s} {b['sharpe']:+.2f}  {b['perte_maximale']:+.1%}  {b['cumul']:+.1%}{extra}")
    print(f"\nAssemblage : {sortie['assemblage_final']}")
    print(f"Indicatif 2024-2026 : assemblage {res_e[DEJA_VU]['assemblage final']['cumul']:+.1%}, "
          f"référence {res_e[DEJA_VU][REFERENCE_EUROS]['cumul']:+.1%}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--figer", action="store_true", help="écrire le protocole avant tout calcul")
    args = parser.parse_args()
    figer() if args.figer else main()

---
<a id="partie-5"></a>
## Partie 5 · Explorations après coup

Ces scripts répondent à des questions posées **après** le test. Ils ne remplacent pas le résultat figé.

### 5.1 · Bancs d'essai

| Fichier | Question |
|:--|:--|
| `banc_essai.py` | Beaucoup de modèles réentraînés mois après mois (constante, AR(1), lissage, tendance, logistique). |
| `banc_essai_jour.py` | La même question sur 4 118 séances journalières, avec les décalages horaires Tokyo/New York. |
| `banc_essai_long.py` | Apprendre depuis 2000 ou depuis 1970 aide-t-il ? (prix seulement) |
| `banc_essai_normalisation.py` | Standardiser sur une fenêtre glissante (24, 36, 60 mois) après le changement de régime. |
| `banc_essai_ouverture.py` | Une décision à l'ouverture de Tokyo sur l'ETF 1321, un produit réellement achetable. |

> **Concepts** · walk-forward, références naïves, calibration (ECE), extrapolation et standardisation glissante.

In [ ]:
# ─── src/banc_essai.py · 232 lignes ───

"""Exploration après coup : beaucoup de modèles, réentraînés mois après mois (walk-forward).

Ce script ne remplace pas le résultat du projet. Le test 2024-2026 a déjà été ouvert une fois
(runs/evaluation_test.json) : tout ce qui sort d'ici est une exploration faite en connaissant la suite.

Principe : pour prédire le mois t, chaque modèle n'apprend que sur les mois strictement antérieurs.
Deux fenêtres sont comparées : tout le passé (« croissante ») ou les 60 derniers mois (« glissante »).
Le choix du « meilleur » modèle se fait sur les mois d'avant le test (2016-2023) ; les 33 mois de test
servent ensuite à voir si ce choix tenait.

Usage : python -m src.banc_essai     écrit runs/exploration_apres_coup.json et ..._predictions.csv
"""
import json
import warnings
from datetime import datetime

import numpy as np
import optuna
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import norm, spearmanr
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss
from sklearn.model_selection import TimeSeriesSplit
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from .config import CFG, COLONNES, RUNS, SEED, commit_courant
from .data import charger_table
from .decision import MARGE, actions, couts_entrainement, rendements
from .pipeline import construire_pipeline, proposer

DEPART = 72            # mois d'historique exigés avant la première prédiction (février 2016)
GLISSANTE = 60         # largeur de la fenêtre glissante, en mois
SORTIE = RUNS / "exploration_apres_coup.json"
PREDICTIONS = RUNS / "exploration_apres_coup_predictions.csv"
PRIX = CFG["features"]["price"] + ["n225_ret_1m"]
SANS_NEWS = [c for c in COLONNES if not c.startswith("news_")]


def officiel(nom: str):
    """Modèle du projet, avec les hyperparamètres retenus par Optuna (runs/trials_log.csv)."""
    journal = pd.read_csv(RUNS / "trials_log.csv")
    ligne = journal[(journal["cible"] == "nikkei") & (journal["modele"] == nom) & journal["retenu"]].iloc[0]
    return proposer(optuna.trial.FixedTrial(json.loads(ligne["params"])), nom)


def simple(modele, colonnes=COLONNES) -> tuple:
    """Imputation et standardisation sur un sous-ensemble de colonnes, sans les régimes k-means."""
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", modele)]), colonnes


# Chaque entrée : nom lisible -> fonction qui renvoie (pipeline, colonnes). Tout est réajusté à chaque mois.
APPRIS = {
    "logistique L2 (modèle du projet)": lambda: (construire_pipeline(officiel("logit")), COLONNES),
    "logistique L1": lambda: (construire_pipeline(LogisticRegression(
        C=0.1, penalty="l1", solver="saga", max_iter=5000, random_state=SEED)), COLONNES),
    "logistique très pénalisée": lambda: (construire_pipeline(LogisticRegression(C=0.003, max_iter=2000)), COLONNES),
    "logistique sans les news": lambda: simple(LogisticRegression(C=0.03, max_iter=2000), SANS_NEWS),
    "logistique, prix seuls": lambda: simple(LogisticRegression(C=0.03, max_iter=2000), PRIX),
    "analyse discriminante linéaire": lambda: (construire_pipeline(LinearDiscriminantAnalysis(shrinkage="auto", solver="lsqr")), COLONNES),
    "Bayes naïf gaussien": lambda: (construire_pipeline(GaussianNB()), COLONNES),
    "15 plus proches voisins": lambda: (construire_pipeline(KNeighborsClassifier(n_neighbors=15)), COLONNES),
    "SVM à noyau gaussien": lambda: (construire_pipeline(SVC(C=0.5, probability=True, random_state=SEED)), COLONNES),
    "Random Forest (projet)": lambda: (construire_pipeline(officiel("rf")), COLONNES),
    "Extra Trees": lambda: (construire_pipeline(ExtraTreesClassifier(
        n_estimators=200, max_depth=3, min_samples_leaf=8, random_state=SEED, n_jobs=1)), COLONNES),
    "LightGBM (projet)": lambda: (construire_pipeline(officiel("lgbm")), COLONNES),
    "gradient boosting peu profond": lambda: (construire_pipeline(GradientBoostingClassifier(
        n_estimators=60, learning_rate=0.03, max_depth=2, subsample=0.8, random_state=SEED)), COLONNES),
    "réseau de neurones (8 neurones)": lambda: (construire_pipeline(MLPClassifier(
        hidden_layer_sizes=(8,), alpha=1.0, max_iter=3000, random_state=SEED)), COLONNES),
}


def p_constante(passe: pd.DataFrame, ligne: pd.DataFrame) -> float:
    """Fréquence des baisses observée dans la fenêtre."""
    return float(passe["y_baisse"].mean())


def p_ar1(passe: pd.DataFrame, ligne: pd.DataFrame) -> float:
    """Série temporelle classique : rendement suivant = a + b x rendement du mois, bruit gaussien."""
    x, y = passe["n225_ret_1m"].to_numpy(), passe["target_rn_next"].to_numpy()
    b, a = np.polyfit(x, y, 1)
    sigma = np.std(y - (a + b * x), ddof=2)
    return float(norm.cdf(-(a + b * ligne["n225_ret_1m"].iloc[0]) / sigma))


def p_lissage(passe: pd.DataFrame, ligne: pd.DataFrame) -> float:
    """Lissage exponentiel (demi-vie 12 mois) de la moyenne et de l'écart-type des rendements."""
    r = passe["target_rn_next"]
    mu, sigma = r.ewm(halflife=12).mean().iloc[-1], r.ewm(halflife=12).std().iloc[-1]
    return float(norm.cdf(-mu / sigma))


def p_tendance(passe: pd.DataFrame, ligne: pd.DataFrame) -> float:
    """Suivi de tendance : fréquence passée des baisses, selon que le momentum 12 mois est positif ou non."""
    meme_signe = (passe["n225_mom_12m"] > 0) == bool(ligne["n225_mom_12m"].iloc[0] > 0)
    y = passe.loc[meme_signe, "y_baisse"]
    return float((y.sum() + 1) / (len(y) + 2))            # lissage de Laplace : jamais 0 ni 1


def p_logit_regle(passe: pd.DataFrame, ligne: pd.DataFrame) -> float:
    """Logistique L2 dont la pénalité est rechoisie à chaque mois, par validation chronologique sur la fenêtre."""
    X, y = passe[COLONNES], passe["y_baisse"].astype(int)
    scores = {}
    for C in np.logspace(-3, 0, 7):
        pipe = construire_pipeline(LogisticRegression(C=C, max_iter=2000))
        scores[C] = np.mean([log_loss(y.iloc[b], pipe.fit(X.iloc[a], y.iloc[a]).predict_proba(X.iloc[b])[:, 1], labels=[0, 1])
                             for a, b in TimeSeriesSplit(n_splits=3).split(X) if y.iloc[a].nunique() == 2])
    pipe = construire_pipeline(LogisticRegression(C=min(scores, key=scores.get), max_iter=2000))
    return float(pipe.fit(X, y).predict_proba(ligne[COLONNES])[0, 1])


FORMULES = {"logistique L2, pénalité rechoisie chaque mois": p_logit_regle,
            "constante (fréquence passée)": p_constante, "AR(1) sur les rendements": p_ar1,
            "lissage exponentiel des rendements": p_lissage, "suivi de tendance 12 mois": p_tendance}


def marcher(nom: str, fenetre: str, table: pd.DataFrame) -> pd.DataFrame:
    """Prédictions walk-forward d'un modèle : une probabilité de baisse et une action par mois."""
    warnings.filterwarnings("ignore")
    lignes = []
    for t in range(DEPART, len(table)):
        debut = 0 if fenetre == "croissante" else max(0, t - GLISSANTE)
        passe, ligne = table.iloc[debut:t], table.iloc[[t]]
        if nom in FORMULES:
            p = FORMULES[nom](passe, ligne)
        else:
            pipe, colonnes = APPRIS[nom]()
            p = float(pipe.fit(passe[colonnes], passe["y_baisse"].astype(int)).predict_proba(ligne[colonnes])[0, 1])
        p = float(np.clip(p, 1e-4, 1 - 1e-4))
        couts = couts_entrainement(table.iloc[:t])            # coûts appris sur tout le passé connu, comme dans le projet
        action = actions(pd.Series([1 - p], index=ligne.index), couts, MARGE).iloc[0]
        lignes.append({"date": ligne.index[0], "modele": nom, "fenetre": fenetre, "p_baisse": p, "action": action})
    return pd.DataFrame(lignes)


def perte(y, p) -> np.ndarray:
    """Log-loss mois par mois."""
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


def ece(y, p, cases: int = 5) -> float:
    """Écart moyen entre probabilité annoncée et fréquence observée, sur des cases de largeur égale."""
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    case = np.minimum((p * cases).astype(int), cases - 1)
    return float(sum(abs(y[case == k].mean() - p[case == k].mean()) * (case == k).mean() for k in np.unique(case)))


def resumer(pred: pd.DataFrame, bloc: pd.DataFrame, reference: pd.Series) -> dict:
    """Métriques d'un modèle sur une période ; `reference` = log-loss mensuelle de la constante."""
    y, p = bloc["y_baisse"].astype(int).to_numpy(), pred["p_baisse"].to_numpy()
    ecart = perte(y, p) - reference.to_numpy()
    gains = rendements(pred["action"], bloc)
    valeur = np.cumprod(1 + gains.to_numpy())
    return {"mois": int(len(y)), "log_loss": float(perte(y, p).mean()),
            "ecart_vs_constante": float(ecart.mean()), "erreur_type_de_l_ecart": float(ecart.std(ddof=1) / np.sqrt(len(ecart))),
            "pr_auc": float(average_precision_score(y, p)), "brier": float(brier_score_loss(y, p)), "ece": ece(y, p),
            "actions": {a: int(n) for a, n in pred["action"].value_counts().items()},
            "cumul_regle": float(valeur[-1] - 1), "perte_maximale": float((valeur / np.maximum.accumulate(valeur) - 1).min()),
            "cumul_achat_couvert": float(np.prod(1 + bloc["target_reur_hedged_next"].to_numpy()) - 1)}


def main() -> None:
    table = charger_table()
    table = table[table["bloc"] != "à prédire"]
    plan = [(nom, f) for nom in list(FORMULES) + list(APPRIS) for f in ["croissante", "glissante"]]
    morceaux = Parallel(n_jobs=-1)(delayed(marcher)(nom, f, table) for nom, f in plan)
    pred = pd.concat(morceaux).set_index("date")

    # Moyenne des trois modèles du projet : l'ensemble le plus simple possible
    for f in ["croissante", "glissante"]:
        trio = pred[(pred["fenetre"] == f) & pred["modele"].isin(
            ["logistique L2 (modèle du projet)", "Random Forest (projet)", "LightGBM (projet)"])]
        moyenne = trio.groupby(level=0)["p_baisse"].mean()
        act = [actions(pd.Series([1 - p], index=[d]), couts_entrainement(table.loc[:d].iloc[:-1]), MARGE).iloc[0]
               for d, p in moyenne.items()]
        pred = pd.concat([pred, pd.DataFrame({"modele": "moyenne logistique + forêt + LightGBM", "fenetre": f,
                                              "p_baisse": moyenne, "action": act})])

    periodes = {"avant_test_2016_2023": table.index[DEPART:][(table["bloc"].iloc[DEPART:] != "test").to_numpy()],
                "test_2024_2026": table.index[table["bloc"] == "test"]}
    const = pred[(pred["modele"] == "constante (fréquence passée)") & (pred["fenetre"] == "croissante")]
    resultats = []
    for (nom, f), d in pred.groupby(["modele", "fenetre"], sort=False):
        ligne = {"modele": nom, "fenetre": f}
        for cle, mois in periodes.items():
            bloc = table.loc[mois]
            ref = pd.Series(perte(bloc["y_baisse"], const.loc[mois, "p_baisse"]), index=mois)
            ligne[cle] = resumer(d.loc[mois], bloc, ref)
        resultats.append(ligne)

    avant = [r["avant_test_2016_2023"]["log_loss"] for r in resultats]
    apres = [r["test_2024_2026"]["log_loss"] for r in resultats]
    choisi, gagnant_test = resultats[int(np.argmin(avant))], resultats[int(np.argmin(apres))]
    rho, _ = spearmanr(avant, apres)
    battent = lambda cle: sum(r[cle]["ecart_vs_constante"] < 0 for r in resultats if not r["modele"].startswith("constante"))
    sortie = {
        "avertissement": "Exploration après coup : le test 2024-2026 avait déjà été ouvert. Ces chiffres ne remplacent pas "
                         "runs/evaluation_test.json, qui reste le résultat du projet.",
        "date": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(),
        "methode": {"depart_mois": DEPART, "fenetre_glissante_mois": GLISSANTE, "reentrainement": "chaque mois",
                    "reference": "constante (fréquence passée), fenêtre croissante",
                    "regle": f"mêmes formules que le projet, marge {MARGE}, coûts appris sur le passé connu à chaque mois",
                    "probabilites": "brutes, sans calibration de Platt"},
        "configurations": len(resultats),
        "choix_sur_2016_2023": {"modele": choisi["modele"], "fenetre": choisi["fenetre"],
                                "avant_test": choisi["avant_test_2016_2023"], "test": choisi["test_2024_2026"]},
        "meilleur_sur_le_test_a_posteriori": {"modele": gagnant_test["modele"], "fenetre": gagnant_test["fenetre"],
                                              "avant_test": gagnant_test["avant_test_2016_2023"], "test": gagnant_test["test_2024_2026"]},
        "battent_la_constante": {"avant_test": battent("avant_test_2016_2023"), "test": battent("test_2024_2026"),
                                 "sur": len(resultats) - 2},
        "correlation_de_rang_avant_test_vs_test": float(rho),
        "resultats": resultats,
    }
    SORTIE.write_text(json.dumps(sortie, indent=2, ensure_ascii=False))
    pred.rename_axis("date").to_csv(PREDICTIONS)
    print(f"{len(resultats)} configurations ; choix sur 2016-2023 : {choisi['modele']} ({choisi['fenetre']})")
    print(f"écrit : {SORTIE.relative_to(RUNS.parent)} et {PREDICTIONS.relative_to(RUNS.parent)}")


if __name__ == "__main__":
    main()

In [ ]:
# ─── src/banc_essai_jour.py · 190 lignes ───

"""Exploration après coup, sur la base journalière (4 118 séances, 2010-2026).

Question : avec vingt fois plus de lignes que la table mensuelle, prédit-on mieux le sens du Nikkei ?
Ce script ne remplace pas le résultat du projet (runs/evaluation_test.json) ; le test a déjà été ouvert.

Précaution sur les horaires. Dans la base, la ligne du jour j contient la clôture de Wall Street du jour j,
connue plusieurs heures APRÈS la clôture de Tokyo. Tout ce qui ne vient pas de la Bourse de Tokyo est donc
décalé d'un jour quand la décision se prend à la clôture. Une expérience garde exprès Wall Street non décalé
pour montrer ce que ce piège rapporte, et une autre le rend honnête en décidant à l'ouverture du lendemain.

Usage : python -m src.banc_essai_jour     écrit runs/exploration_journaliere.json
"""
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .config import ROOT, RUNS, SEED, commit_courant

SORTIE = RUNS / "exploration_journaliere.json"
DEBUT = "2016-01-01"                 # première prédiction : six ans d'historique au moins
TEST = ("2024-01-01", "2026-09-30")  # mêmes mois que le test du projet
REFAIRE = 20                         # réentraînement toutes les 20 séances (un mois environ)
VOL_CIBLE = 0.15                     # volatilité annuelle visée par la stratégie de dosage


def charger() -> pd.DataFrame:
    for dossier in [ROOT / "data" / "2_construites"]:
        if (dossier / "dataset_daily_v2.csv").exists():
            return pd.read_csv(dossier / "dataset_daily_v2.csv", parse_dates=["date"]).set_index("date")
    raise SystemExit("dataset_daily_v2.csv introuvable (cherché dans data/2_construites/)")


def variables(d: pd.DataFrame) -> dict:
    """Trois familles de variables, selon le moment où elles sont connues."""
    tokyo = d[["n225_ret_1d", "n225_ret_5d", "n225_ret_20d", "n225_ret_60d", "n225_mom_12m", "n225_dist_sma20",
               "n225_dist_sma50", "n225_dist_sma200", "n225_vol_20d", "n225_vol_ewma", "n225_rsi14", "n225_macd_hist",
               "n225_bb_pctb", "n225_atr14_pct", "boj_days_to_next", "boj_days_since_last", "boj_meeting_day"]].copy()
    tokyo["taiex_r1"], tokyo["taiex_r5"] = d["taiex"].pct_change(), d["taiex"].pct_change(5)   # Taipei ferme avant Tokyo

    ailleurs = pd.DataFrame(index=d.index)
    for nom, jours in [("sp500", [1, 5, 20]), ("msci_em_eem", [5]), ("msci_japan_etf", [1, 5]), ("shanghai", [5]),
                       ("nifty50", [5]), ("eurjpy", [5, 20]), ("usdjpy", [5, 20])]:
        for j in jours:
            ailleurs[f"{nom}_r{j}"] = d[nom].pct_change(j)
    ailleurs["vix"], ailleurs["vix_d5"] = d["vix"], d["vix"].diff(5)
    for nom in ["us10y", "jgb10y", "spread_us_jgb10y"]:
        ailleurs[f"{nom}_d20"] = d[nom].diff(20)
    news = ["news_japan_sent_5d", "news_japan_sent_20d", "news_boj_sent_20d", "news_japan_neg_share_5d",
            "gdelt_japan_tone_5d", "gdelt_japan_attention"]
    ailleurs[news] = d[news]
    ailleurs["news_dispo"] = d["news_japan_sent_20d"].notna().astype(int)

    soir = pd.DataFrame({"sp500_soir": d["sp500"].pct_change(), "vix_soir": d["vix"].diff(),
                         "etf_japon_soir": d["msci_japan_etf"].pct_change(), "us10y_soir": d["us10y"].diff()})
    return {"tokyo": tokyo, "ailleurs_veille": ailleurs.shift(1), "wall_street_du_soir": soir}


def modeles() -> dict:
    prep = lambda m: Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", m)])
    return {"logistique L2": prep(LogisticRegression(C=0.01, max_iter=2000)),
            "LightGBM bridé": prep(LGBMClassifier(n_estimators=150, learning_rate=0.02, num_leaves=4, min_child_samples=100,
                                                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=5.0,
                                                   random_state=SEED, n_jobs=1, verbose=-1))}


def marcher(X: pd.DataFrame, y: pd.Series, horizon: int) -> pd.DataFrame:
    """Probabilités de baisse walk-forward. La cible d'une ligne d'entraînement est entièrement connue avant la prédiction."""
    ok = y.notna() & X.notna().any(axis=1)
    X, y = X[ok], y[ok].astype(int)
    depart = int(np.searchsorted(X.index, pd.Timestamp(DEBUT)))
    sorties = []
    for i in range(depart, len(X), REFAIRE):
        passe, bloc = slice(0, i - horizon), slice(i, min(i + REFAIRE, len(X)))
        p = {"constante": np.full(bloc.stop - bloc.start, y.iloc[passe].mean())}
        for nom, pipe in modeles().items():
            p[nom] = pipe.fit(X.iloc[passe], y.iloc[passe]).predict_proba(X.iloc[bloc])[:, 1]
        p["moitié logistique, moitié LightGBM"] = (p["logistique L2"] + p["LightGBM bridé"]) / 2
        p["70 % constante + 30 % logistique"] = 0.7 * p["constante"] + 0.3 * p["logistique L2"]
        sorties.append(pd.DataFrame(p, index=X.index[bloc]).assign(y=y.iloc[bloc]))
    return pd.concat(sorties)


def mesurer(pred: pd.DataFrame, horizon: int) -> dict:
    """Métriques sur des dates qui ne se chevauchent pas (une sur `horizon`), par période."""
    pred = pred.iloc[::horizon]
    perte = lambda p: -(pred["y"].reindex(p.index) * np.log(p) + (1 - pred["y"].reindex(p.index)) * np.log(1 - p))
    periodes = {"avant_test_2016_2023": pred.index < TEST[0], "test_2024_2026": (pred.index >= TEST[0]) & (pred.index <= TEST[1])}
    sortie = {}
    for cle, masque in periodes.items():
        y, lignes = pred["y"][masque], {}
        for nom in [c for c in pred.columns if c != "y"]:
            p = pred[nom][masque].clip(1e-4, 1 - 1e-4)
            ecart = (perte(p) - perte(pred["constante"][masque])).to_numpy()
            lignes[nom] = {"log_loss": float(perte(p).mean()), "ecart_vs_constante": float(ecart.mean()),
                           "erreur_type": float(ecart.std(ddof=1) / np.sqrt(len(ecart))),
                           "auc": float(roc_auc_score(y, p)) if nom != "constante" else 0.5,
                           "bon_sens_%": float(((p > 0.5) == (y == 1)).mean() * 100)}
        sortie[cle] = {"observations": int(masque.sum()), "part_de_baisses_%": float(y.mean() * 100), "modeles": lignes}
    return sortie


def volatilite(d: pd.DataFrame) -> dict:
    """Prévoir l'agitation des 20 prochaines séances, puis doser la position en conséquence."""
    r = d["n225_close"].pct_change()
    realisee = lambda n: r.rolling(n).std() * np.sqrt(252)
    X = pd.DataFrame({"vol5": np.log(realisee(5)), "vol20": np.log(realisee(20)), "vol60": np.log(realisee(60)),
                      "vix": np.log(d["vix"].shift(1))}).dropna()
    y = np.log(realisee(20).shift(-20)).reindex(X.index)                 # volatilité des 20 séances suivantes
    X, y = X[y.notna()], y[y.notna()]
    depart, lignes = int(np.searchsorted(X.index, pd.Timestamp(DEBUT))), []
    for i in range(depart, len(X), REFAIRE):
        passe = slice(0, i - 20)
        modele = LinearRegression().fit(X.iloc[passe], y.iloc[passe])
        lignes.append({"date": X.index[i], "reelle": y.iloc[i], "modele": modele.predict(X.iloc[[i]])[0],
                       "comme_le_mois_passe": X["vol20"].iloc[i], "moyenne_passee": y.iloc[passe].mean()})
    v = pd.DataFrame(lignes).set_index("date")

    def qualite(masque) -> dict:
        w, ref = v[masque], ((v["reelle"] - v["moyenne_passee"])[masque] ** 2).sum()
        return {"observations": int(masque.sum()),
                "r2_modele": float(1 - ((w["reelle"] - w["modele"]) ** 2).sum() / ref),
                "r2_comme_le_mois_passe": float(1 - ((w["reelle"] - w["comme_le_mois_passe"]) ** 2).sum() / ref),
                "correlation_prevue_reelle": float(w["reelle"].corr(w["modele"]))}

    # Dosage : part investie = volatilité visée / volatilité prévue, plafonnée à 100 %, revue toutes les 20 séances
    part = (VOL_CIBLE / np.exp(v["modele"])).clip(upper=1.0).reindex(r.index).ffill().shift(1)   # appliquée dès le lendemain
    test = (r.index >= TEST[0]) & (r.index <= TEST[1])

    def bilan(gains: pd.Series) -> dict:
        valeur = (1 + gains).cumprod()
        return {"cumul": float(valeur.iloc[-1] - 1), "volatilite_annuelle": float(gains.std() * np.sqrt(252)),
                "perte_maximale": float((valeur / valeur.cummax() - 1).min()),
                "rendement_par_unite_de_risque": float(gains.mean() / gains.std() * np.sqrt(252))}

    return {"prevision": {"avant_test_2016_2023": qualite(v.index < TEST[0]),
                          "test_2024_2026": qualite((v.index >= TEST[0]) & (v.index <= TEST[1]))},
            "dosage_sur_le_test": {"note": "rendements du Nikkei en yens, sans frais ni rémunération du cash : ordre de grandeur, "
                                           "pas comparable au centime avec les +82,3 % en euros couverts du projet",
                                   "part_investie_moyenne_%": float(part[test].mean() * 100),
                                   "acheter_et_garder": bilan(r[test]), "position_dosee": bilan((part * r)[test])}}


def main() -> None:
    warnings.filterwarnings("ignore")
    d = charger()
    v = variables(d)
    cloture, ouverture = d["n225_close"], d["n225_open"]
    baisse = lambda rendement: (rendement <= 0).astype(float).where(rendement.notna())
    connu_a_la_cloture = pd.concat([v["tokyo"], v["ailleurs_veille"]], axis=1)
    avec_le_soir = pd.concat([connu_a_la_cloture, v["wall_street_du_soir"]], axis=1)
    a_l_ouverture = avec_le_soir.assign(saut_d_ouverture=ouverture.shift(-1) / cloture - 1)
    experiences = {
        "lendemain, décision à la clôture de Tokyo": (connu_a_la_cloture, baisse(cloture.shift(-1) / cloture - 1), 1),
        "lendemain, en connaissant Wall Street du soir (piège : pas encore connu à la clôture)":
            (avec_le_soir, baisse(cloture.shift(-1) / cloture - 1), 1),
        "séance du lendemain, décision à l'ouverture (version honnête du piège)":
            (a_l_ouverture, baisse(cloture.shift(-1) / ouverture.shift(-1) - 1), 1),
        "5 séances": (connu_a_la_cloture, baisse(cloture.shift(-5) / cloture - 1), 5),
        "20 séances (un mois)": (connu_a_la_cloture, baisse(cloture.shift(-20) / cloture - 1), 20),
    }
    sens = {}
    for nom, (X, y, horizon) in experiences.items():
        sens[nom] = {"horizon_seances": horizon, "variables": int(X.shape[1]), **mesurer(marcher(X, y, horizon), horizon)}
        t = sens[nom]["test_2024_2026"]
        meilleur = min((m for m in t["modeles"] if m != "constante"), key=lambda m: t["modeles"][m]["log_loss"])
        print(f"{nom[:60]:60s} test n={t['observations']:4d} | {meilleur}: écart {t['modeles'][meilleur]['ecart_vs_constante']:+.4f} "
              f"± {t['modeles'][meilleur]['erreur_type']:.4f}, AUC {t['modeles'][meilleur]['auc']:.3f}")
    sortie = {"avertissement": "Exploration après coup : le test 2024-2026 avait déjà été ouvert. Ces chiffres ne remplacent pas "
                               "runs/evaluation_test.json, qui reste le résultat du projet.",
              "date": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(),
              "base": {"seances": int(len(d)), "debut": str(d.index[0].date()), "fin": str(d.index[-1].date())},
              "methode": {"premiere_prediction": DEBUT, "reentrainement_toutes_les_seances": REFAIRE,
                          "fenetre": "tout le passé, en retirant les lignes dont la cible n'est pas encore connue",
                          "mesure": "sur des dates qui ne se chevauchent pas ; écart = log-loss du modèle moins celle de la constante"},
              "sens_du_marche": sens, "volatilite": volatilite(d)}
    SORTIE.write_text(json.dumps(sortie, indent=2, ensure_ascii=False))
    print(f"écrit : {SORTIE.relative_to(ROOT)}")


if __name__ == "__main__":
    main()

In [ ]:
# ─── src/banc_essai_long.py · 115 lignes ───

"""Exploration après coup : apprendre sur plus de mois (depuis 2000, ou depuis 1970) aide-t-il ?

Seul le cours de clôture du Nikkei existe sur une aussi longue période dans nos fichiers
(data/1_brutes/nikkei225_long.csv, depuis 1970). Les variables sont donc tirées des prix seulement.
La comparaison porte sur la date de départ de l'apprentissage, toutes choses égales par ailleurs.
Ce script ne remplace pas le résultat du projet (runs/evaluation_test.json) ; le test a déjà été ouvert.

Usage : python -m src.banc_essai_long     écrit runs/exploration_historique_long.json
"""
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .config import ROOT, RUNS, SEED, commit_courant

SORTIE = RUNS / "exploration_historique_long.json"
DEPARTS = ["2010-02", "2000-01", "1985-01", "1971-06"]        # premier mois d'apprentissage
PERIODES = {"avant_test_2016_2023": ("2016-02", "2023-12"), "test_2024_2026": ("2024-01", "2026-09")}


def table_mensuelle() -> pd.DataFrame:
    """Variables de prix en fin de mois et sens du mois suivant, à partir des clôtures journalières."""
    for dossier in [ROOT / "data" / "1_brutes"]:
        if (dossier / "nikkei225_long.csv").exists():
            jour = pd.read_csv(dossier / "nikkei225_long.csv", parse_dates=[0], index_col=0)["close"].dropna()
            break
    else:
        raise SystemExit("nikkei225_long.csv introuvable")
    r_jour = jour.pct_change()
    mois = jour.groupby(jour.index.to_period("M")).last()
    r = mois.pct_change()
    t = pd.DataFrame({"ret_1m": r, "mom_3m": mois.pct_change(3), "mom_6m": mois.pct_change(6), "mom_12m": mois.pct_change(12),
                      "ecart_moyenne_10m": mois / mois.rolling(10).mean() - 1, "recul_depuis_le_plus_haut_12m": mois / mois.rolling(12).max() - 1,
                      "vol_1m": r_jour.groupby(r_jour.index.to_period("M")).std() * np.sqrt(252),
                      "vol_6m": r.rolling(6).std() * np.sqrt(12)})
    t["rendement_suivant"] = r.shift(-1)
    t["y_baisse"] = (t["rendement_suivant"] <= 0).astype(float).where(t["rendement_suivant"].notna())
    return t.dropna()


def modeles() -> dict:
    prep = lambda m: Pipeline([("scale", StandardScaler()), ("model", m)])
    return {"logistique L2": prep(LogisticRegression(C=0.03, max_iter=2000)),
            "logistique très pénalisée": prep(LogisticRegression(C=0.003, max_iter=2000)),
            "LightGBM bridé": prep(LGBMClassifier(n_estimators=100, learning_rate=0.02, num_leaves=4, min_child_samples=30,
                                                   subsample=0.8, subsample_freq=1, reg_lambda=5.0, random_state=SEED,
                                                   n_jobs=1, verbose=-1))}


def marcher(t: pd.DataFrame, depart: str) -> pd.DataFrame:
    """Réentraînement chaque mois sur tous les mois depuis `depart`, prédiction du mois suivant."""
    X, y = t.drop(columns=["rendement_suivant", "y_baisse"]), t["y_baisse"].astype(int)
    lignes = []
    for mois in t.loc[PERIODES["avant_test_2016_2023"][0]:PERIODES["test_2024_2026"][1]].index:
        passe = (t.index >= pd.Period(depart, "M")) & (t.index < mois)
        p = {"constante": y[passe].mean()}
        for nom, pipe in modeles().items():
            p[nom] = pipe.fit(X[passe], y[passe]).predict_proba(X.loc[[mois]])[0, 1]
        lignes.append({"mois": mois, "y": y[mois], "mois_d_apprentissage": int(passe.sum()), **p})
    return pd.DataFrame(lignes).set_index("mois")


def mesurer(pred: pd.DataFrame, reference: pd.DataFrame) -> dict:
    """Écart de log-loss avec la constante apprise depuis 2010 (la même référence pour tous les départs)."""
    perte = lambda y, p: -(y * np.log(p) + (1 - y) * np.log(1 - p))
    sortie = {}
    for cle, (a, b) in PERIODES.items():
        d, ref = pred.loc[a:b], reference.loc[a:b]
        lignes = {}
        for nom in ["constante"] + list(modeles()):
            ecart = (perte(d["y"], d[nom]) - perte(ref["y"], ref["constante"])).to_numpy()
            lignes[nom] = {"log_loss": float(perte(d["y"], d[nom]).mean()), "ecart_vs_constante_2010": float(ecart.mean()),
                           "erreur_type": float(ecart.std(ddof=1) / np.sqrt(len(ecart))),
                           "auc": float(roc_auc_score(d["y"], d[nom])) if nom != "constante" else 0.5,
                           "proba_moyenne_de_baisse_%": float(d[nom].mean() * 100)}
        sortie[cle] = {"mois": int(len(d)), "baisses": int(d["y"].sum()),
                       "mois_d_apprentissage": [int(d["mois_d_apprentissage"].iloc[0]), int(d["mois_d_apprentissage"].iloc[-1])],
                       "modeles": lignes}
    return sortie


def main() -> None:
    warnings.filterwarnings("ignore")
    t = table_mensuelle()
    pred = {depart: marcher(t, depart) for depart in DEPARTS}
    resultats = {depart: mesurer(pred[depart], pred["2010-02"]) for depart in DEPARTS}
    frequences = {depart: float(t.loc[depart:"2023-12", "y_baisse"].mean() * 100) for depart in DEPARTS}
    for depart in DEPARTS:
        test = resultats[depart]["test_2024_2026"]
        meilleur = min(modeles(), key=lambda m: test["modeles"][m]["log_loss"])
        print(f"depuis {depart} ({test['mois_d_apprentissage'][0]} mois, {frequences[depart]:.1f} % de baisses) : constante "
              f"{test['modeles']['constante']['log_loss']:.3f} | {meilleur} {test['modeles'][meilleur]['log_loss']:.3f} "
              f"(écart {test['modeles'][meilleur]['ecart_vs_constante_2010']:+.3f} ± {test['modeles'][meilleur]['erreur_type']:.3f})")
    sortie = {"avertissement": "Exploration après coup : le test 2024-2026 avait déjà été ouvert. Ces chiffres ne remplacent pas "
                               "runs/evaluation_test.json, qui reste le résultat du projet.",
              "date": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(),
              "methode": {"variables": [c for c in t.columns if c not in ("rendement_suivant", "y_baisse")],
                          "reentrainement": "chaque mois, sur tous les mois depuis la date de départ",
                          "reference": "constante apprise depuis 2010-02, identique pour tous les départs"},
              "part_de_baisses_jusqu_a_2023_%": frequences, "resultats_par_depart": resultats}
    SORTIE.write_text(json.dumps(sortie, indent=2, ensure_ascii=False))
    print(f"test : {resultats['2010-02']['test_2024_2026']['mois']} mois, {resultats['2010-02']['test_2024_2026']['baisses']} baisses ; "
          f"écrit : {SORTIE.relative_to(ROOT)}")


if __name__ == "__main__":
    main()

In [ ]:
# ─── src/banc_essai_normalisation.py · 69 lignes ───

"""Exploration après coup : normaliser les variables sur une fenêtre glissante aide-t-il après le changement de régime ?

Dans le projet, chaque variable est centrée et réduite avec la moyenne et l'écart-type de tout l'entraînement.
Après 2022, plusieurs variables sortent de cette plage (le portage est à six écarts-types). Ici, chaque valeur est
comparée aux 24, 36 ou 60 derniers mois seulement : z = (x - moyenne récente) / écart-type récent, sans regarder le futur.
Les modèles sont ensuite réentraînés chaque mois, comme dans src.banc_essai. Le test avait déjà été ouvert.

Usage : python -m src.banc_essai_normalisation     écrit runs/exploration_normalisation.json
"""
import json
import warnings
from datetime import datetime

import pandas as pd
from joblib import Parallel, delayed

from .banc_essai import DEPART, marcher, perte, resumer
from .config import COLONNES, ROOT, RUNS, commit_courant
from .data import charger_table

SORTIE = RUNS / "exploration_normalisation.json"
FENETRES = [24, 36, 60]
MODELES = ["logistique L2 (modèle du projet)", "logistique très pénalisée", "logistique L1", "Random Forest (projet)", "LightGBM (projet)"]


def normaliser(table: pd.DataFrame, mois: int) -> pd.DataFrame:
    """Chaque variable continue est exprimée en écarts-types par rapport à ses `mois` derniers mois (mois courant inclus)."""
    t = table.copy()
    for col in [c for c in COLONNES if t[c].nunique() > 2]:           # l'indicateur de présence des news reste tel quel
        glissant = t[col].rolling(mois, min_periods=12)
        t[col] = ((t[col] - glissant.mean()) / glissant.std()).clip(-5, 5)
    return t


def main() -> None:
    warnings.filterwarnings("ignore")
    table = charger_table()
    table = table[table["bloc"] != "à prédire"]
    variantes = {"tout l'entraînement (projet)": table} | {f"{m} derniers mois": normaliser(table, m) for m in FENETRES}
    plan = [(v, nom) for v in variantes for nom in MODELES]
    morceaux = Parallel(n_jobs=-1)(delayed(marcher)(nom, "croissante", variantes[v]) for v, nom in plan)
    const = marcher("constante (fréquence passée)", "croissante", table).set_index("date")
    periodes = {"avant_test_2016_2023": table.index[DEPART:][(table["bloc"].iloc[DEPART:] != "test").to_numpy()],
                "test_2024_2026": table.index[table["bloc"] == "test"]}
    resultats = []
    for (v, nom), pred in zip(plan, morceaux):
        pred, ligne = pred.set_index("date"), {"normalisation": v, "modele": nom}
        for cle, mois in periodes.items():
            bloc = table.loc[mois]
            ref = pd.Series(perte(bloc["y_baisse"], const.loc[mois, "p_baisse"]), index=mois)
            ligne[cle] = resumer(pred.loc[mois], bloc, ref)
        resultats.append(ligne)
    sortie = {"avertissement": "Exploration après coup : le test 2024-2026 avait déjà été ouvert. Ces chiffres ne remplacent pas "
                               "runs/evaluation_test.json, qui reste le résultat du projet.",
              "date": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(),
              "methode": {"normalisation": "z = (x - moyenne des N derniers mois) / écart-type des N derniers mois, borné à ±5",
                          "fenetres_mois": FENETRES, "reentrainement": "chaque mois, sur tout le passé",
                          "reference": "constante (fréquence passée)"},
              "resultats": resultats}
    SORTIE.write_text(json.dumps(sortie, indent=2, ensure_ascii=False))
    for r in sorted(resultats, key=lambda r: r["test_2024_2026"]["log_loss"]):
        a, t = r["avant_test_2016_2023"], r["test_2024_2026"]
        print(f"{r['normalisation']:30s} {r['modele'][:34]:34s} avant {a['ecart_vs_constante']:+.3f} ± {a['erreur_type_de_l_ecart']:.3f} | "
              f"test LL {t['log_loss']:.3f} écart {t['ecart_vs_constante']:+.3f} ± {t['erreur_type_de_l_ecart']:.3f} PR-AUC {t['pr_auc']:.2f} règle {t['cumul_regle']:+.1%}")
    print(f"écrit : {SORTIE.relative_to(ROOT)}")


if __name__ == "__main__":
    main()

In [ ]:
# ─── src/banc_essai_ouverture.py · 115 lignes ───

"""Exploration après coup : une décision prise chaque matin à l'ouverture de Tokyo, sur un produit réellement achetable.

Sur l'indice, un signal apparaissait à l'ouverture (src.banc_essai_jour), mais l'indice ouvre avec des cours en partie
périmés. Ici, on le teste sur l'ETF Nikkei 225 coté à Tokyo (code 1321), dont l'ouverture est un vrai prix d'échange
(data/1_brutes/nikkei_etf_1321.csv, Yahoo Finance).

Décision : à l'ouverture du jour J+1, en connaissant tout ce qui était connu à la clôture de Tokyo du jour J,
la séance de Wall Street de la nuit, et le saut d'ouverture de l'ETF. Cible : l'ETF baisse-t-il entre l'ouverture et la
clôture de J+1 ? Le test 2024-2026 avait déjà été ouvert : ces chiffres ne remplacent pas le résultat du projet.

Usage : python -m src.banc_essai_ouverture     écrit runs/exploration_ouverture.json
"""
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

from .banc_essai_jour import TEST, charger, marcher, mesurer, variables
from .config import ROOT, RUNS, commit_courant

SORTIE = RUNS / "exploration_ouverture.json"
FRAIS = [0.0, 0.0002, 0.0005, 0.0010]          # coût d'un aller-retour (achat à l'ouverture, revente à la clôture)
MARGE = 0.03                                   # on ne prend position que si la probabilité s'écarte de 50 % d'au moins 3 points


def charger_etf() -> pd.DataFrame:
    for dossier in [ROOT / "data" / "1_brutes"]:
        if (dossier / "nikkei_etf_1321.csv").exists():
            return pd.read_csv(dossier / "nikkei_etf_1321.csv", parse_dates=[0], index_col=0)
    raise SystemExit("nikkei_etf_1321.csv introuvable (cherché dans data/1_brutes/)")


def bilan(gains: pd.Series, positions: pd.Series) -> dict:
    """Résultat d'une suite de paris d'une séance : `gains` nets, un par séance (0 quand on reste à l'écart)."""
    pris = positions != 0
    valeur = (1 + gains).cumprod()
    return {"seances_jouees": int(pris.sum()), "part_des_seances_%": float(pris.mean() * 100),
            "paris_gagnants_%": float((gains[pris] > 0).mean() * 100) if pris.any() else None,
            "gain_moyen_par_pari_pb": float(gains[pris].mean() * 1e4) if pris.any() else None,
            "cumul": float(valeur.iloc[-1] - 1), "perte_maximale": float((valeur / valeur.cummax() - 1).min()),
            "rendement_par_unite_de_risque": float(gains.mean() / gains.std() * np.sqrt(252)) if gains.std() > 0 else 0.0}


def main() -> None:
    warnings.filterwarnings("ignore")
    d, etf = charger(), charger_etf()
    etf = etf.reindex(d.index)
    ouverture, cloture = etf["Open"], etf["Close"]
    saut = (ouverture.shift(-1) / cloture - 1).where(lambda s: s.abs() < 0.15)       # connu à l'ouverture de J+1
    seance = cloture.shift(-1) / ouverture.shift(-1) - 1                             # ce qu'on cherche à prévoir
    seance = seance.where(etf["Volume"].shift(-1) > 0)
    v = variables(d)
    X = pd.concat([v["tokyo"], v["ailleurs_veille"], v["wall_street_du_soir"]], axis=1).assign(saut_d_ouverture=saut)
    y = (seance <= 0).astype(float).where(seance.notna() & saut.notna())

    pred = marcher(X, y, 1)
    mesures = mesurer(pred, 1)

    # Stratégies d'une séance, sur le test : +1 = acheter à l'ouverture, -1 = vendre à découvert, 0 = rester à l'écart
    test = pred.loc[TEST[0]:TEST[1]]
    r = seance.reindex(test.index)
    p_hausse = 1 - test["moitié logistique, moitié LightGBM"]
    nuit = d["sp500"].pct_change().reindex(test.index)
    regles = {"modèle (moitié logistique, moitié LightGBM)": np.sign(p_hausse - 0.5).where((p_hausse - 0.5).abs() >= MARGE, 0.0),
              "modèle, achats seulement": (p_hausse >= 0.5 + MARGE).astype(float),
              "règle simple : suivre Wall Street de la nuit": np.sign(nuit).fillna(0.0),
              "règle simple : suivre le saut d'ouverture": np.sign(saut.reindex(test.index)).fillna(0.0),
              "toujours acheter à l'ouverture, revendre à la clôture": pd.Series(1.0, index=test.index)}
    strategies = {nom: {f"frais {f * 1e4:.0f} pb": bilan(pos * r - pos.abs() * f, pos) for f in FRAIS} for nom, pos in regles.items()}
    etf_test = cloture.loc[TEST[0]:TEST[1]].dropna()
    garder = bilan(etf_test.pct_change().dropna(), pd.Series(1.0, index=etf_test.index[1:]))

    # Deux contrôles : l'ouverture de l'ETF est-elle un vrai prix ? et le signal dépend-il de quelques séances ?
    indice_saut, indice_seance = d["n225_open"].shift(-1) / d["n225_close"] - 1, d["n225_close"].shift(-1) / d["n225_open"].shift(-1) - 1
    controles = {"correlation_saut_puis_seance_sur_l_indice": float(indice_saut.corr(indice_seance)),
                 "correlation_saut_puis_seance_sur_l_etf": float(saut.corr(seance)),
                 "correlation_wall_street_puis_seance_de_l_etf": float(d["sp500"].pct_change().corr(seance)),
                 "correlation_wall_street_puis_saut_de_l_etf": float(d["sp500"].pct_change().corr(saut))}
    par_annee = {}
    for an, bloc in pred.groupby(pred.index.year):
        p, yy = bloc["moitié logistique, moitié LightGBM"].clip(1e-4, 1 - 1e-4), bloc["y"]
        perte = lambda q: -(yy * np.log(q) + (1 - yy) * np.log(1 - q))
        ecart = perte(p) - perte(bloc["constante"])
        par_annee[int(an)] = {"seances": int(len(bloc)), "ecart_vs_constante": float(ecart.mean()), "erreur_type": float(ecart.std(ddof=1) / np.sqrt(len(ecart)))}

    sortie = {"avertissement": "Exploration après coup : le test 2024-2026 avait déjà été ouvert. Ces chiffres ne remplacent pas "
                               "runs/evaluation_test.json, qui reste le résultat du projet.",
              "date": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(),
              "methode": {"produit": "ETF Nikkei 225 coté à Tokyo (1321.T), prix non ajustés, Yahoo Finance",
                          "decision": "à l'ouverture de J+1 ; position tenue jusqu'à la clôture de J+1",
                          "variables": int(X.shape[1]), "marge_de_decision": MARGE, "frais_testes_aller_retour": FRAIS,
                          "reentrainement": "toutes les 20 séances, sur tout le passé"},
              "prevision": mesures, "ecart_par_annee": par_annee, "controles": controles,
              "strategies_sur_le_test": strategies, "acheter_et_garder_l_etf_sur_le_test": garder}
    SORTIE.write_text(json.dumps(sortie, indent=2, ensure_ascii=False))

    for cle, b in mesures.items():
        print(f"\n{cle} : {b['observations']} séances, {b['part_de_baisses_%']:.1f} % de baisses")
        for m, x in b["modeles"].items():
            print(f"   {m:38s} écart {x['ecart_vs_constante']:+.4f} ± {x['erreur_type']:.4f}  AUC {x['auc']:.3f}  bon sens {x['bon_sens_%']:.1f} %")
    print("\ncontrôles :", {k: round(x, 3) for k, x in controles.items()})
    print("par année :", {a: f"{x['ecart_vs_constante']:+.4f}±{x['erreur_type']:.4f}" for a, x in par_annee.items()})
    for nom, f in strategies.items():
        print(f"\n{nom}")
        for cle, b in f.items():
            print(f"   {cle:12s} {b['seances_jouees']:4d} paris, {b['paris_gagnants_%']:.1f} % gagnants, {b['gain_moyen_par_pari_pb']:+.1f} pb/pari, "
                  f"cumul {b['cumul']:+.1%}, perte max {b['perte_maximale']:.1%}, ratio {b['rendement_par_unite_de_risque']:.2f}")
    print(f"\nacheter et garder l'ETF : cumul {garder['cumul']:+.1%}, perte max {garder['perte_maximale']:.1%}, ratio {garder['rendement_par_unite_de_risque']:.2f}")
    print(f"écrit : {SORTIE.relative_to(ROOT)}")


if __name__ == "__main__":
    main()

### 5.2 · Prévoir l'agitation plutôt que le sens

Le sens du marché ne se prévoit pas, mais sa volatilité oui : elle vient par périodes.

- `volatilite.py` prévoit la volatilité du mois suivant (GARCH, régression sur la volatilité passée) et la probabilité d'un mois agité ;
- `signal_risque.py` en déduit chaque jour la part à investir (100 %, 50 %, 0 %) ;
- `signal_risque_variantes.py` teste trois améliorations écrites avant de voir leurs résultats.

> **Concepts** · volatilité conditionnelle, GARCH, $R^2$ hors échantillon, ciblage de volatilité, ratio de Sharpe.

In [ ]:
# ─── src/volatilite.py · 230 lignes ───

"""Second volet, ajouté après coup : prévoir l'agitation du Nikkei plutôt que son sens.

Le sens du mois suivant ne se prévoit pas (runs/evaluation_test.json, résultat figé du projet).
Ce script pose une autre question sur les mêmes mois et le même découpage : quelle sera la volatilité
du mois prochain, et ce mois sera-t-il agité ? Il a été écrit après l'ouverture du test.

Trois parties :
  1. prévision de la volatilité du mois suivant, sept méthodes réentraînées chaque mois ;
  2. classification « mois agité » (volatilité réalisée au-dessus de 20 %), avec la démarche du projet ;
  3. usage : le risque annoncé est-il fiable, et que donne une position dosée selon ce risque ?

Usage : python -m src.volatilite     écrit runs/exploration_volatilite.json et ..._predictions.csv
"""
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor
from scipy.optimize import minimize
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .banc_essai import DEPART, ece, perte
from .banc_essai_jour import charger as charger_jours
from .config import ROOT, RUNS, SEED, commit_courant
from .data import charger_table

SORTIE = RUNS / "exploration_volatilite.json"
PREDICTIONS = RUNS / "exploration_volatilite_predictions.csv"
SEUIL_AGITE = 0.20        # un mois est « agité » si sa volatilité réalisée dépasse 20 % par an (la moyenne du Nikkei)
VOL_CIBLE = 0.15          # volatilité visée par la position dosée
RETENU = "régression : 1, 3, 6 mois + VIX + rendement"       # méthode utilisée dans la partie « usage »
HAR = ["lv_1m", "lv_3m", "lv_6m"]
TOUTES = HAR + ["lv_ewma", "lv_vix", "n225_ret_1m", "boj_meeting_next_month"]


def construire() -> tuple[pd.DataFrame, pd.Series]:
    """Table mensuelle du projet, enrichie des volatilités réalisées calculées sur les séances."""
    jours = charger_jours()
    r = jours["n225_close"].pct_change().dropna()
    mois = r.index.to_period("M")
    annuel = np.sqrt(252)
    fin = pd.DataFrame({"lv_1m": np.log(r.groupby(mois).std() * annuel),
                        "lv_3m": np.log((r.rolling(63).std() * annuel).groupby(mois).last()),
                        "lv_6m": np.log((r.rolling(126).std() * annuel).groupby(mois).last())})
    complet = r.groupby(mois).count() >= 15                       # un mois entamé n'a pas de volatilité réalisée
    fin["lv_suivant"] = fin["lv_1m"].where(complet).shift(-1)
    t = charger_table()
    t = pd.concat([t, fin.reindex(t.index.to_period("M")).set_index(t.index)], axis=1)
    t["lv_ewma"], t["lv_vix"] = np.log(t["n225_vol_ewma"]), np.log(t["vix"] / 100)
    t["agite"] = (np.exp(t["lv_suivant"]) > SEUIL_AGITE).astype(float).where(t["lv_suivant"].notna())
    return t, r


class Garch:
    """GARCH(1,1) gaussien, variance de long terme fixée à la variance observée, ajusté par maximum de vraisemblance."""

    def __init__(self):
        self.alpha, self.beta = 0.08, 0.90

    @staticmethod
    def _variances(r, alpha, beta):
        longue = r.var()
        s2 = np.empty(len(r) + 1)
        s2[0] = longue
        for i, x in enumerate(r):
            s2[i + 1] = longue * (1 - alpha - beta) + alpha * x * x + beta * s2[i]
        return s2, longue

    def ajuster(self, r: np.ndarray) -> "Garch":
        def moins_vraisemblance(theta):
            s2, _ = self._variances(r, *theta)
            return 0.5 * np.sum(np.log(s2[:-1]) + r * r / s2[:-1])
        res = minimize(moins_vraisemblance, [self.alpha, self.beta], method="L-BFGS-B",
                       bounds=[(0.01, 0.3), (0.5, 0.98)], options={"maxiter": 60})
        if res.x.sum() < 0.999:
            self.alpha, self.beta = res.x
        return self

    def prevoir(self, r: np.ndarray, seances: int = 21) -> float:
        """Volatilité annualisée moyenne attendue sur les `seances` prochaines séances."""
        s2, longue = self._variances(r, self.alpha, self.beta)
        persistance = self.alpha + self.beta
        chemin = longue + persistance ** np.arange(seances) * (s2[-1] - longue)
        return float(np.sqrt(chemin.mean() * 252))


def prevoir_volatilite(t: pd.DataFrame, r: pd.Series) -> pd.DataFrame:
    """Prévisions walk-forward du log de la volatilité du mois suivant."""
    connu = t["lv_suivant"].notna()
    garch, lignes = Garch(), []
    for i in range(DEPART, len(t)):
        passe, ligne = t.iloc[:i][connu.iloc[:i]].dropna(subset=TOUTES), t.iloc[[i]]
        jusqu_ici = r.loc[:t.index[i]].to_numpy()
        if (i - DEPART) % 6 == 0:                                   # GARCH réajusté tous les six mois
            garch.ajuster(jusqu_ici)
        y = passe["lv_suivant"]
        lgbm = LGBMRegressor(n_estimators=150, learning_rate=0.03, num_leaves=4, min_child_samples=15, subsample=0.8,
                             subsample_freq=1, reg_lambda=5.0, random_state=SEED, n_jobs=1, verbose=-1)
        lignes.append({
            "date": t.index[i], "reelle": t["lv_suivant"].iloc[i],
            "moyenne passée": y.mean(),
            "comme le mois passé": ligne["lv_1m"].iloc[0],
            "lissage exponentiel": ligne["lv_ewma"].iloc[0],
            "régression : 1, 3 et 6 mois": LinearRegression().fit(passe[HAR], y).predict(ligne[HAR])[0],
            RETENU: LinearRegression().fit(passe[TOUTES], y).predict(ligne[TOUTES])[0],
            "GARCH(1,1)": np.log(garch.prevoir(jusqu_ici)),
            "LightGBM bridé": lgbm.fit(passe[TOUTES], y).predict(ligne[TOUTES])[0],
            "ecart_type_des_residus": float(np.std(y - LinearRegression().fit(passe[TOUTES], y).predict(passe[TOUTES]), ddof=len(TOUTES) + 1)),
        })
    return pd.DataFrame(lignes).set_index("date")


def prevoir_agitation(t: pd.DataFrame) -> pd.DataFrame:
    """Probabilité walk-forward que le mois suivant soit agité."""
    connu, lignes = t["agite"].notna(), []
    for i in range(DEPART, len(t)):
        passe, ligne = t.iloc[:i][connu.iloc[:i]].dropna(subset=TOUTES), t.iloc[[i]]
        y = passe["agite"].astype(int)
        logit = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(C=0.3, max_iter=2000))])
        lgbm = LGBMClassifier(n_estimators=150, learning_rate=0.03, num_leaves=4, min_child_samples=15, subsample=0.8,
                              subsample_freq=1, reg_lambda=5.0, random_state=SEED, n_jobs=1, verbose=-1)
        lignes.append({"date": t.index[i], "y": t["agite"].iloc[i], "constante": y.mean(),
                       "logistique": logit.fit(passe[TOUTES], y).predict_proba(ligne[TOUTES])[0, 1],
                       "LightGBM bridé": lgbm.fit(passe[TOUTES], y).predict_proba(ligne[TOUTES])[0, 1]})
    a = pd.DataFrame(lignes).set_index("date")
    probas = ["constante", "logistique", "LightGBM bridé"]
    a[probas] = a[probas].clip(1e-4, 1 - 1e-4)
    return a


def periodes(t: pd.DataFrame, index: pd.Index) -> dict:
    bloc = t["bloc"].reindex(index)
    return {"avant_test_2016_2023": (bloc.isin(["entraînement", "validation"])).to_numpy(), "test_2024_2026": (bloc == "test").to_numpy()}


def bilan_prevision(v: pd.DataFrame, masque) -> dict:
    w, sortie = v[masque], {}
    reference = ((w["reelle"] - w["moyenne passée"]) ** 2).sum()
    for nom in [c for c in v.columns if c not in ("reelle", "ecart_type_des_residus")]:
        rapport = np.exp(2 * (w["reelle"] - w[nom]))               # variance réalisée / variance prévue
        sortie[nom] = {"r2_hors_echantillon": float(1 - ((w["reelle"] - w[nom]) ** 2).sum() / reference),
                       "erreur_moyenne_points_de_vol": float((np.exp(w["reelle"]) - np.exp(w[nom])).abs().mean() * 100),
                       "qlike": float((rapport - np.log(rapport) - 1).mean()),
                       "correlation": float(w["reelle"].corr(w[nom])) if nom != "moyenne passée" else 0.0}
    return {"mois": int(masque.sum()), "methodes": sortie}


def bilan_agitation(a: pd.DataFrame, masque) -> dict:
    w, sortie = a[masque], {}
    y = w["y"].astype(int).to_numpy()
    for nom in ["constante", "logistique", "LightGBM bridé"]:
        p = w[nom].to_numpy()
        ecart = perte(y, p) - perte(y, w["constante"].to_numpy())
        sortie[nom] = {"log_loss": float(perte(y, p).mean()), "ecart_vs_constante": float(ecart.mean()),
                       "erreur_type": float(ecart.std(ddof=1) / np.sqrt(len(ecart))),
                       "auc": float(roc_auc_score(y, p)) if nom != "constante" else 0.5,
                       "pr_auc": float(average_precision_score(y, p)), "brier": float(brier_score_loss(y, p)), "ece": ece(y, p)}
    return {"mois": int(len(y)), "mois_agites": int(y.sum()), "modeles": sortie}


def bilan_usage(t: pd.DataFrame, v: pd.DataFrame, masque) -> dict:
    """Fiabilité du risque annoncé, et position dosée, en euros couverts comme dans le projet."""
    w = v[masque]
    rendement, cash = t.loc[w.index, "target_reur_hedged_next"], t.loc[w.index, "eur_overnight"] / 100 / 12
    sortie = {"mois": int(len(w))}
    for nom, colonne in [("volatilité prévue", RETENU), ("volatilité constante", "moyenne passée")]:
        sigma = np.exp(w[colonne]) / np.sqrt(12)                    # écart-type mensuel annoncé
        sortie[nom] = {"pertes_au_dela_du_seuil_5_%": int((rendement < -1.645 * sigma).sum()), "attendues_5_%": round(0.05 * len(w), 1),
                       "pertes_au_dela_du_seuil_10_%": int((rendement < -1.282 * sigma).sum()), "attendues_10_%": round(0.10 * len(w), 1),
                       "mois_dans_la_bande_a_1_ecart_type_%": float((rendement.abs() <= sigma).mean() * 100), "attendu_bande_%": 68.3}
    part = (VOL_CIBLE / np.exp(w[RETENU])).clip(upper=1.0)

    def bilan(gains: pd.Series) -> dict:
        valeur = (1 + gains).cumprod()
        exces = gains - cash
        return {"cumul": float(valeur.iloc[-1] - 1), "volatilite_annuelle": float(gains.std() * np.sqrt(12)),
                "perte_maximale": float((valeur / valeur.cummax() - 1).min()), "pire_mois": float(gains.min()),
                "rendement_par_unite_de_risque": float(exces.mean() / gains.std() * np.sqrt(12))}

    sortie["position"] = {"part_investie_moyenne_%": float(part.mean() * 100), "part_investie_minimale_%": float(part.min() * 100),
                          "acheter_et_garder": bilan(rendement), "position_dosee": bilan(part * rendement + (1 - part) * cash)}
    return sortie


def main() -> None:
    warnings.filterwarnings("ignore")
    t, r = construire()
    v, a = prevoir_volatilite(t, r), prevoir_agitation(t)
    a_predire = v.index[v["reelle"].isna()]
    evalue_v, evalue_a = v.drop(index=a_predire), a.drop(index=a_predire)
    per = periodes(t, evalue_v.index)

    derniere = v.loc[a_predire[-1]] if len(a_predire) else v.iloc[-1]
    vol_prevue = float(np.exp(derniere[RETENU]))
    prochain = {"donnees_arretees_au": str((a_predire[-1] if len(a_predire) else v.index[-1]).date()),
                "volatilite_prevue_%": vol_prevue * 100, "volatilite_moyenne_passee_%": float(np.exp(derniere["moyenne passée"]) * 100),
                "probabilite_mois_agite_%": float(a.loc[a_predire[-1], "logistique"] * 100) if len(a_predire) else None,
                "perte_mensuelle_depassee_1_fois_sur_20_%": float(-1.645 * vol_prevue / np.sqrt(12) * 100),
                "part_investie_conseillee_%": float(min(1.0, VOL_CIBLE / vol_prevue) * 100)}

    sortie = {"avertissement": "Second volet ajouté après l'ouverture du test 2024-2026. Il ne remplace pas runs/evaluation_test.json : "
                               "la réponse à la question du projet (acheter, neutre ou vendre) reste « acheter et garder ».",
              "date": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(),
              "methode": {"cible": "volatilité réalisée du mois suivant (écart-type des rendements journaliers, annualisé)",
                          "mois_agite": f"volatilité réalisée au-dessus de {SEUIL_AGITE:.0%}", "variables": TOUTES,
                          "reentrainement": "chaque mois, sur tout le passé connu", "premiere_prediction": str(v.index[0].date()),
                          "methode_retenue_pour_l_usage": RETENU, "volatilite_cible_de_la_position_dosee": VOL_CIBLE},
              "prevision_de_la_volatilite": {cle: bilan_prevision(evalue_v, m) for cle, m in per.items()},
              "mois_agite_ou_non": {cle: bilan_agitation(evalue_a, m) for cle, m in per.items()},
              "usage": {cle: bilan_usage(t, evalue_v, m) for cle, m in per.items()},
              "mois_prochain": prochain}
    SORTIE.write_text(json.dumps(sortie, indent=2, ensure_ascii=False))

    export = pd.DataFrame({"bloc": t["bloc"].reindex(v.index), "vol_reelle": np.exp(v["reelle"]), "vol_prevue": np.exp(v[RETENU]),
                           "vol_moyenne_passee": np.exp(v["moyenne passée"]), "vol_mois_passe": np.exp(v["comme le mois passé"]),
                           "p_agite": a["logistique"], "agite": a["y"], "rendement_suivant": t["target_reur_hedged_next"].reindex(v.index)})
    export.round(5).rename_axis("date").to_csv(PREDICTIONS)
    test = sortie["prevision_de_la_volatilite"]["test_2024_2026"]["methodes"]
    for nom, m in test.items():
        print(f"{nom:45s} R² {m['r2_hors_echantillon']:+.3f}  erreur {m['erreur_moyenne_points_de_vol']:.1f} pts  QLIKE {m['qlike']:.3f}")
    print(f"écrit : {SORTIE.relative_to(ROOT)} et {PREDICTIONS.relative_to(ROOT)}")


if __name__ == "__main__":
    main()

In [ ]:
# ─── src/signal_risque.py · 295 lignes ───

"""Signal d'achat fondé sur le risque : chaque jour, on prévoit la volatilité du Nikkei sur le mois qui vient
et on en déduit combien investir (100 %, 50 % ou 0 %). On ne cherche plus à deviner la hausse ou la baisse.

En clair
- Le risque d'un marché (sa volatilité) varie par périodes : calme, puis agité, puis calme. Il se prévoit, le sens
  du marché non (voir src/banc_essai*.py).
- Chaque soir, après la clôture de Tokyo, on prévoit la volatilité des 20 prochaines séances.
- Risque prévu bas : acheter plein. Moyen : acheter à moitié. Très élevé : ne pas acheter (l'argent reste en euros).
- Pour ne pas acheter et vendre sans arrêt, un changement de niveau doit être confirmé trois soirs de suite.
- On exécute le lendemain à la clôture : aucun prix inconnu au moment de décider n'est utilisé.

Protocole, en deux temps (pour ne pas régler la règle sur les années qui servent à la juger)
  python -m src.signal_risque --figer     choisit le modèle et les seuils avec les seules données 2000-2015,
                                          écrit runs/signal_risque_regle.json (refuse d'écraser une règle existante)
  python -m src.signal_risque             applique la règle figée à 2003-2026 et écrit runs/signal_risque.json

Honnêteté : les années 2016-2026 n'ont pas servi aux choix, mais l'auteur les connaît, et une version mensuelle de
cette idée avait déjà été regardée sur 2024-2026 (src/volatilite.py). Ce volet s'ajoute au résultat officiel du
projet (runs/evaluation_test.json), il ne le remplace pas. Indices de prix sans dividendes (des deux côtés).

Données (data/1_brutes/depuis_2000) : Nikkei 225, S&P 500, VIX, dollar-yen (Yahoo Finance), euro-yen et Euribor
3 mois (BCE), taux 3 mois japonais (OCDE, à partir d'avril 2002 ; avant, la première valeur connue est reprise).
"""
import argparse
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

from .config import ROOT, RUNS, commit_courant
from .volatilite import Garch

DOSSIER = ROOT / "data" / "1_brutes" / "depuis_2000"
REGLE = RUNS / "signal_risque_regle.json"
SORTIE = RUNS / "signal_risque.json"
SERIE = RUNS / "signal_risque_journalier.csv"

DEBUT, FIN_REGLAGE = "2000-01-01", "2015-12-31"
PERIODES = {"2003-2015 (réglage)": ("2003-01-01", "2015-12-31"), "2016-2023": ("2016-01-01", "2023-12-31"),
            "2024-2026": ("2024-01-01", "2026-12-31"), "2003-2026 (tout)": ("2003-01-01", "2026-12-31")}
HORIZON = 20            # séances : la volatilité prévue est celle du mois qui vient
DECALAGE = 1            # on décide le soir t, on exécute à la clôture de t+1 : la position gagne le rendement de t+2
PAS = 20                # réentraînement toutes les 20 séances, sur tout le passé disponible
MINIMUM = 756           # trois ans d'historique avant la première prévision
QUANTILES = (0.60, 0.90)   # 60 % des jours « calmes », 30 % « moyens », 10 % « très risqués » sur 2003-2015
CONFIRMATION = 3        # soirs de suite dans la nouvelle zone avant de changer de niveau
FRAIS = 0.0005          # 0,05 % par 100 % acheté ou vendu (ETF Nikkei coté à Tokyo, écart compris)


# ---------------------------------------------------------------- données
def charger() -> pd.DataFrame:
    n = pd.read_csv(DOSSIER / "nikkei225_ohlc.csv", index_col=0, parse_dates=True).loc[DEBUT:].dropna(subset=["Close"])
    n = n[n["Close"].pct_change().abs().fillna(0) < 0.2]
    m = pd.read_csv(DOSSIER / "yahoo_marches.csv", index_col=0, parse_dates=True)
    ecb = pd.read_csv(DOSSIER / "ecb_eurjpy.csv", usecols=["TIME_PERIOD", "OBS_VALUE"], parse_dates=["TIME_PERIOD"]).set_index("TIME_PERIOD")["OBS_VALUE"].sort_index()
    eur = pd.read_csv(DOSSIER / "ecb_euribor3m.csv", usecols=["TIME_PERIOD", "OBS_VALUE"]).assign(d=lambda x: pd.to_datetime(x.TIME_PERIOD) + pd.offsets.MonthEnd(0)).set_index("d")["OBS_VALUE"].sort_index()
    jp = pd.read_csv(DOSSIER / "oecd_taux3m_japon.csv", usecols=["TIME_PERIOD", "OBS_VALUE"]).assign(d=lambda x: pd.to_datetime(x.TIME_PERIOD) + pd.offsets.MonthEnd(0)).set_index("d")["OBS_VALUE"].sort_index()

    t = n.index
    veille = t - pd.Timedelta(days=1)                      # Wall Street et la BCE de la veille : connus avant la clôture de Tokyo
    d = pd.DataFrame(index=t)
    d["close"], d["high"], d["low"] = n["Close"], n["High"], n["Low"]
    d["r"] = np.log(d["close"]).diff()
    for col in ["sp500", "vix", "jpy_par_usd"]:
        s = m[col].dropna()
        d[col + "_veille"] = s.asof(veille).to_numpy()
    d["eurjpy"] = ecb.asof(t).to_numpy()                     # sert seulement à compter les rendements en euros
    # taux mensuels : le taux du mois m est connu à la fin du mois m, utilisé à partir du mois suivant
    d["euribor"] = eur.asof(t - pd.offsets.MonthEnd(1)).to_numpy()
    d["taux_japon"] = jp.asof(t - pd.offsets.MonthEnd(1)).to_numpy()
    d["taux_japon"] = d["taux_japon"].bfill()
    return d


def variables(d: pd.DataFrame) -> pd.DataFrame:
    """Tout ce qui est connu le soir t, à la clôture de Tokyo. Volatilités annualisées, en logarithme."""
    vol = lambda r, n: np.log(np.sqrt(252 * (r ** 2).rolling(n).mean()).clip(lower=0.02))
    x = pd.DataFrame(index=d.index)
    x["vol_1j"] = np.log((d["r"].abs() * np.sqrt(252)).clip(lower=0.02))
    x["vol_5j"], x["vol_22j"], x["vol_66j"] = vol(d["r"], 5), vol(d["r"], 22), vol(d["r"], 66)
    park = np.log(d["high"] / d["low"]) ** 2 / (4 * np.log(2))                # écart haut-bas de la séance (Parkinson)
    x["vol_haut_bas_5j"] = np.log(np.sqrt(252 * park.rolling(5).mean()).clip(lower=0.02))
    x["vix_veille"] = np.log(d["vix_veille"])
    x["vol_sp500_5j"] = vol(np.log(d["sp500_veille"]).diff(), 5)
    x["vol_dollar_yen_22j"] = vol(np.log(d["jpy_par_usd_veille"]).diff(), 22)
    x["rendement_22j"] = np.log(d["close"]).diff(22)                           # les baisses font monter le risque
    x["recul_depuis_le_plus_haut"] = d["close"] / d["close"].rolling(252, min_periods=60).max() - 1
    return x


def cible(d: pd.DataFrame) -> pd.Series:
    """Log de la volatilité réalisée sur la fenêtre où la position est tenue : séances t+2 à t+21."""
    r2 = d["r"] ** 2
    futur = r2[::-1].rolling(HORIZON).mean()[::-1].shift(-(1 + DECALAGE))
    return np.log(np.sqrt(252 * futur).clip(lower=0.02))


MODELES = {"naïf : la volatilité du dernier mois": ["vol_22j"],
           "HAR : volatilité du jour, de la semaine, du mois, du trimestre": ["vol_1j", "vol_5j", "vol_22j", "vol_66j"],
           "HAR + écart haut-bas + marchés (VIX, S&P 500, dollar-yen) + baisses": None,
           "GARCH(1,1) journalier": "garch"}


def marcher(d: pd.DataFrame, x: pd.DataFrame, y: pd.Series, fin: str | None = None) -> pd.DataFrame:
    """Prévisions hors échantillon : à chaque réentraînement, seules les lignes dont la cible est déjà connue servent."""
    idx = x.dropna().index.intersection(d.index[d["r"].notna()])
    if fin:
        idx = idx[idx <= fin]
    pos = np.arange(len(idx))
    pred = {nom: pd.Series(np.nan, index=idx) for nom in MODELES}
    tout_x = x.loc[idx]
    for debut in range(MINIMUM, len(idx), PAS):
        bloc = idx[debut:debut + PAS]
        connu = idx[:max(0, debut - HORIZON - DECALAGE)]                       # cibles entièrement observées le jour du réentraînement
        connu = connu[y.loc[connu].notna().to_numpy()]
        for nom, cols in MODELES.items():
            if cols == "garch":
                r = d["r"].loc[:idx[debut - 1]].dropna().to_numpy()
                g = Garch().ajuster(r[-2520:])                                  # dix ans de rendements
                for jour in bloc:
                    rj = d["r"].loc[:jour].dropna().to_numpy()[-2520:]
                    pred[nom].loc[jour] = np.log(g.prevoir(rj, HORIZON + DECALAGE))
                continue
            cols = cols or list(x.columns)
            mod = LinearRegression().fit(tout_x.loc[connu, cols], y.loc[connu])
            pred[nom].loc[bloc] = mod.predict(tout_x.loc[bloc, cols])
    return pd.DataFrame(pred).dropna()


def r2(pred: pd.Series, y: pd.Series) -> float:
    ok = y.reindex(pred.index).notna()
    p, v = pred[ok], y.reindex(pred.index)[ok]
    return float(1 - ((v - p) ** 2).sum() / ((v - v.mean()) ** 2).sum())


# ---------------------------------------------------------------- règle
def niveaux(vol_prevue: pd.Series, seuils, confirmation: int, fin_de_mois: bool = False) -> pd.Series:
    """100 %, 50 % ou 0 % selon la zone de risque ; on change de niveau après `confirmation` soirs dans la nouvelle zone."""
    zone = pd.Series(np.select([vol_prevue < seuils[0], vol_prevue < seuils[1]], [1.0, 0.5], 0.0), index=vol_prevue.index)
    if fin_de_mois:
        decide = zone.index.to_series().groupby(zone.index.to_period("M")).transform("max") == zone.index.to_series()
        return zone.where(decide).ffill().fillna(1.0)
    sortie, actuel, attente, candidat = [], 1.0, 0, None
    for z in zone.to_numpy():
        if z == actuel:
            attente, candidat = 0, None
        elif z == candidat:
            attente += 1
        else:
            candidat, attente = z, 1
        if candidat is not None and attente >= confirmation:
            actuel, attente, candidat = candidat, 0, None
        sortie.append(actuel)
    return pd.Series(sortie, index=zone.index)


def rendements(d: pd.DataFrame) -> pd.DataFrame:
    """Rendements quotidiens pour un investisseur en euros : du Nikkei couvert, non couvert, et de l'argent laissé en euros."""
    jours = d.index.to_series().diff().dt.days.fillna(1)
    cash = d["euribor"] / 100 * jours / 360
    r_local = d["close"].pct_change()
    couvert = r_local + (d["euribor"] - d["taux_japon"]) / 100 * jours / 360       # le contrat de change rapporte l'écart de taux
    non_couvert = (1 + r_local) * d["eurjpy"].shift(1) / d["eurjpy"] - 1
    return pd.DataFrame({"euros": cash, "couvert": couvert, "non_couvert": non_couvert})


def appliquer(position: pd.Series, r: pd.DataFrame, cote: str, frais: float) -> pd.Series:
    """Position décidée le soir t, exécutée à la clôture de t+1, donc rendement de t+2."""
    w = position.shift(1 + DECALAGE).reindex(r.index).dropna()
    rr = r.loc[w.index]
    cout = w.diff().abs().fillna(0) * frais
    return w * rr[cote] + (1 - w) * rr["euros"] - cout


def bilan(g: pd.Series, cash: pd.Series, w: pd.Series) -> dict:
    v = (1 + g).cumprod()
    ans = len(g) / 252
    exces = g - cash.reindex(g.index)
    mois = (1 + g).groupby(g.index.to_period("M")).prod() - 1
    return {"rendement_annuel_%": float((v.iloc[-1] ** (1 / ans) - 1) * 100), "cumul_%": float((v.iloc[-1] - 1) * 100),
            "volatilite_%": float(g.std() * np.sqrt(252) * 100),
            "rendement_par_unite_de_risque": float(exces.mean() / exces.std() * np.sqrt(252)),
            "perte_maximale_%": float((v / v.cummax() - 1).min() * 100), "pire_mois_%": float(mois.min() * 100),
            "part_investie_moyenne_%": float(w.reindex(g.index).mean() * 100),
            "changements_par_an": float((w.reindex(g.index).diff().abs() > 0).sum() / ans)}


def ecart_ratio(a: pd.Series, b: pd.Series, cash: pd.Series, tirages: int = 1000, bloc: int = 20) -> dict:
    """Écart de rendement par unité de risque (a - b) et son incertitude, par tirage de blocs de 20 séances."""
    ea, eb = (a - cash.reindex(a.index)).to_numpy(), (b - cash.reindex(b.index)).to_numpy()
    s = lambda e: e.mean() / e.std() * np.sqrt(252)
    rng, n = np.random.default_rng(0), len(ea)
    tir = []
    for _ in range(tirages):
        debuts = rng.integers(0, n - bloc, n // bloc + 1)
        i = (debuts[:, None] + np.arange(bloc)).ravel()[:n]
        tir.append(s(ea[i]) - s(eb[i]))
    return {"ecart": float(s(ea) - s(eb)), "erreur_type": float(np.std(tir)),
            "part_des_tirages_ou_le_signal_fait_mieux_%": float(np.mean(np.array(tir) > 0) * 100)}


# ---------------------------------------------------------------- les deux temps
def figer() -> None:
    if REGLE.exists():
        raise SystemExit(f"{REGLE.relative_to(ROOT)} existe déjà : la règle est figée. Supprimez-la seulement si vous assumez de la refaire.")
    d = charger()
    d = d.loc[:FIN_REGLAGE]
    x, y = variables(d), cible(d)
    pred = marcher(d, x, y, FIN_REGLAGE)
    scores = {nom: r2(pred[nom], y) for nom in MODELES}
    choisi = max(scores, key=scores.get)
    vol = np.exp(pred[choisi])
    seuils = [float(vol.quantile(q)) for q in QUANTILES]
    regle = {"figee_le": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(),
             "donnees_utilisees": f"{DEBUT} -> {FIN_REGLAGE} uniquement", "r2_hors_echantillon_2003_2015": scores,
             "modele": choisi, "seuils_volatilite_annuelle": seuils,
             "lecture": f"risque prévu < {seuils[0]:.1%} : acheter plein ; entre les deux : acheter à moitié ; > {seuils[1]:.1%} : ne pas acheter",
             "quantiles": QUANTILES, "confirmation_soirs": CONFIRMATION, "frais_par_100_%_echange": FRAIS,
             "horizon_seances": HORIZON, "execution": "décision le soir t, achat ou vente à la clôture de t+1"}
    REGLE.write_text(json.dumps(regle, indent=2, ensure_ascii=False))
    print(json.dumps(regle, indent=2, ensure_ascii=False))


def evaluer() -> None:
    if not REGLE.exists():
        raise SystemExit("Lancez d'abord : python -m src.signal_risque --figer")
    regle = json.loads(REGLE.read_text())
    d = charger()
    x, y = variables(d), cible(d)
    pred = marcher(d, x, y)
    vol = np.exp(pred[regle["modele"]])
    seuils, conf = regle["seuils_volatilite_annuelle"], regle["confirmation_soirs"]
    r = rendements(d)
    cible_vol = float(np.sqrt(seuils[0] * seuils[1]))                                   # pour la version dosée en continu
    positions = {"acheter et garder": pd.Series(1.0, index=vol.index),
                 "signal risque, chaque jour (règle figée)": niveaux(vol, seuils, conf),
                 "signal risque, chaque jour, sans confirmation": niveaux(vol, seuils, 1),
                 "signal risque, seulement en fin de mois": niveaux(vol, seuils, 1, fin_de_mois=True),
                 "position dosée en continu (risque visé / risque prévu)": (cible_vol / vol).clip(upper=1.0)}

    sortie = {"avertissement": "Volet construit après l'ouverture du test 2024-2026 du projet. Il s'ajoute au résultat officiel "
                               "(runs/evaluation_test.json), il ne le remplace pas. Règle choisie avec les seules données 2000-2015.",
              "date": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(), "regle": regle,
              "prevision_r2_log_volatilite": {}, "strategies": {}, "comparaisons": {}, "sensibilite": {}}
    for per, (a, b) in PERIODES.items():
        sortie["prevision_r2_log_volatilite"][per] = {nom: r2(pred[nom].loc[a:b], y) for nom in MODELES}

    for cote in ["couvert", "non_couvert"]:
        gains = {nom: appliquer(w, r, cote, FRAIS if nom != "acheter et garder" else 0.0) for nom, w in positions.items()}
        for per, (a, b) in PERIODES.items():
            for nom, g in gains.items():
                sortie["strategies"].setdefault(cote, {}).setdefault(per, {})[nom] = bilan(g.loc[a:b], r["euros"], positions[nom])
            sortie["comparaisons"].setdefault(cote, {})[per] = ecart_ratio(gains["signal risque, chaque jour (règle figée)"].loc[a:b],
                                                                          gains["acheter et garder"].loc[a:b], r["euros"])
    # Sensibilité (information seulement : ces variantes n'ont pas servi à choisir la règle)
    for q in [(0.5, 0.8), (0.6, 0.9), (0.7, 0.95)]:
        s = [float(vol.loc[:FIN_REGLAGE].quantile(v)) for v in q]
        for c in [1, 3, 5]:
            g = appliquer(niveaux(vol, s, c), r, "couvert", FRAIS).loc["2016":]
            sortie["sensibilite"][f"quantiles {q}, confirmation {c}"] = round(bilan(g, r["euros"], niveaux(vol, s, c))["rendement_par_unite_de_risque"], 3)

    dernier = vol.index[-1]
    w = positions["signal risque, chaque jour (règle figée)"]
    sortie["aujourd_hui"] = {"date": str(dernier.date()), "volatilite_prevue_%": float(vol.iloc[-1] * 100),
                             "zone_du_jour": {1.0: "risque bas", 0.5: "risque moyen", 0.0: "risque très élevé"}[niveaux(vol, seuils, 1).iloc[-1]],
                             "signal": {1.0: "acheter plein", 0.5: "acheter à moitié", 0.0: "ne pas acheter"}[w.iloc[-1]]}
    SORTIE.write_text(json.dumps(sortie, indent=2, ensure_ascii=False))
    pd.DataFrame({"volatilite_prevue": vol, "volatilite_realisee": np.exp(y.reindex(vol.index)),
                  "signal": w, **{f"valeur_{k}": (1 + appliquer(v, r, "couvert", FRAIS if k != "acheter et garder" else 0)).cumprod()
                                  for k, v in positions.items()}}).to_csv(SERIE, float_format="%.5g")

    print("R² hors échantillon du log de la volatilité :")
    for per, s in sortie["prevision_r2_log_volatilite"].items():
        print(f"  {per:20s}", "  ".join(f"{k.split(':')[0][:14]} {v:+.2f}" for k, v in s.items()))
    for cote, pp in sortie["strategies"].items():
        print(f"\n=== investisseur en euros, {cote.replace('_', ' ')}")
        for per, st in pp.items():
            c = sortie["comparaisons"][cote][per]
            print(f"  {per}   (écart de ratio signal - garder : {c['ecart']:+.2f} ± {c['erreur_type']:.2f})")
            for nom, b in st.items():
                print(f"    {nom[:52]:52s} {b['rendement_annuel_%']:+6.1f} %/an  vol {b['volatilite_%']:4.1f}  ratio {b['rendement_par_unite_de_risque']:+.2f}"
                      f"  perte max {b['perte_maximale_%']:6.1f}  pire mois {b['pire_mois_%']:6.1f}  investi {b['part_investie_moyenne_%']:3.0f} %  {b['changements_par_an']:4.1f} chgt/an")
    print("\nsensibilité (ratio 2016-2026, couvert) :", sortie["sensibilite"])
    print("aujourd'hui :", sortie["aujourd_hui"])
    print(f"écrit : {SORTIE.relative_to(ROOT)}, {SERIE.relative_to(ROOT)}")


if __name__ == "__main__":
    warnings.filterwarnings("ignore")
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--figer", action="store_true", help="choisir et figer la règle avec les données 2000-2015")
    figer() if p.parse_args().figer else evaluer()

In [ ]:
# ─── src/signal_risque_variantes.py · 138 lignes ───

"""Trois améliorations du signal fondé sur le risque, plus leur combinaison, testées sans tricher.

Les variantes ont été écrites AVANT de voir leurs résultats, pour une raison de principe :
  A. Levier en période calme : part investie = c / variance prévue, jusqu'à 150 % (contrats à terme). c est choisi pour
     que la part investie moyenne vaille 100 % sur 2003-2015 (aucun rendement n'intervient dans ce choix).
  B. Sortir lentement, revenir vite : on remonte d'un niveau dès le premier soir, on descend après trois soirs ; et l'on
     ne sort entièrement que si le cours est aussi sous sa moyenne des 200 dernières séances (filtre de tendance).
  C. Décider seulement en fin de mois, avec la règle figée de src.signal_risque.
  D. Les trois ensemble : part c / variance prévue revue en fin de mois, jusqu'à 150 %, ramenée à 50 % au plus quand le
     cours est sous sa moyenne 200 séances et le risque très élevé.

Protocole : `--figer` calcule c avec les seules données 2000-2015 et écrit runs/signal_risque_variantes_regle.json ;
sans option, le script applique ces réglages figés à 2003-2026. Toutes les variantes sont publiées. Quatre essais au lieu
d'un augmentent la chance qu'un d'eux réussisse par hasard : un écart doit dépasser nettement son erreur type.
Volet après coup : il ne remplace pas le résultat du projet.
"""
import argparse
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

from . import signal_risque as S
from .config import ROOT, RUNS, commit_courant

REGLE = RUNS / "signal_risque_variantes_regle.json"
SORTIE = RUNS / "signal_risque_variantes.json"
PLAFOND = 1.5
MOYENNE_LONGUE = 200


def fin_de_mois(w: pd.Series) -> pd.Series:
    """Garde la valeur du dernier jour de chaque mois, tenue tout le mois suivant."""
    dernier = w.index.to_series().groupby(w.index.to_period("M")).transform("max") == w.index.to_series()
    return w.where(dernier).ffill().bfill()


def variante_b(vol: pd.Series, tendance_baissiere: pd.Series, seuils) -> pd.Series:
    zone = pd.Series(np.select([vol < seuils[0], vol < seuils[1]], [1.0, 0.5], 0.0), index=vol.index)
    zone = zone.where(~((zone == 0.0) & ~tendance_baissiere.reindex(vol.index).fillna(False)), 0.5)   # pas de sortie totale en tendance haussière
    sortie, actuel, attente = [], 1.0, 0
    for z in zone.to_numpy():
        if z > actuel:
            actuel, attente = z, 0                       # revenir : tout de suite
        elif z < actuel:
            attente += 1
            if attente >= S.CONFIRMATION:                # sortir : après trois soirs
                actuel, attente = z, 0
        else:
            attente = 0
        sortie.append(actuel)
    return pd.Series(sortie, index=vol.index)


def positions(vol: pd.Series, d: pd.DataFrame, seuils, c: float) -> dict:
    baissier = d["close"] < d["close"].rolling(MOYENNE_LONGUE).mean()
    levier = (c / vol ** 2).clip(upper=PLAFOND)
    combine = fin_de_mois(levier)
    tres_risque = fin_de_mois(pd.Series((vol >= seuils[1]).astype(float), index=vol.index)) > 0
    combine = combine.where(~(tres_risque & fin_de_mois(baissier.reindex(vol.index).astype(float)).astype(bool)), combine.clip(upper=0.5))
    return {"acheter et garder": pd.Series(1.0, index=vol.index),
            "règle figée (jour, 3 niveaux)": S.niveaux(vol, seuils, S.CONFIRMATION),
            "A. levier en période calme (c / variance, jusqu'à 150 %)": levier,
            "B. sortir lentement, revenir vite, filtre de tendance": variante_b(vol, baissier, seuils),
            "C. décision en fin de mois": S.niveaux(vol, seuils, 1, fin_de_mois=True),
            "D. les trois ensemble": combine}


def prevoir(fin=None):
    d = S.charger()
    if fin:
        d = d.loc[:fin]
    regle = json.loads(S.REGLE.read_text())
    x, y = S.variables(d), S.cible(d)
    pred = S.marcher(d, x, y, fin)
    return d, regle, np.exp(pred[regle["modele"]])


def figer() -> None:
    if REGLE.exists():
        raise SystemExit(f"{REGLE.relative_to(ROOT)} existe déjà : les variantes sont figées.")
    _, regle, vol = prevoir(S.FIN_REGLAGE)
    c = float(1 / (1 / vol.loc["2003":S.FIN_REGLAGE] ** 2).mean())        # part moyenne de 100 % sur 2003-2015
    sortie = {"figee_le": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(),
              "donnees_utilisees": f"{S.DEBUT} -> {S.FIN_REGLAGE} uniquement", "c": c,
              "lecture_c": f"part investie = {c:.4f} / variance prévue, soit 100 % quand la volatilité prévue vaut {np.sqrt(c):.1%}",
              "plafond": PLAFOND, "moyenne_longue_seances": MOYENNE_LONGUE, "seuils_repris_de": "runs/signal_risque_regle.json",
              "seuils": regle["seuils_volatilite_annuelle"], "frais": S.FRAIS}
    REGLE.write_text(json.dumps(sortie, indent=2, ensure_ascii=False))
    print(json.dumps(sortie, indent=2, ensure_ascii=False))


def evaluer() -> None:
    if not REGLE.exists():
        raise SystemExit("Lancez d'abord : python -m src.signal_risque_variantes --figer")
    fige = json.loads(REGLE.read_text())
    d, regle, vol = prevoir()
    r = S.rendements(d)
    pos = positions(vol, d, fige["seuils"], fige["c"])
    resultats = {}
    for cote in ["couvert", "non_couvert"]:
        gains = {nom: S.appliquer(w, r, cote, 0.0 if nom == "acheter et garder" else S.FRAIS) for nom, w in pos.items()}
        for per, (a, b) in S.PERIODES.items():
            bloc = resultats.setdefault(cote, {}).setdefault(per, {})
            for nom, g in gains.items():
                g = g.loc[a:b]
                ligne = S.bilan(g, r["euros"], pos[nom])
                if nom != "acheter et garder":
                    part = pos[nom].loc[a:b].mean()                                         # même part moyenne, mais fixe
                    fixe = S.appliquer(pd.Series(part, index=vol.index), r, cote, 0.0).loc[a:b]
                    ligne["face_a_acheter_et_garder"] = S.ecart_ratio(g, gains["acheter et garder"].loc[a:b], r["euros"])
                    ligne["face_a_une_part_fixe_equivalente"] = S.ecart_ratio(g, fixe, r["euros"]) | {"part_fixe_%": float(part * 100)}
                bloc[nom] = ligne
    sortie = {"avertissement": "Volet après coup : variantes écrites avant de voir leurs résultats, réglées sur 2000-2015, toutes publiées. "
                               "Il ne remplace pas runs/evaluation_test.json.",
              "date": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(), "reglages_figes": fige,
              "resultats": resultats,
              "aujourd_hui": {nom: float(w.iloc[-1]) for nom, w in pos.items()}}
    SORTIE.write_text(json.dumps(sortie, indent=2, ensure_ascii=False))
    for cote in ["couvert"]:
        for per, bloc in resultats[cote].items():
            print(f"\n=== {cote}, {per}")
            for nom, b in bloc.items():
                e = b.get("face_a_acheter_et_garder"), b.get("face_a_une_part_fixe_equivalente")
                comp = f"  vs garder {e[0]['ecart']:+.2f}±{e[0]['erreur_type']:.2f}  vs part fixe {e[1]['part_fixe_%']:.0f}% {e[1]['ecart']:+.2f}±{e[1]['erreur_type']:.2f}" if e[0] else ""
                print(f"  {nom[:44]:44s} {b['rendement_annuel_%']:+5.1f}%/an vol {b['volatilite_%']:4.1f} ratio {b['rendement_par_unite_de_risque']:+.2f} "
                      f"perte max {b['perte_maximale_%']:6.1f} investi {b['part_investie_moyenne_%']:3.0f}% {b['changements_par_an']:5.1f}/an{comp}")
    print("\naujourd'hui :", {k[:20]: round(v, 2) for k, v in sortie["aujourd_hui"].items()})
    print(f"écrit : {SORTIE.relative_to(ROOT)}")


if __name__ == "__main__":
    warnings.filterwarnings("ignore")
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--figer", action="store_true")
    figer() if p.parse_args().figer else evaluer()

---
<a id="partie-6"></a>
## Partie 6 · Annexe news × prix

### 6.1 · Signal des news, signal des prix, régimes cachés

Étude préalable (dossier `notebooks/06_annexe_news_prix/`) : le signal des news et celui des prix, le temps qu'ils mettent à se réaliser, des backtests de stratégies et un modèle de Markov caché pour les régimes.

> **Concepts** · corrélations décalées, backtest, HMM (états cachés, matrice de transition, algorithme de Viterbi).

In [ ]:
# ─── notebooks/06_annexe_news_prix/nikkei_lab.py · 1248 lignes ───

"""
nikkei_lab — moteur du notebook « Nikkei : signal news × signal prix ».

Contenu
-------
1. Chargement des données (marché + articles)                     -> load_data
2. Signal NEWS  (sentiment FinBERT des articles / ton GDELT)        -> news_signal
3. Signal PRIX  (tendance, momentum, moyennes mobiles, drawdown,   -> price_signal
                 régimes de marché par HMM)                        -> hmm_regimes (moteur : nikkei_hmm.py)
4. Combinaison + passage signal -> position + backtest réaliste     -> run_strategy
5. Analyse des horizons : en combien de temps un signal se réalise  -> horizon_profile, placebo_ic,
                                                                       decay_matrix, event_study, holding_curve
6. Optimisation : grille, IS/OOS, walk-forward, plateau, Deflated Sharpe
7. Méta-modèle ML walk-forward purgé (bonus)

Convention temporelle (importante pour éviter le biais d'anticipation)
--------------------------------------------------------------------
* Toute série indexée en t n'utilise que l'information disponible à la clôture de la séance t.
  Les articles sont déjà rattachés à leur séance (`session`) : ceux vus avant ~15h JST le jour t.
* La position décidée en t est exécutée à la clôture de t + exec_lag (exec_lag=1 par défaut),
  puis encaisse les rendements des jours suivants.
"""
from __future__ import annotations

import itertools
import math
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy import stats
from scipy.ndimage import uniform_filter
from scipy.signal import lfilter

import nikkei_hmm as nh

TD = 252  # jours de bourse par an

# =============================================================================
# 0. Paramètres par défaut (tous modifiables depuis le notebook)
# =============================================================================
DEFAULT_PARAMS: dict = dict(
    # ---------------- NEWS ----------------
    news_source="articles",   # 'articles' (FinBERT) | 'gdelt' (ton GDELT) | 'blend'
    news_score="prob",        # score article : 'prob' (p_pos-p_neg) | 'vote' (+1/0/-1) | 'neg' | 'pos'
    news_mode="level",        # 'level' (niveau du sentiment) | 'momentum' (rapide - lent) | 'attention' (volume anormal)
    news_halflife=20,         # demi-vie (en séances) de la mémoire des news
    news_slow_mult=4,         # momentum/attention : demi-vie lente = halflife * slow_mult
    news_prior=5.0,           # shrinkage : nb d'articles « fictifs » à la moyenne long terme
    news_w_boj=1.0,           # poids des articles thème BoJ
    news_w_japan=1.0,         # poids des articles thème Japon
    news_min_conf=0.0,        # filtre : garde les articles avec 1 - p_neu >= min_conf
    news_conf_power=0.0,      # pondère chaque article par (1 - p_neu) ** power
    news_dedup_days=3,        # supprime un titre déjà vu dans les N jours (0 = off)
    # --- qualité de la collecte (voir collection_quality) ---
    news_gap_min_run=2,       # N séances d'affilée sans article (ou panne GDELT signalée) = trou de collecte
                              # -> donnée manquante (0 = off : une séance vide compte comme « calme »)
    news_burst_mult=3.0,      # pic = nb d'articles > mult × médiane des 60 séances précédentes, écrêté à ce niveau (0 = off)
    news_burst_confirm=True,  # un pic que l'attention GDELT (mesurée sur tout le corpus) ne confirme pas = donnée manquante
    news_burst_confirm_z=0.0, # seuil de confirmation : max(attention GDELT Japon, BoJ) en z > seuil
    news_norm="rolling",      # normalisation : 'rolling' | 'expanding' | 'none'
    news_z_window=250,        # fenêtre de normalisation (séances)
    news_gdelt_weight=0.5,    # poids du GDELT si news_source='blend'
    news_direction=1,         # +1 = suivre le sentiment, -1 = contrarien
    # ---------------- PRIX ----------------
    price_kind="tsmom",       # 'tsmom' | 'composite' | 'ma' | 'sma_dist' | 'breakout' | 'drawdown' | 'hmm'
    price_lookback=120,       # fenêtre principale (séances)
    price_lookbacks=(20, 60, 120, 250),  # pour 'composite' : moyenne de tsmom sur ces fenêtres
    price_skip=0,             # tsmom : ignore les N dernières séances (12-1 momentum…)
    price_fast=20,            # 'ma' : EMA rapide
    price_slow=100,           # 'ma' : EMA lente
    price_vol_halflife=40,    # volatilité utilisée pour normaliser
    price_norm="auto",        # 'auto' | 'rms' (sans dé-moyenner) | 'rolling' | 'expanding' | 'none'
                              # (auto : 'rolling' pour drawdown/breakout, 'none' pour le régime HMM, sinon 'rms')
    price_z_window=500,
    price_direction=1,        # +1 = suivi de tendance, -1 = retour à la moyenne
    # ---------------- PRIX : régimes HMM (price_kind='hmm', voir nikkei_hmm.py) ----------------
    price_hmm_features="ret_vol",  # variables observées : 'ret' (rendement seul) | 'ret_vol' (+ log volatilité réalisée)
                                   # | 'trend_vol' (rendement moyen récent + log volatilité)
    price_hmm_states=2,            # nombre de régimes (2 = calme / agité ; 3 = calme / intermédiaire / agité)
    price_hmm_signal="regime",     # 'regime' : P(calme) - P(agité) | 'exp_ret' : rendement attendu par le modèle
    price_hmm_horizon=60,          # 'exp_ret' : horizon de la prévision (séances)
    price_hmm_sticky=0.0,          # pseudo-transitions ajoutées sur la diagonale : régimes plus persistants
    price_hmm_vol_halflife=10,     # demi-vie (séances) de la volatilité réalisée et de la tendance observées
    price_hmm_min_train=250,       # séances d'historique avant la première estimation
    price_hmm_refit=63,            # ré-estimation toutes les N séances, sur le passé seulement
    price_hmm_window=0,            # 0 = fenêtre croissante ; sinon N dernières séances
    # ---------------- COMBINAISON ----------------
    combo="linear",           # 'linear' | 'agree' | 'filter'
    w_news=0.5,               # poids du signal news (0 = prix seul, 1 = news seul)
    filter_threshold=-1.0,    # combo='filter' : si news < seuil, l'exposition est multipliée par filter_scale
    filter_scale=0.0,
    clip_z=3.0,               # écrêtage des signaux (en écarts-types)
    # ---------------- POSITION ----------------
    mapping="threshold",      # 'threshold' (hystérésis) | 'linear' | 'sigmoid'
    long_only=True,
    entry=0.0,                # threshold : entre long si score > entry
    exit=-0.25,               # threshold : sort si score < exit (hystérésis)
    scale=1.0,                # linear/sigmoid : pente
    base=0.0,                 # exposition « neutre » (threshold : exposition quand pas de signal)
    max_leverage=1.0,
    rebalance=5,              # on ne regarde le signal que toutes les N séances
    min_trade=0.0,            # bande de non-transaction (variation minimale d'exposition)
    vol_target=0.0,           # 0 = off ; sinon volatilité annuelle cible (ex. 0.15)
    vol_target_halflife=20,
    # ---------------- FILTRE DE RISQUE (surcouche multiplicative sur l'exposition) ----------------
    risk_overlay="none",      # 'none' | 'hmm' : exposition × (1 - overlay_k × P(régime agité du HMM price_hmm_*))
                              # | 'vol' : idem avec 1[volatilité réalisée > sa médiane passée] (référence simple)
    overlay_k=0.5,            # force de la réduction : 0,5 = exposition divisée par 2 en régime agité certain
    overlay_vol_halflife=10,  # 'vol' : demi-vie (séances) de la volatilité réalisée
    # ---------------- EXÉCUTION ----------------
    exec_lag=1,               # séances entre la décision et l'exécution (à la clôture)
    cost_bps=5.0,             # coût par unité d'exposition échangée (bps)
    use_cash_rate=True,       # le cash rapporte le TONA (taux au jour le jour JPY)
    div_yield=0.0,            # dividendes annuels de l'indice quand investi (~0.017 pour approx. total return)
)

NEWS_KEYS = [k for k in DEFAULT_PARAMS if k.startswith("news_")] + ["clip_z"]
PRICE_KEYS = [k for k in DEFAULT_PARAMS if k.startswith("price_")] + ["clip_z"]


def make_params(params: dict | None = None, **overrides) -> dict:
    """Fusionne les paramètres avec les valeurs par défaut et refuse les clés inconnues (fautes de frappe)."""
    p = dict(DEFAULT_PARAMS)
    for src in (params or {}), overrides:
        unknown = set(src) - set(DEFAULT_PARAMS)
        if unknown:
            raise KeyError(f"Paramètre(s) inconnu(s) : {sorted(unknown)}")
        p.update(src)
    return p


# =============================================================================
# 1. Données
# =============================================================================
class Data:
    """Données de marché + articles, avec un cache des signaux (accélère grilles et widgets)."""

    def __init__(self, market: pd.DataFrame, articles: pd.DataFrame):
        self.market = market
        self.articles = articles
        self.close = market["n225_close"].astype(float)
        self.ret = self.close.pct_change().fillna(0.0)
        rf = market["tona"] / 100 / TD if "tona" in market else pd.Series(0.0, index=market.index)
        self.rf = rf.ffill().fillna(0.0)
        sessions = pd.DatetimeIndex(articles["session"].unique())
        # calendrier des news : séances du marché + séances d'articles (avant le début / après la fin du marché)
        self.calendar = market.index.union(sessions).sort_values()
        self._cache: dict = {}

    def clear_cache(self):
        self._cache.clear()

    def __repr__(self):
        s = f"Data(marché {self.market.index[0].date()} -> {self.market.index[-1].date()}, {len(self.market)} séances"
        if len(self.articles):
            s += (f" ; {len(self.articles)} articles "
                  f"{self.articles['session'].min().date()} -> {self.articles['session'].max().date()}")
        return s + ")"


def load_price_history(path) -> Data:
    """Historique de cours seul (CSV date, close), par ex. le Nikkei 225 depuis 1970 : pour les analyses côté prix
    (HMM, momentum, filtre de risque). Sans articles ni taux : le cash ne rapporte rien (rf = 0)."""
    c = pd.read_csv(path, parse_dates=["date"]).set_index("date").sort_index()["close"].astype(float)
    market = c[~c.index.duplicated(keep="last")].dropna().to_frame("n225_close")
    return Data(market, pd.DataFrame({"session": pd.to_datetime(pd.Series([], dtype="object"))}))


def fetch_yahoo_history(symbol: str = "^N225", path="../../data/1_brutes/nikkei225_long.csv", start: str = "1970-01-01") -> pd.DataFrame:
    """Met à jour l'historique long depuis l'API publique de graphiques de Yahoo Finance (clôtures quotidiennes).
    Écrit un CSV (date, close) et le renvoie."""
    import json
    import urllib.parse
    import urllib.request
    p1 = int(pd.Timestamp(start).timestamp())
    url = (f"https://query1.finance.yahoo.com/v8/finance/chart/{urllib.parse.quote(symbol)}"
           f"?period1={max(p1, 0)}&period2=4102444800&interval=1d")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        r = json.load(resp)["chart"]["result"][0]
    tz = r["meta"].get("exchangeTimezoneName", "Asia/Tokyo")
    dates = pd.to_datetime(r["timestamp"], unit="s", utc=True).tz_convert(tz).date
    df = pd.DataFrame({"close": r["indicators"]["quote"][0]["close"]}, index=pd.DatetimeIndex(dates, name="date"))
    df = df[~df.index.duplicated(keep="last")].dropna()
    df.to_csv(path, float_format="%.2f")
    return df


def load_data(market_path, articles_path) -> Data:
    m = pd.read_csv(market_path, parse_dates=["date"]).set_index("date").sort_index()
    m = m[~m.index.duplicated(keep="last")]
    a = pd.read_csv(articles_path, parse_dates=["session", "seendate"])
    a["title"] = a["title"].fillna("").astype(str)
    a = a.sort_values("seendate", kind="stable").reset_index(drop=True)
    return Data(m, a)


# =============================================================================
# 2. Outils numériques
# =============================================================================
def _lam(halflife: float) -> float:
    return 0.5 ** (1.0 / max(float(halflife), 1e-9))


def ewsum(x: np.ndarray, halflife: float) -> np.ndarray:
    """Somme à décroissance exponentielle : y_t = lam * y_{t-1} + x_t."""
    return lfilter([1.0], [1.0, -_lam(halflife)], np.asarray(x, dtype=float))


def ewmean(x: pd.Series, halflife: float) -> pd.Series:
    """Moyenne exponentielle corrigée du biais d'initialisation, robuste aux NaN (ignorés)."""
    v = x.to_numpy(dtype=float)
    ok = ~np.isnan(v)
    num = ewsum(np.where(ok, v, 0.0), halflife)
    den = ewsum(ok.astype(float), halflife)
    return pd.Series(np.where(den > 1e-12, num / np.where(den > 1e-12, den, 1.0), np.nan), index=x.index)


def normalize(x: pd.Series, how: str = "rolling", window: int = 250, min_periods: int = 60) -> pd.Series:
    """Normalisation causale. 'rms' met à l'échelle sans retirer la moyenne (garde le signe d'une tendance)."""
    if how == "none":
        return x
    mp = min(min_periods, window) if window else min_periods
    if how == "rms":
        sd = np.sqrt((x ** 2).rolling(window, min_periods=mp).mean()) if window else np.sqrt((x ** 2).expanding(mp).mean())
        return x / sd
    if how == "expanding" or not window:
        mu, sd = x.expanding(mp).mean(), x.expanding(mp).std()
    elif how == "rolling":
        r = x.rolling(window, min_periods=mp)
        mu, sd = r.mean(), r.std()
    else:
        raise ValueError(f"normalisation inconnue : {how}")
    return (x - mu) / sd.replace(0, np.nan)


# =============================================================================
# 3. Signal NEWS
# =============================================================================
def prepare_articles(data: Data, dedup_days: float = 3, min_conf: float = 0.0) -> pd.DataFrame:
    """Filtre de confiance + dédoublonnage (même URL, ou même titre déjà vu dans les `dedup_days` jours)."""
    key = ("articles", dedup_days, min_conf)
    if key in data._cache:
        return data._cache[key]
    a = data.articles
    if min_conf > 0:
        a = a[(1 - a["p_neu"]) >= min_conf]
    if dedup_days and dedup_days > 0:
        a = a.drop_duplicates("url", keep="first")
        norm = a["title"].str.lower().str.replace(r"[^0-9a-z]+", " ", regex=True).str.strip()
        prev = a["seendate"].groupby(norm).shift(1)
        dup = ((a["seendate"] - prev) <= pd.Timedelta(days=dedup_days)) & (norm != "")
        a = a[~dup.to_numpy()]
    data._cache[key] = a
    return a


def article_scores(a: pd.DataFrame, kind: str = "prob") -> pd.Series:
    if kind == "prob":
        return a["p_pos"] - a["p_neg"]
    cls = a[["p_pos", "p_neg", "p_neu"]].to_numpy().argmax(1)  # 0=pos 1=neg 2=neu
    if kind == "vote":
        v = np.select([cls == 0, cls == 1], [1.0, -1.0], 0.0)
    elif kind == "neg":
        v = -(cls == 1).astype(float)
    elif kind == "pos":
        v = (cls == 0).astype(float)
    else:
        raise ValueError(f"news_score inconnu : {kind}")
    return pd.Series(v, index=a.index)


def news_daily(data: Data, p: dict) -> pd.DataFrame:
    """Agrégation par séance : S = somme pondérée des scores, N = somme des poids, C = nb d'articles."""
    key = ("daily", p["news_dedup_days"], p["news_min_conf"], p["news_score"], p["news_w_boj"],
           p["news_w_japan"], p["news_conf_power"])
    if key in data._cache:
        return data._cache[key]
    a = prepare_articles(data, p["news_dedup_days"], p["news_min_conf"])
    s = article_scores(a, p["news_score"]).to_numpy()
    w = np.where(a["theme"].to_numpy() == "boj", p["news_w_boj"], p["news_w_japan"]).astype(float)
    if p["news_conf_power"]:
        w = w * (1 - a["p_neu"].to_numpy()) ** p["news_conf_power"]
    df = pd.DataFrame({"S": w * s, "N": w, "C": (w > 0).astype(float)}).groupby(a["session"].to_numpy()).sum()
    df = df.reindex(data.calendar).fillna(0.0)
    data._cache[key] = df
    return df


def _sentiment_level(S: np.ndarray, N: np.ndarray, halflife: float, prior: float) -> np.ndarray:
    """Sentiment = moyenne exponentielle des scores pondérée par le nb d'articles,
    rétrécie vers la moyenne long terme quand il y a peu d'articles (shrinkage bayésien)."""
    num, den = ewsum(S, halflife), ewsum(N, halflife)
    cumS, cumN = np.cumsum(S), np.cumsum(N)
    m = np.divide(cumS, cumN, out=np.zeros_like(cumS), where=cumN > 0)  # moyenne long terme causale
    tot = den + prior
    return np.where(tot > 1e-12, (num + prior * m) / np.where(tot > 1e-12, tot, 1.0), np.nan)


def collection_quality(data: Data, p: dict) -> pd.DataFrame:
    """Contrôle qualité de la collecte d'articles, séance par séance (calendrier des news).

    * **trou** : au moins `news_gap_min_run` séances d'affilée sans aucun article, ou séance vide que le dataset
      signale aussi comme panne GDELT (`gdelt_gap`). C'est une donnée manquante, pas un « calme » médiatique.
    * **pic** : nombre d'articles > `news_burst_mult` × médiane des 60 séances observées précédentes.
      - **confirmé** si l'attention GDELT du jour (max Japon/BoJ, mesurée par GDELT sur tout son corpus, donc
        indépendante de la collecte) dépasse `news_burst_confirm_z` : gardé, mais écrêté au seuil (influence bornée) ;
      - **non confirmé** : traité comme donnée manquante (si `news_burst_confirm`).

    Colonnes utiles : `missing` (séance ignorée), `count_eff` (compte utilisé pour l'attention, NaN si manquant),
    `weight` (facteur appliqué aux articles du jour pour le sentiment)."""
    key = ("quality", p["news_dedup_days"], p["news_min_conf"], p["news_gap_min_run"], p["news_burst_mult"],
           p["news_burst_confirm"], p["news_burst_confirm_z"])
    if key in data._cache:
        return data._cache[key]
    a = prepare_articles(data, p["news_dedup_days"], p["news_min_conf"])
    cal = data.calendar
    m = data.market
    n = a.groupby("session").size().reindex(cal, fill_value=0).astype(float)
    gflag = m["gdelt_gap"].astype(bool) if "gdelt_gap" in m else pd.Series(False, index=m.index)
    zero = n == 0
    run = zero.groupby((zero != zero.shift()).cumsum()).transform("sum")
    if p["news_gap_min_run"]:  # 0 = aucune détection de trou (comportement brut)
        gap = zero & ((run >= p["news_gap_min_run"]) | gflag.reindex(cal).fillna(False).astype(bool))
    else:
        gap = pd.Series(False, index=cal)
    med = n.where(~gap).shift(1).rolling(60, min_periods=20).median().clip(lower=1.0)
    ratio = n / med
    att_cols = [c for c in ("gdelt_japan_attention", "gdelt_boj_attention") if c in m]
    if att_cols:  # valeurs recopiées pendant une panne GDELT = inutilisables pour confirmer
        gatt = m[att_cols].max(axis=1).where(~gflag).reindex(cal)
    else:
        gatt = pd.Series(np.nan, index=cal)
    mult = float(p["news_burst_mult"] or 0.0)
    burst = (ratio > mult) if mult > 0 else pd.Series(False, index=cal)
    confirmed = burst & (gatt > p["news_burst_confirm_z"])
    suspect = (burst & ~confirmed) if p["news_burst_confirm"] else pd.Series(False, index=cal)
    missing = gap | suspect
    cap = mult * med if mult > 0 else pd.Series(np.inf, index=cal)
    count_eff = np.minimum(n, cap.fillna(np.inf)).where(~missing)
    weight = (count_eff / n.where(n > 0)).where(n > 0, 1.0).fillna(0.0)
    q = pd.DataFrame({"n": n, "median_60": med, "ratio": ratio, "gdelt_attention": gatt, "gap": gap,
                      "burst": burst, "confirmed": confirmed, "suspect": suspect, "missing": missing,
                      "cap": cap, "count_eff": count_eff, "weight": weight})
    data._cache[key] = q
    return q


def _articles_raw(data: Data, p: dict) -> pd.Series:
    d = news_daily(data, p)
    q = collection_quality(data, p)
    hl, slow = p["news_halflife"], p["news_halflife"] * p["news_slow_mult"]
    w = q["weight"].to_numpy()
    S, N = d["S"].to_numpy() * w, d["N"].to_numpy() * w  # séances douteuses retirées, pics écrêtés
    mode = p["news_mode"]
    if mode == "level":
        x = _sentiment_level(S, N, hl, p["news_prior"])
    elif mode == "momentum":
        x = _sentiment_level(S, N, hl, p["news_prior"]) - _sentiment_level(S, N, slow, p["news_prior"])
    elif mode == "attention":  # volume de news anormal ; les séances manquantes sont sautées (la moyenne est tenue)
        C = (d["C"] * q["weight"]).where(~q["missing"])
        x = (np.log1p(ewmean(C, hl)) - np.log1p(ewmean(C, slow))).to_numpy()
    else:
        raise ValueError(f"news_mode inconnu : {mode}")
    return pd.Series(x, index=d.index)


def _gdelt_raw(data: Data, p: dict) -> pd.Series:
    m = data.market
    wj, wb = float(p["news_w_japan"]), float(p["news_w_boj"])
    col = "attention" if p["news_mode"] == "attention" else "tone"
    x = (wj * m[f"gdelt_japan_{col}"] + wb * m[f"gdelt_boj_{col}"]) / max(wj + wb, 1e-12)
    hl = p["news_halflife"]
    fast = ewmean(x, hl)
    if p["news_mode"] in ("momentum", "attention"):
        return fast - ewmean(x, hl * p["news_slow_mult"])
    return fast


def news_signal(data: Data, p: dict, on: str = "market") -> pd.Series:
    """Signal news normalisé (≈ z-score), écrêté et orienté. on='market' (séances de bourse) ou 'calendar'
    (inclut les séances d'articles postérieures à la dernière cotation : utile pour le signal du jour)."""
    key = ("news",) + tuple(p[k] for k in NEWS_KEYS)
    if key not in data._cache:
        src, how, win = p["news_source"], p["news_norm"], p["news_z_window"]
        parts = []
        if src in ("articles", "blend"):
            za = normalize(_articles_raw(data, p), how, win)
            parts.append((1 - p["news_gdelt_weight"] if src == "blend" else 1.0, za))
        if src in ("gdelt", "blend"):
            zg = normalize(_gdelt_raw(data, p), how, win).reindex(data.calendar).ffill()
            parts.append((p["news_gdelt_weight"] if src == "blend" else 1.0, zg))
        if not parts:
            raise ValueError(f"news_source inconnu : {src}")
        z = sum(w * x for w, x in parts)
        z = z.clip(-p["clip_z"], p["clip_z"]) * p["news_direction"]
        data._cache[key] = z.rename("news")
    z = data._cache[key]
    return z.reindex(data.market.index) if on == "market" else z


# =============================================================================
# 4. Signal PRIX
# =============================================================================
def price_raw(close: pd.Series, p: dict) -> pd.Series:
    lc = np.log(close)
    vol = lc.diff().ewm(halflife=p["price_vol_halflife"], min_periods=20).std()
    kind, L, S = p["price_kind"], int(p["price_lookback"]), int(p["price_skip"])
    if kind == "tsmom":      # rendement sur L séances / (vol * sqrt(L)) : « t-stat » de la tendance
        return (lc.shift(S) - lc.shift(L)) / (vol * np.sqrt(max(L - S, 1)))
    if kind == "composite":  # moyenne de tsmom multi-horizons (plus robuste)
        return pd.concat([(lc - lc.shift(int(l))) / (vol * np.sqrt(l)) for l in p["price_lookbacks"]], axis=1).mean(1, skipna=False)
    if kind == "ma":         # croisement de moyennes mobiles exponentielles (en log-prix)
        f, s = int(p["price_fast"]), int(p["price_slow"])
        return (lc.ewm(span=f, adjust=False).mean() - lc.ewm(span=s, adjust=False).mean()) / (vol * np.sqrt(s))
    if kind == "sma_dist":   # distance à la moyenne mobile simple L
        return (lc - lc.rolling(L).mean()) / (vol * np.sqrt(L))
    if kind == "breakout":   # position dans le canal de Donchian L, dans [-1, 1]
        hi, lo = close.rolling(L).max(), close.rolling(L).min()
        return 2 * (close - lo) / (hi - lo) - 1
    if kind == "drawdown":   # distance au plus haut sur L séances (≤ 0)
        return (lc - lc.rolling(L, min_periods=20).max()) / (vol * np.sqrt(L))
    raise ValueError(f"price_kind inconnu : {kind}")


HMM_FIT_KEYS = ["price_hmm_features", "price_hmm_states", "price_hmm_sticky", "price_hmm_vol_halflife",
                "price_hmm_min_train", "price_hmm_refit", "price_hmm_window"]


def hmm_regimes(data: Data, p: dict) -> dict:
    """Estimation glissante du HMM (mise en cache) : {'probs': probabilités filtrées par régime, 'fits': [...]}.
    Régime 0 = le plus calme, dernier régime = le plus agité. Voir nikkei_hmm.hmm_walk_forward."""
    p = make_params(p)
    key = ("hmm",) + tuple(p[k] for k in HMM_FIT_KEYS)
    if key not in data._cache:
        X = nh.hmm_features(data.close, p["price_hmm_features"], p["price_hmm_vol_halflife"])
        data._cache[key] = nh.hmm_walk_forward(X, n_states=p["price_hmm_states"], min_train=p["price_hmm_min_train"],
                                               refit=p["price_hmm_refit"], window=p["price_hmm_window"],
                                               sticky=p["price_hmm_sticky"])
    return data._cache[key]


def hmm_raw(data: Data, p: dict) -> pd.Series:
    """Signal HMM brut. 'regime' : P(calme) - P(agité), entre -1 et +1 ; 'exp_ret' : rendement cumulé attendu
    sur price_hmm_horizon séances selon le modèle, divisé par sa volatilité sur ce même horizon."""
    wf = hmm_regimes(data, p)
    if p["price_hmm_signal"] == "regime":
        pr = wf["probs"]
        return pr.iloc[:, 0] - pr.iloc[:, -1]
    if p["price_hmm_signal"] == "exp_ret":
        return nh.hmm_expected_return(wf, int(p["price_hmm_horizon"]))
    raise ValueError(f"price_hmm_signal inconnu : {p['price_hmm_signal']}")


REGIME_NAMES = {2: ["calme", "agité"], 3: ["calme", "intermédiaire", "agité"]}


def hmm_hindsight(data: Data, p: dict) -> pd.DataFrame:
    """Probabilités LISSÉES d'un HMM estimé sur tout l'historique : elles utilisent le futur (paramètres estimés
    avec toutes les données, et observations postérieures à t). Uniquement pour montrer l'écart avec les
    probabilités filtrées : un backtest qui les utilise est faux."""
    p = make_params(p)
    X = nh.hmm_features(data.close, p["price_hmm_features"], p["price_hmm_vol_halflife"]).dropna()
    vol_col = list(X.columns).index("logvol") if "logvol" in X.columns else None
    m = nh.GaussianHMM(p["price_hmm_states"], sticky=p["price_hmm_sticky"]).fit(X.to_numpy(), vol_col=vol_col)
    return pd.DataFrame(m.smooth(X.to_numpy()), index=X.index, columns=[f"régime {k}" for k in range(m.n_states)])


def hmm_regime_table(data: Data, p: dict, start=None, end=None) -> pd.DataFrame:
    """Description des régimes, d'après les probabilités filtrées (hors-échantillon) : part du temps, volatilité et
    rendement réalisés les jours où le régime est le plus probable, durée moyenne observée, et durée moyenne
    selon le modèle (dernière estimation). Le rendement est contemporain : il décrit, il ne prédit pas."""
    p = make_params(p)
    wf = hmm_regimes(data, p)
    pr = wf["probs"].loc[start:end].dropna()
    K = pr.shape[1]
    lab = pd.Series(pr.to_numpy().argmax(1), index=pr.index)
    r = np.log(data.close).diff().reindex(pr.index)
    runs = lab.groupby((lab != lab.shift()).cumsum()).agg(["first", "size"])
    last = wf["fits"][-1]
    rows = {}
    for k, name in enumerate(REGIME_NAMES.get(K, [f"régime {i}" for i in range(K)])):
        rk = r[lab == k]
        rows[name] = {"part du temps": float((lab == k).mean()),
                      "volatilité annualisée": float(rk.std() * np.sqrt(TD)),
                      "rendement annualisé": float(rk.mean() * TD),
                      "durée moyenne observée (séances)": float(runs.loc[runs["first"] == k, "size"].mean()),
                      "durée moyenne selon le modèle (séances)": float(last["durées"][k])}
    return pd.DataFrame(rows).T


def hmm_fit_history(data: Data, p: dict) -> pd.DataFrame:
    """Paramètres de chaque ré-estimation : volatilité des régimes extrêmes (si le rendement est observé),
    durées moyennes et temps de réalisation implicites. Des valeurs stables = des régimes bien identifiés."""
    p = make_params(p)
    rows = []
    for f in hmm_regimes(data, p)["fits"]:
        m = f["modèle"]
        row = {"estimé le": f["date"].date()}
        if p["price_hmm_features"] in ("ret", "ret_vol"):
            row["vol. calme"] = float(np.sqrt(m.vars_[0, 0] * TD))
            row["vol. agité"] = float(np.sqrt(m.vars_[-1, 0] * TD))
        row.update({"durée calme": float(f["durées"][0]), "durée agité": float(f["durées"][-1]),
                    "T50 implicite": f["T50"], "T90 implicite": f["T90"]})
        rows.append(row)
    return pd.DataFrame(rows).set_index("estimé le")


def _period_label(a, b) -> str:
    return f"{str(a)[:4]}-{str(b)[:4] if b else 'fin'}"


def hmm_forward_by_period(data: Data, wf: dict, periods, horizon: int = 60, lag: int = 1) -> pd.DataFrame:
    """Par période : part du temps où le régime agité est le plus probable (filtré), et rendement moyen des
    `horizon` séances suivantes selon le régime le plus probable."""
    pr = wf["probs"].dropna()
    K = pr.shape[1]
    lab = pd.Series(pr.to_numpy().argmax(1), index=pr.index)
    y = fwd_log_return(data.close, horizon, lag).reindex(pr.index) * 100
    rows = {}
    for a, b in periods:
        lk, yk = lab.loc[a:b], y.loc[a:b]
        rows[_period_label(a, b)] = {"part du temps en régime agité": float((lk == K - 1).mean()),
                                     f"rdt {horizon} j après « calme » (%)": float(yk[lk == 0].mean()),
                                     f"rdt {horizon} j après « agité » (%)": float(yk[lk == K - 1].mean())}
    return pd.DataFrame(rows).T


def stats_by_period(results: dict, periods, metrics=("CAGR", "Sharpe", "Max drawdown")) -> pd.DataFrame:
    """Mesures de plusieurs StrategyResult (plus le buy & hold) sur plusieurs périodes.
    Une ligne par stratégie, une colonne par (période, mesure)."""
    first = next(iter(results.values()))
    rows = {"Buy & Hold": {}}
    rows.update({name: {} for name in results})
    for a, b in periods:
        lab = _period_label(a, b)
        bt = first.bt.loc[a:b]
        bh = perf_stats(bt["bench"], rf=bt["rf"], expo=pd.Series(1.0, index=bt.index))
        rows["Buy & Hold"].update({(lab, m): bh[m] for m in metrics})
        for name, r in results.items():
            st = r.stats(a, b)
            rows[name].update({(lab, m): st[m] for m in metrics})
    return pd.DataFrame(rows).T


def hmm_report(data: Data, p: dict, horizons=(20, 60, 125, 250), lag: int = 1, start=None, end=None) -> dict:
    """Une ligne pour comparer des variantes de HMM : ce que dit le modèle (durées des régimes, temps de
    réalisation implicite) et ce qui s'est passé ensuite (IC de P(agité) avec le rendement futur ; rendement
    moyen à 60 séances selon le régime filtré le plus probable)."""
    p = make_params(p)
    wf = hmm_regimes(data, p)
    last, K = wf["fits"][-1], p["price_hmm_states"]
    pr = wf["probs"]
    out = {"durée calme (modèle)": float(last["durées"][0]), "durée agité (modèle)": float(last["durées"][-1]),
           "T50 implicite": last["T50"], "T90 implicite": last["T90"]}
    ic = ic_table({"x": pr.iloc[:, -1]}, data.close, horizons, lag, start, end).iloc[0]
    out.update({f"IC P(agité) h={h}": float(ic[h]) for h in horizons})
    lab = pr.dropna().idxmax(axis=1)
    y = fwd_log_return(data.close, 60, lag) * 100
    g = pd.concat([lab.rename("r"), y.rename("y")], axis=1).loc[start:end].dropna().groupby("r")["y"].mean()
    out["rdt 60 j après « calme » (%)"] = float(g.get("régime 0", np.nan))
    out["rdt 60 j après « agité » (%)"] = float(g.get(f"régime {K - 1}", np.nan))
    return out


def price_signal(data: Data, p: dict) -> pd.Series:
    key = ("price",) + tuple(p[k] for k in PRICE_KEYS)
    if key not in data._cache:
        how = p["price_norm"]
        if how == "auto":
            if p["price_kind"] == "hmm" and p["price_hmm_signal"] == "regime":
                how = "none"   # P(calme) - P(agité) est déjà borné entre -1 et +1
            else:
                how = "rolling" if p["price_kind"] in ("drawdown", "breakout") else "rms"
        raw = hmm_raw(data, p) if p["price_kind"] == "hmm" else price_raw(data.close, p)
        z = normalize(raw, how, p["price_z_window"])
        z = z.clip(-p["clip_z"], p["clip_z"]) * p["price_direction"]
        data._cache[key] = z.rename("price")
    return data._cache[key]


def risk_overlay(data: Data, p: dict) -> pd.Series:
    """Multiplicateur d'exposition entre 1 - overlay_k (risque maximal) et 1 (aucun risque détecté).
    'hmm' : probabilité filtrée du régime le plus agité ; 'vol' : volatilité réalisée au-dessus de sa médiane
    passée (règle simple, pour vérifier ce que le HMM apporte de plus)."""
    idx = data.market.index
    kind = p["risk_overlay"]
    if kind == "none" or not p["overlay_k"]:
        return pd.Series(1.0, index=idx, name="overlay")
    if kind == "hmm":
        risk = hmm_regimes(data, p)["probs"].iloc[:, -1]
    elif kind == "vol":
        r = np.log(data.close).diff()
        vol = r.ewm(halflife=p["overlay_vol_halflife"], min_periods=20).std()
        risk = (vol > vol.expanding(250).median()).astype(float).where(vol.notna())
    else:
        raise ValueError(f"risk_overlay inconnu : {kind}")
    return (1.0 - p["overlay_k"] * risk.reindex(idx).fillna(0.0)).rename("overlay")


# =============================================================================
# 5. Combinaison, positions, backtest
# =============================================================================
def combine(news: pd.Series, price: pd.Series, p: dict):
    """Retourne (score combiné, multiplicateur d'exposition)."""
    n, pr = news.fillna(0.0), price.fillna(0.0)
    w, c = p["w_news"], p["combo"]
    mult = pd.Series(1.0, index=n.index)
    if c == "linear":
        s = w * n + (1 - w) * pr
    elif c == "agree":  # on ne prend position que si les deux signaux sont du même côté
        s = (w * n + (1 - w) * pr).where(np.sign(n) == np.sign(pr), 0.0)
    elif c == "filter":  # le prix décide, les news coupent le risque quand elles sont très mauvaises
        s = pr
        mult = pd.Series(np.where(n < p["filter_threshold"], p["filter_scale"], 1.0), index=n.index)
    else:
        raise ValueError(f"combo inconnu : {c}")
    return s.rename("score"), mult


def _position_loop(score, mult, vscale, mapping, long_only, entry, exit_, scale, base,
                   max_lev, rebalance, min_trade):
    n = len(score)
    lo, hi = (0.0 if long_only else -max_lev), max_lev
    pos = np.zeros(n)
    cur, state = 0.0, 0
    reb = max(int(rebalance), 1)
    for i in range(n):
        if i % reb == 0:
            s = score[i]
            if np.isnan(s):
                target = cur
            else:
                if mapping == "threshold":
                    if s > entry:
                        state = 1
                    elif not long_only and s < -entry:
                        state = -1
                    elif state == 1 and s < exit_:
                        state = 0
                    elif state == -1 and s > -exit_:
                        state = 0
                    target = hi if state == 1 else (-max_lev if state == -1 else base)
                elif mapping == "linear":
                    target = base + scale * s
                elif mapping == "sigmoid":
                    target = lo + (hi - lo) / (1.0 + math.exp(-scale * (s - entry)))
                else:
                    raise ValueError(f"mapping inconnu : {mapping}")
                target = min(max(target * mult[i] * vscale[i], lo), hi)
            if abs(target - cur) >= min_trade or target == 0.0:
                cur = target
        pos[i] = cur
    return pos


def build_position(score: pd.Series, mult: pd.Series, data: Data, p: dict) -> pd.Series:
    if p["vol_target"] and p["vol_target"] > 0:
        rv = data.ret.ewm(halflife=p["vol_target_halflife"], min_periods=10).std() * np.sqrt(TD)
        vscale = (p["vol_target"] / rv).fillna(1.0).to_numpy()
    else:
        vscale = np.ones(len(score))
    pos = _position_loop(score.to_numpy(dtype=float), mult.to_numpy(dtype=float), vscale, p["mapping"],
                         bool(p["long_only"]), float(p["entry"]), float(p["exit"]), float(p["scale"]),
                         float(p["base"]), float(p["max_leverage"]), p["rebalance"], float(p["min_trade"]))
    return pd.Series(pos, index=score.index, name="pos")


def backtest(data: Data, pos: pd.Series, p: dict) -> pd.DataFrame:
    """Rendement = cash + exposition × (rendement indice + dividendes - cash) - coûts (formulation « futures »)."""
    r = data.ret
    rf = data.rf if p["use_cash_rate"] else pd.Series(0.0, index=r.index)
    d = p["div_yield"] / TD
    held = pos.shift(p["exec_lag"]).fillna(0.0)   # position en portefeuille après la clôture de t
    expo = held.shift(1).fillna(0.0)               # exposition qui encaisse le rendement du jour t
    turnover = held.diff().abs().fillna(held.abs())
    cost = turnover * p["cost_bps"] / 1e4
    gross = rf + expo * (r + d - rf)
    return pd.DataFrame({"pos": pos, "held": held, "expo": expo, "ret": r, "rf": rf, "turnover": turnover,
                         "cost": cost, "gross": gross, "net": gross - cost, "bench": r + d})


@dataclass
class StrategyResult:
    params: dict
    news: pd.Series
    price: pd.Series
    score: pd.Series
    pos: pd.Series
    bt: pd.DataFrame

    def stats(self, start=None, end=None) -> dict:
        b = self.bt.loc[start:end]
        return perf_stats(b["net"], rf=b["rf"], expo=b["expo"], turnover=b["turnover"], bench=b["bench"])

    def compare(self, start=None, end=None) -> pd.DataFrame:
        """Tableau stratégie vs buy & hold vs « B&H à exposition constante égale » (même exposition moyenne
        que la stratégie, sans timing) : c'est ce dernier qui mesure vraiment la valeur du timing."""
        b = self.bt.loc[start:end]
        s = self.stats(start, end)
        bh = perf_stats(b["bench"], rf=b["rf"], expo=pd.Series(1.0, index=b.index))
        e = float(b["expo"].mean())
        eq_ret = b["rf"] + e * (b["bench"] - b["rf"])
        be = perf_stats(eq_ret, rf=b["rf"], expo=pd.Series(e, index=b.index))
        return pd.DataFrame({"Stratégie": s, "Buy & Hold": bh, f"B&H expo constante {e:.0%}": be})

    def yearly(self, start=None, end=None) -> pd.DataFrame:
        b = self.bt.loc[start:end]
        g = b.groupby(b.index.year)
        out = pd.DataFrame({"Stratégie": g["net"].apply(lambda x: (1 + x).prod() - 1),
                            "Buy & Hold": g["bench"].apply(lambda x: (1 + x).prod() - 1),
                            "Exposition moy.": g["expo"].mean()})
        out["Écart"] = out["Stratégie"] - out["Buy & Hold"]
        return out


def run_strategy(data: Data, params: dict | None = None, **overrides) -> StrategyResult:
    p = make_params(params, **overrides)
    news = news_signal(data, p)
    price = price_signal(data, p)
    score, mult = combine(news, price, p)
    pos = build_position(score, mult * risk_overlay(data, p), data, p)
    return StrategyResult(p, news, price, score, pos, backtest(data, pos, p))


def run_from_score(data: Data, score: pd.Series, params: dict | None = None, **overrides) -> StrategyResult:
    """Backtest d'un score quelconque (ex. sortie du méta-modèle ML) avec les mêmes règles de position/exécution."""
    p = make_params(params, **overrides)
    sc = score.reindex(data.market.index).rename("score")
    pos = build_position(sc, risk_overlay(data, p), data, p)
    nan = pd.Series(np.nan, index=sc.index)
    return StrategyResult(p, nan, nan, sc, pos, backtest(data, pos, p))


def run_price_only(data: Data, params: dict | None = None, **overrides) -> StrategyResult:
    """Stratégie sur le seul signal prix (sans news), y compris sur un historique de cours seul
    (load_price_history). Le filtre de risque `risk_overlay` s'applique comme dans run_strategy."""
    p = make_params(params, **overrides)
    price = price_signal(data, p).reindex(data.market.index)
    score = price.fillna(0.0).rename("score")
    pos = build_position(score, risk_overlay(data, p), data, p)
    nan = pd.Series(np.nan, index=data.market.index)
    return StrategyResult(p, nan, price, score, pos, backtest(data, pos, p))


# =============================================================================
# 6. Mesures de performance
# =============================================================================
def max_drawdown(ret: pd.Series) -> float:
    eq = (1 + ret).cumprod()
    return float((eq / eq.cummax() - 1).min())


def perf_stats(ret: pd.Series, rf=None, expo=None, turnover=None, bench=None) -> dict:
    ret = ret.dropna()
    n = len(ret)
    if n < 2:
        return {}
    yrs = n / TD
    rf = (rf.reindex(ret.index).fillna(0.0) if isinstance(rf, pd.Series) else pd.Series(0.0, index=ret.index))
    ex = ret - rf
    eq = (1 + ret).cumprod()
    cagr = eq.iloc[-1] ** (1 / yrs) - 1
    vol = ret.std() * np.sqrt(TD)
    sharpe = ex.mean() / ex.std() * np.sqrt(TD) if ex.std() > 0 else np.nan
    dn = np.sqrt((np.minimum(ex, 0) ** 2).mean())
    mdd = max_drawdown(ret)
    out = {"CAGR": cagr, "Volatilité": vol, "Sharpe": sharpe,
           "Sortino": ex.mean() / dn * np.sqrt(TD) if dn > 0 else np.nan,
           "Max drawdown": mdd, "Calmar": cagr / abs(mdd) if mdd < 0 else np.nan,
           "% jours positifs": float((ret > 0).mean()), "Années": yrs}
    if expo is not None:
        e = expo.reindex(ret.index)
        out["Exposition moy."] = float(e.abs().mean())
        out["% temps investi"] = float((e.abs() > 1e-9).mean())
    if turnover is not None:
        t = turnover.reindex(ret.index).fillna(0.0)
        n_tr = int((t > 1e-9).sum())
        out["Rotation / an"] = float(t.sum() / yrs)
        out["Nb transactions"] = n_tr
        out["Durée moy. position (j)"] = n / max(n_tr, 1)
    if bench is not None:
        b = bench.reindex(ret.index).fillna(0.0)
        beta = np.cov(ret, b)[0, 1] / b.var()
        out["Bêta"] = beta
        out["Alpha annuel"] = (ret.mean() - beta * b.mean()) * TD
        te = (ret - b).std() * np.sqrt(TD)
        out["Info ratio vs B&H"] = (ret - b).mean() * TD / te if te > 0 else np.nan
        if expo is not None:  # valeur du timing : comparaison à une exposition constante égale
            e = float(expo.reindex(ret.index).mean())
            cagr_eq = (1 + rf + e * (b - rf)).prod() ** (1 / yrs) - 1
            out["CAGR B&H même expo"] = cagr_eq
            out["Gain de timing (CAGR)"] = cagr - cagr_eq
    return out


def drawdown_series(ret: pd.Series) -> pd.Series:
    eq = (1 + ret.fillna(0)).cumprod()
    return eq / eq.cummax() - 1


# =============================================================================
# 7. Analyse des horizons : « en combien de temps le signal se réalise-t-il ? »
# =============================================================================
def fwd_log_return(close: pd.Series, h: int, lag: int = 1) -> pd.Series:
    """log(C[t+lag+h] / C[t+lag]) : rendement réalisé entre l'exécution et la sortie h séances plus tard."""
    lc = np.log(close)
    return lc.shift(-(lag + h)) - lc.shift(-lag)


def _ols_hac(y: np.ndarray, x: np.ndarray, lags: int):
    """Régression y = a + b x, erreurs standard de Newey-West (gère le chevauchement des rendements)."""
    X = np.column_stack([np.ones_like(x), x])
    XtX_inv = np.linalg.inv(X.T @ X)
    beta = XtX_inv @ (X.T @ y)
    u = y - X @ beta
    Xu = X * u[:, None]
    S = Xu.T @ Xu
    for L in range(1, min(lags, len(y) - 1) + 1):
        G = Xu[L:].T @ Xu[:-L]
        S += (1 - L / (lags + 1)) * (G + G.T)
    V = XtX_inv @ S @ XtX_inv
    return beta[1], math.sqrt(max(V[1, 1], 0.0))


def horizon_profile(signal: pd.Series, close: pd.Series, horizons, lag: int = 1, start=None, end=None) -> pd.DataFrame:
    """Pour chaque horizon h : corrélation de rang (IC) et « projection locale » :
    beta(h) = rendement cumulé moyen (en %) sur h séances pour un signal à +1 écart-type,
    avec IC 95 % Newey-West (lags = h + lag) pour tenir compte des rendements qui se chevauchent."""
    rows = []
    for h in horizons:
        y = fwd_log_return(close, h, lag) * 100
        df = pd.concat([signal.rename("x"), y.rename("y")], axis=1).loc[start:end].dropna()
        if len(df) < 30:
            continue
        x = df["x"].to_numpy()
        xs = (x - x.mean()) / x.std()
        b, se = _ols_hac(df["y"].to_numpy(), xs, lags=h + lag)
        ic = stats.spearmanr(x, df["y"]).statistic
        rows.append({"h": h, "n": len(df), "n_indep": len(df) / h, "IC": ic, "beta_%": b, "se": se,
                     "t_hac": b / se if se > 0 else np.nan, "lo95": b - 1.96 * se, "hi95": b + 1.96 * se,
                     "beta_%_par_an": b * TD / h})
    return pd.DataFrame(rows).set_index("h")


def placebo_ic(signal: pd.Series, close: pd.Series, horizons, lag: int = 1, start=None, end=None,
               min_shift: int = 250, quantiles=(0.025, 0.5, 0.975)) -> pd.DataFrame:
    """Bande « hasard » : IC obtenus en décalant circulairement le signal (garde son autocorrélation).
    Si l'IC réel sort de la bande, il est peu probable qu'il soit dû au hasard."""
    out = {}
    for h in horizons:
        y = fwd_log_return(close, h, lag)
        df = pd.concat([signal.rename("x"), y.rename("y")], axis=1).loc[start:end].dropna()
        n = len(df)
        if n < 2 * min_shift + 10:
            continue
        rx = df["x"].rank().to_numpy(); ry = df["y"].rank().to_numpy()
        rx = (rx - rx.mean()) / rx.std(); ry = (ry - ry.mean()) / ry.std()
        cc = np.real(np.fft.ifft(np.fft.fft(rx) * np.conj(np.fft.fft(ry)))) / n
        null = cc[min_shift:n - min_shift]
        out[h] = np.quantile(null, quantiles)
    return pd.DataFrame(out, index=[f"q{q:g}" for q in quantiles]).T.rename_axis("h")


def realization_summary(prof: pd.DataFrame, t_min: float = 1.96) -> dict:
    """Résume la courbe beta(h) :
    * h_meilleur_t : horizon où l'évidence statistique est maximale (meilleur rapport signal/bruit) ;
    * h_signif_max : horizon le plus long où l'effet reste significatif (|t| >= t_min) ;
    * T50 / T90   : horizon où 50 % / 90 % de l'effet (mesuré jusqu'à h_signif_max) est réalisé.
    Au-delà de h_signif_max, les estimations sont trop bruitées pour conclure."""
    if prof.empty:
        return {}
    sgn = float(np.sign(prof.loc[prof["t_hac"].abs().idxmax(), "beta_%"])) or 1.0  # sens dominant du signal
    b, t = prof["beta_%"] * sgn, prof["t_hac"] * sgn
    # bloc contigu d'horizons significatifs autour de l'horizon du meilleur t
    hs = list(t.index)
    i = hs.index(t.idxmax())
    j = i
    while j + 1 < len(hs) and t.iloc[j + 1] >= t_min:
        j += 1
    h_sig = hs[j] if t.iloc[i] >= t_min else None
    out = {"sens": "+ (prédit dans le sens configuré)" if sgn > 0 else "− (prédit à l'envers : inverser le sens)",
           "h_meilleur_t": int(t.idxmax()), "t_max": float(t.max()),
           "effet_%_à_h_meilleur_t": float(b[t.idxmax()]),
           "h_signif_max": int(h_sig) if h_sig is not None else None, "T50": None, "T90": None}
    if h_sig is not None:
        bb = b.loc[:h_sig]
        bmax = bb.max()
        if bmax > 0:
            out["effet_max_%_signif"] = float(bmax)
            out["T50"] = int(bb.index[(bb >= 0.5 * bmax).to_numpy().argmax()])
            out["T90"] = int(bb.index[(bb >= 0.9 * bmax).to_numpy().argmax()])
    return out


def ic_table(signals: dict, close: pd.Series, horizons, lag: int = 1, start=None, end=None) -> pd.DataFrame:
    """IC de rang (Spearman) de plusieurs variantes de signal (lignes) pour plusieurs horizons (colonnes)."""
    fwd = {h: fwd_log_return(close, h, lag) for h in horizons}
    out = {}
    for name, s in signals.items():
        row = {}
        for h in horizons:
            df = pd.concat([s.rename("x"), fwd[h].rename("y")], axis=1).loc[start:end].dropna()
            row[h] = stats.spearmanr(df["x"], df["y"]).statistic if len(df) > 30 else np.nan
        out[name] = row
    return pd.DataFrame(out).T.rename_axis(columns="horizon h")


def subperiod_ic(signal: pd.Series, close: pd.Series, horizons, periods, lag: int = 1) -> pd.DataFrame:
    """Stabilité : IC par sous-période. Un signal fiable garde le même signe dans toutes les périodes."""
    out = {}
    for a, b in periods:
        out[f"{str(a)[:7]} à {str(b)[:7] if b else 'fin'}"] = ic_table({"x": signal}, close, horizons, lag, a, b).iloc[0]
    return pd.DataFrame(out).T


def horizon_table(signals: dict, close: pd.Series, horizons, lag: int = 1, start=None, end=None) -> pd.DataFrame:
    """Résumé « temps de réalisation » pour plusieurs signaux."""
    rows = {}
    for name, s in signals.items():
        rows[name] = realization_summary(horizon_profile(s, close, horizons, lag, start, end))
    return pd.DataFrame(rows).T


def decay_matrix(signal: pd.Series, close: pd.Series, delays, horizons, start=None, end=None) -> pd.DataFrame:
    """IC (rang) entre le signal en t et le rendement de [t+D, t+D+h] : montre si l'information
    met du temps à être intégrée (IC qui apparaît seulement pour D > 0) ou si elle s'use vite."""
    m = pd.DataFrame(index=pd.Index(delays, name="délai D"), columns=pd.Index(horizons, name="horizon h"), dtype=float)
    for D in delays:
        for h in horizons:
            y = fwd_log_return(close, h, D)
            df = pd.concat([signal.rename("x"), y.rename("y")], axis=1).loc[start:end].dropna()
            if len(df) > 30:
                m.loc[D, h] = stats.spearmanr(df["x"], df["y"]).statistic
    return m


def event_study(signal: pd.Series, close: pd.Series, threshold: float = 1.0, side: str = "above",
                horizon: int = 250, lag: int = 1, min_gap: int = 20, start=None, end=None,
                n_boot: int = 1000, seed: int = 0) -> dict:
    """Étude d'événements : chaque fois que le signal FRANCHIT le seuil (avec un délai minimal entre deux
    événements), on suit le rendement cumulé « anormal » (moins la dérive moyenne du marché) sur `horizon` séances."""
    s = signal.loc[start:end]
    cond = (s > threshold) if side == "above" else (s < threshold)
    onset = cond & ~cond.shift(1, fill_value=False)
    lc = np.log(close).to_numpy()
    idx = close.index
    pos_all = idx.get_indexer(onset[onset].index)
    events, last = [], -10 ** 9
    for i in pos_all:
        if i - last >= min_gap:
            events.append(i)
            last = i
    K = np.arange(horizon + 1)
    paths = np.full((len(events), horizon + 1), np.nan)
    for j, i in enumerate(events):
        a = i + lag
        if a >= len(lc):
            continue
        seg = lc[a:a + horizon + 1] - lc[a]
        paths[j, :len(seg)] = seg
    rr = np.diff(np.log(close.loc[start:end].to_numpy()))
    drift = np.nanmean(rr)
    abn = (paths - K * drift) * 100
    rng = np.random.default_rng(seed)
    if len(events) >= 3:
        boot = np.array([np.nanmean(abn[rng.integers(0, len(events), len(events))], axis=0) for _ in range(n_boot)])
        lo, hi = np.nanpercentile(boot, [2.5, 97.5], axis=0)
    else:
        lo = hi = np.full(horizon + 1, np.nan)
    with np.errstate(invalid="ignore"):
        above = np.where(np.isnan(abn), np.nan, (abn > 0).astype(float))
        hit = np.nanmean(above, axis=0) if len(events) else np.full(horizon + 1, np.nan)
    hit[0] = np.nan  # à k=0 tous les rendements valent 0 par construction
    return {"k": K, "mean_abn_%": np.nanmean(abn, axis=0) if len(events) else np.full(horizon + 1, np.nan),
            "lo": lo, "hi": hi, "hit_rate": hit, "n_events": len(events),
            "dates": idx[events], "paths_abn_%": abn, "drift_daily": drift,
            "raw_mean_%": np.nanmean(paths, axis=0) * 100 if len(events) else None}


def holding_curve(signal: pd.Series, data: Data, holds, lag: int = 1, long_only: bool = False,
                  cost_bps: float = 0.0, start=None, end=None) -> pd.DataFrame:
    """Version « portefeuille » : chaque jour on investit 1/H du capital selon le signal du jour et on le garde
    H séances (tranches qui se chevauchent, à la Jegadeesh-Titman). Le Sharpe en fonction de H indique
    l'horizon de détention où le signal paie le mieux."""
    f = signal.clip(-1, 1).fillna(0.0)
    if long_only:
        f = (f > 0).astype(float)
    p = make_params(exec_lag=lag, cost_bps=cost_bps)
    rows = []
    for H in holds:
        pos = f.rolling(int(H), min_periods=1).mean()
        bt = backtest(data, pos, p).loc[start:end]
        st = perf_stats(bt["net"], rf=bt["rf"], expo=bt["expo"], turnover=bt["turnover"], bench=bt["bench"])
        timing = (bt["expo"] - bt["expo"].mean()) * (bt["ret"] - bt["rf"])  # composante « market timing » pure
        rows.append({"H": H, "Sharpe": st["Sharpe"], "CAGR": st["CAGR"], "Max drawdown": st["Max drawdown"],
                     "Alpha annuel": st["Alpha annuel"],
                     "Sharpe timing pur": timing.mean() / timing.std() * np.sqrt(TD) if timing.std() > 0 else np.nan})
    return pd.DataFrame(rows).set_index("H")


# =============================================================================
# 8. Optimisation
# =============================================================================
def expand_grid(grid: dict) -> pd.DataFrame:
    keys = list(grid)
    return pd.DataFrame(list(itertools.product(*[grid[k] for k in keys])), columns=keys)


def run_grid(data: Data, base: dict, grid: dict, start=None, end=None, verbose: bool = True):
    """Évalue toutes les combinaisons de la grille. Retourne (configs, matrice des rendements nets quotidiens).
    Les signaux sont mis en cache : seules les parties qui changent sont recalculées."""
    configs = expand_grid(grid)
    base = make_params(base)
    t0, R = time.time(), {}
    for i, row in enumerate(configs.itertuples(index=False)):
        res = run_strategy(data, base, **row._asdict())
        R[i] = res.bt["net"].loc[start:end]
        if verbose and (i + 1) % max(1, len(configs) // 10) == 0:
            print(f"  {i + 1}/{len(configs)} configs  ({time.time() - t0:.1f}s)")
    rets = pd.DataFrame(R)
    rf = data.rf.reindex(rets.index).fillna(0.0)
    return configs, rets, rf


def score_returns(rets: pd.DataFrame, rf: pd.Series | None = None, objective: str = "sharpe") -> pd.Series:
    """Score vectorisé d'une matrice de rendements (colonnes = configs)."""
    rf = pd.Series(0.0, index=rets.index) if rf is None else rf.reindex(rets.index).fillna(0.0)
    ex = rets.sub(rf, axis=0)
    yrs = len(rets) / TD
    if objective == "sharpe":
        return ex.mean() / ex.std() * np.sqrt(TD)
    if objective == "sortino":
        return ex.mean() / np.sqrt((np.minimum(ex, 0) ** 2).mean()) * np.sqrt(TD)
    eq = (1 + rets).cumprod()
    cagr = eq.iloc[-1] ** (1 / yrs) - 1
    if objective == "cagr":
        return cagr
    mdd = (eq / eq.cummax() - 1).min()
    if objective == "calmar":
        return cagr / mdd.abs()
    if objective == "mar_sharpe":  # compromis : Sharpe pénalisé par le drawdown
        return (ex.mean() / ex.std() * np.sqrt(TD)) * (1 + mdd)
    raise ValueError(f"objectif inconnu : {objective}")


def plateau_scores(scores: pd.Series, grid: dict, size: int = 3) -> pd.Series:
    """Moyenne du score sur le voisinage de chaque config dans la grille (dimensions numériques ordonnées).
    Choisir le meilleur « plateau » plutôt que le meilleur pic réduit fortement le sur-apprentissage."""
    shape = tuple(len(v) for v in grid.values())
    arr = scores.sort_index().to_numpy().reshape(shape)
    sizes = [size if all(isinstance(x, (int, float, np.number)) and not isinstance(x, bool) for x in v) and len(v) > 2 else 1
             for v in grid.values()]
    filled = np.where(np.isnan(arr), np.nanmean(arr), arr)
    sm = uniform_filter(filled, size=sizes, mode="nearest")
    return pd.Series(sm.ravel(), index=scores.sort_index().index)


def is_oos_split(configs: pd.DataFrame, rets: pd.DataFrame, rf: pd.Series, split, objective="sharpe") -> pd.DataFrame:
    split = pd.Timestamp(split)
    a = score_returns(rets.loc[:split - pd.Timedelta(days=1)], rf, objective)
    b = score_returns(rets.loc[split:], rf, objective)
    out = configs.copy()
    out["IS"], out["OOS"] = a, b
    return out


def walk_forward(configs: pd.DataFrame, rets: pd.DataFrame, rf: pd.Series, grid: dict | None = None,
                 first_test="2020-01-01", objective="sharpe", train_years: float | None = None,
                 selection="best", freq="YS") -> dict:
    """Ré-optimisation périodique : à chaque début de période, on choisit la config qui a le mieux marché
    sur le passé (fenêtre croissante ou glissante de `train_years` ans) et on la trade sur la période suivante.
    La courbe obtenue est 100 % hors-échantillon."""
    starts = pd.date_range(first_test, rets.index[-1], freq=freq)
    pieces, chosen = [], []
    for k, ts in enumerate(starts):
        te = starts[k + 1] if k + 1 < len(starts) else rets.index[-1] + pd.Timedelta(days=1)
        tr0 = rets.index[0] if train_years is None else ts - pd.DateOffset(days=int(365.25 * train_years))
        train = rets.loc[tr0:ts - pd.Timedelta(days=1)]
        test = rets.loc[ts:te - pd.Timedelta(days=1)]
        if len(train) < TD or len(test) == 0:
            continue
        sc = score_returns(train, rf, objective)
        if selection == "plateau" and grid is not None:
            sc = plateau_scores(sc, grid)
        best = int(sc.idxmax())
        pieces.append(test[best])
        row = configs.loc[best].to_dict()
        row.update({"début test": ts.date(), "score IS": sc[best],
                    "score OOS": float(score_returns(test[[best]], rf, objective).iloc[0]) if len(test) > 20 else np.nan,
                    "config": best})
        chosen.append(row)
    oos = pd.concat(pieces) if pieces else pd.Series(dtype=float)
    return {"oos_returns": oos, "chosen": pd.DataFrame(chosen)}


def effective_trials(rets: pd.DataFrame, market: pd.Series | None = None, corr_threshold: float = 0.8,
                     max_cols: int = 1500, seed: int = 0) -> int:
    """Nombre « effectif » d'essais indépendants : on retire d'abord le facteur commun (le marché, que toutes les
    configs détiennent en partie), puis on regroupe les configs dont les rendements actifs sont corrélés à plus de
    `corr_threshold` (classification hiérarchique). Nombre de groupes = nombre d'idées réellement différentes testées."""
    from scipy.cluster.hierarchy import fcluster, linkage

    X = rets.T.drop_duplicates().T  # configs identiques (ex. w_news=0 avec des params news différents)
    if X.shape[1] > max_cols:
        X = X[np.random.default_rng(seed).choice(X.columns, max_cols, replace=False)]
    if market is not None:
        m = market.reindex(X.index).fillna(0.0).to_numpy()
        beta = (X.to_numpy() - X.to_numpy().mean(0)).T @ (m - m.mean()) / ((m - m.mean()) ** 2).sum()
        X = X - np.outer(m, beta)
    X = X.loc[:, X.std() > 1e-12]
    if X.shape[1] < 2:
        return X.shape[1]
    Z = linkage(X.T.to_numpy(), method="average", metric="correlation")
    return int(len(set(fcluster(Z, 1 - corr_threshold, criterion="distance"))))


def deflated_sharpe(best_ret: pd.Series, trials_sharpe_ann: pd.Series, n_trials: float, rf=None) -> dict:
    """Deflated Sharpe Ratio (Bailey & López de Prado, 2014) : probabilité que le meilleur Sharpe trouvé soit
    réellement > 0 une fois corrigé du nombre d'essais, de la longueur de l'historique et de la non-normalité."""
    r = best_ret.dropna()
    if rf is not None:
        r = r - rf.reindex(r.index).fillna(0.0)
    T = len(r)
    sr = r.mean() / r.std()  # Sharpe quotidien
    g3, g4 = stats.skew(r), stats.kurtosis(r, fisher=False)
    var_sr = np.nanvar(trials_sharpe_ann / np.sqrt(TD))
    emc = 0.5772156649
    N = max(n_trials, 2)
    sr0 = math.sqrt(var_sr) * ((1 - emc) * stats.norm.ppf(1 - 1 / N) + emc * stats.norm.ppf(1 - 1 / (N * math.e)))
    z = (sr - sr0) * math.sqrt(T - 1) / math.sqrt(max(1 - g3 * sr + (g4 - 1) / 4 * sr ** 2, 1e-12))
    return {"Sharpe annuel": sr * np.sqrt(TD), "Sharpe « chance pure » attendu": sr0 * np.sqrt(TD),
            "Nb essais (effectif)": N, "DSR (proba que le vrai Sharpe > 0)": float(stats.norm.cdf(z))}


# =============================================================================
# 9. Signal du jour
# =============================================================================
def _target_from_score(s: float, p: dict, prev_pos: float) -> float:
    lo, hi = (0.0 if p["long_only"] else -p["max_leverage"]), p["max_leverage"]
    if p["mapping"] == "linear":
        t = p["base"] + p["scale"] * s
    elif p["mapping"] == "sigmoid":
        t = lo + (hi - lo) / (1.0 + math.exp(-p["scale"] * (s - p["entry"])))
    else:  # threshold : on repart de l'état courant (hystérésis)
        state = 1 if prev_pos > p["base"] + 1e-9 else (-1 if prev_pos < -1e-9 else 0)
        if s > p["entry"]:
            state = 1
        elif not p["long_only"] and s < -p["entry"]:
            state = -1
        elif state == 1 and s < p["exit"]:
            state = 0
        elif state == -1 and s > -p["exit"]:
            state = 0
        t = hi if state == 1 else (-p["max_leverage"] if state == -1 else p["base"])
    return float(min(max(t, lo), hi))


def live_snapshot(data: Data, params: dict | None = None) -> pd.Series:
    """Dernières valeurs des signaux et exposition cible. Les news peuvent être plus récentes que le dernier
    cours disponible (articles rattachés à des séances postérieures) : on les utilise."""
    res = run_strategy(data, params)
    p = res.params
    news_cal = news_signal(data, p, on="calendar").dropna()
    price = res.price.dropna()
    n_last, n_date = float(news_cal.iloc[-1]), news_cal.index[-1]
    p_last, p_date = float(price.iloc[-1]), price.index[-1]
    s, mult = combine(pd.Series([n_last]), pd.Series([p_last]), p)
    prev = float(res.pos.iloc[-1])
    overlay = float(risk_overlay(data, p).iloc[-1])
    target = _target_from_score(float(s.iloc[0]), p, prev) * float(mult.iloc[0]) * overlay
    if p["vol_target"] and p["vol_target"] > 0:
        rv = data.ret.ewm(halflife=p["vol_target_halflife"]).std().iloc[-1] * np.sqrt(TD)
        target = min(target * p["vol_target"] / rv, p["max_leverage"])
    q = collection_quality(data, p).iloc[-20:]
    return pd.Series({"Dernière séance de cotation": p_date.date(), "Dernière séance avec news": n_date.date(),
                      "Nikkei (dernière clôture)": float(data.close.iloc[-1]),
                      "Signal news (z)": n_last, "Signal prix (z)": p_last, "Score combiné": float(s.iloc[0]),
                      "Filtre de risque (multiplicateur)": overlay,
                      "Exposition actuelle (backtest)": prev, "Exposition cible": target,
                      "20 dernières séances : trous de collecte": int(q["gap"].sum()),
                      "20 dernières séances : pics confirmés (écrêtés)": int(q["confirmed"].sum()),
                      "20 dernières séances : pics non confirmés (ignorés)": int(q["suspect"].sum())})


# =============================================================================
# 10. Méta-modèle ML walk-forward purgé (bonus)
# =============================================================================
def make_features(data: Data, base: dict | None = None, news_halflives=(5, 20, 60),
                  price_lookbacks=(20, 60, 120, 250)) -> pd.DataFrame:
    """Variables explicatives causales : plusieurs mémoires de news, ton GDELT, momentum multi-horizons, etc."""
    p = make_params(base)
    feats = {}
    for hl in news_halflives:
        feats[f"news_sent_{hl}"] = news_signal(data, {**p, "news_halflife": hl, "news_mode": "level", "news_direction": 1})
    feats["news_mom"] = news_signal(data, {**p, "news_halflife": 10, "news_mode": "momentum", "news_direction": 1})
    feats["news_attention"] = news_signal(data, {**p, "news_halflife": 5, "news_mode": "attention", "news_direction": 1})
    feats["gdelt_tone_20"] = news_signal(data, {**p, "news_source": "gdelt", "news_halflife": 20, "news_mode": "level", "news_direction": 1})
    for L in price_lookbacks:
        feats[f"tsmom_{L}"] = price_signal(data, {**p, "price_kind": "tsmom", "price_lookback": L, "price_skip": 0, "price_direction": 1})
    feats["drawdown_250"] = price_signal(data, {**p, "price_kind": "drawdown", "price_lookback": 250, "price_direction": 1})
    r = data.ret
    feats["vol_ratio"] = np.log(r.rolling(20).std() / r.rolling(250, min_periods=60).std())
    return pd.DataFrame(feats)


def ml_walk_forward(X: pd.DataFrame, close: pd.Series, horizon: int = 60, lag: int = 1, retrain_every: int = 21,
                    min_train: int = 500, model: str = "logit", C: float = 0.1, start=None):
    """Entraîne périodiquement un modèle sur le passé uniquement, avec purge : un exemple daté τ n'est utilisé
    que si son label (rendement τ+lag -> τ+lag+horizon) est entièrement connu au moment de prédire."""
    from sklearn.linear_model import LogisticRegression, Ridge
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    y_ret = fwd_log_return(close, horizon, lag)
    Xc = X.dropna()
    idx = Xc.index
    first = idx[0] if start is None else max(idx[0], pd.Timestamp(start))
    pred = pd.Series(np.nan, index=X.index, name="ml_score")
    coefs = []
    pos_first = idx.get_indexer([idx[idx >= first][0]])[0]
    for k in range(pos_first, len(idx), retrain_every):
        t = idx[k]
        cutoff = k - horizon - lag  # dernier exemple dont le label est connu en t
        if cutoff < min_train:
            continue
        tr = idx[:cutoff + 1]
        ytr = y_ret.loc[tr]
        ok = ytr.notna()
        Xtr, ytr = Xc.loc[tr][ok], ytr[ok]
        if model == "logit":
            mdl = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
            mdl.fit(Xtr, (ytr > 0).astype(int))
        else:
            mdl = make_pipeline(StandardScaler(), Ridge(alpha=1.0 / C))
            mdl.fit(Xtr, ytr)
        block = idx[k:k + retrain_every]
        if model == "logit":
            pr = np.clip(mdl.predict_proba(Xc.loc[block])[:, 1], 1e-6, 1 - 1e-6)
            s = np.log(pr / (1 - pr))  # log-odds : > 0 si le modèle juge la hausse plus probable que la baisse
        else:
            pr = mdl.predict(Xc.loc[block])
            s = pr / ytr.std()  # rendement prédit en écarts-types : > 0 si hausse attendue
        pred.loc[block] = s
        coefs.append(pd.Series(mdl[-1].coef_.ravel(), index=X.columns, name=t))
    return pred, pd.DataFrame(coefs)

In [ ]:
# ─── notebooks/06_annexe_news_prix/nikkei_hmm.py · 336 lignes ───

"""
nikkei_hmm — régimes du Nikkei par modèle de Markov caché (HMM) gaussien, sans biais d'anticipation.

Un HMM suppose que le marché passe d'un régime caché à l'autre (calme, agité…) selon une chaîne de Markov,
et que, dans chaque régime, les observations (rendement quotidien, volatilité réalisée…) suivent une loi
gaussienne qui lui est propre.

* `GaussianHMM`      : estimation par EM (Baum-Welch). Les récurrences avant / arrière sont calculées par
                       produits matriciels préfixes (log2(T) étapes vectorisées au lieu d'une boucle sur T).
* `hmm_walk_forward` : ré-estime le modèle toutes les `refit` séances sur les seules données passées et ne
                       garde que les probabilités FILTRÉES P(régime_t | données jusqu'à t). Les probabilités
                       lissées (forward-backward) et le chemin de Viterbi utilisent le futur : c'est le piège
                       classique des backtests HMM.
* `implied_timing`   : durée moyenne de chaque régime et vitesse à laquelle le modèle « oublie » le régime
                       courant (T50 / T90 implicites), à comparer aux temps de réalisation mesurés.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

LOG2PI = float(np.log(2 * np.pi))


# =============================================================================
# Récurrences avant / arrière vectorisées
# =============================================================================
def _prefix_products(M: np.ndarray):
    """P_t = M_0 @ M_1 @ … @ M_t pour tout t (balayage de Hillis-Steele, log2(T) produits vectorisés).
    Chaque P_t est divisé par son maximum ; la vraie valeur vaut P_t * exp(log_scale_t)."""
    P = np.array(M, dtype=float)
    m = np.maximum(P.max(axis=(1, 2)), 1e-300)
    P /= m[:, None, None]
    ls = np.log(m)
    d, n = 1, len(P)
    while d < n:
        Q = P[:-d] @ P[d:]
        m = np.maximum(Q.reshape(n - d, -1).max(axis=1), 1e-300)
        Q /= m[:, None, None]
        P[d:] = Q
        ls[d:] = ls[:-d] + ls[d:] + np.log(m)
        d *= 2
    return P, ls


def _emission_logpdf(X: np.ndarray, means: np.ndarray, variances: np.ndarray) -> np.ndarray:
    """log N(x_t | mu_k, diag(var_k)) -> (T, K). Une ligne avec une valeur manquante n'apporte aucune information."""
    bad = np.isnan(X).any(axis=1)
    Xf = np.where(np.isnan(X), 0.0, X)
    ll = -0.5 * (LOG2PI * X.shape[1] + np.log(variances).sum(1)[None, :]
                 + (((Xf[:, None, :] - means[None]) ** 2) / variances[None]).sum(2))
    ll[bad] = 0.0
    return ll


def _forward(logb: np.ndarray, startprob: np.ndarray, transmat: np.ndarray):
    """Probabilités filtrées P(s_t | x_0..t) et log-vraisemblance. Retourne aussi b et M pour l'arrière."""
    c = logb.max(axis=1)
    b = np.exp(logb - c[:, None])
    a0 = startprob * b[0]
    s0 = a0.sum()
    a0 = a0 / s0
    if len(b) == 1:
        return a0[None], float(np.log(s0) + c.sum()), b, None
    M = transmat[None, :, :] * b[1:, None, :]          # M_t = A diag(b_t), t = 1..T-1
    P, ls = _prefix_products(M)
    au = np.einsum("k,tkj->tj", a0, P)                  # ∝ alpha_t
    s = au.sum(1)
    alpha = np.vstack([a0, au / s[:, None]])
    loglik = float(np.log(s0) + np.log(s[-1]) + ls[-1] + c.sum())
    return alpha, loglik, b, M


def _posteriors(logb: np.ndarray, startprob: np.ndarray, transmat: np.ndarray):
    """E-step : probabilités lissées gamma_t, transitions attendues sum_t xi_t, log-vraisemblance."""
    alpha, loglik, b, M = _forward(logb, startprob, transmat)
    K = len(startprob)
    if M is None:
        return alpha, np.zeros((K, K)), loglik
    Q, _ = _prefix_products(np.transpose(M[::-1], (0, 2, 1)))   # Q_k = (M_{T-1-k} … M_{T-1})^T
    beta = Q.sum(axis=1)[::-1]                                   # beta_0 … beta_{T-2}
    beta = np.vstack([beta / beta.sum(1, keepdims=True), np.ones((1, K)) / K])
    g = alpha * beta
    g /= g.sum(1, keepdims=True)
    xi = alpha[:-1, :, None] * transmat[None] * (b[1:] * beta[1:])[:, None, :]
    xi /= xi.sum(axis=(1, 2), keepdims=True)
    return g, xi.sum(0), loglik


def stationary(transmat: np.ndarray) -> np.ndarray:
    """Distribution stationnaire pi = pi A."""
    w, v = np.linalg.eig(transmat.T)
    pi = np.real(v[:, np.argmin(np.abs(w - 1))])
    return np.abs(pi) / np.abs(pi).sum()


# =============================================================================
# Modèle
# =============================================================================
class GaussianHMM:
    """HMM gaussien (covariance diagonale) estimé par EM.

    sticky    : pseudo-transitions ajoutées sur la diagonale de la matrice de transition (a priori de Dirichlet).
                Plus il est grand, plus les régimes estimés sont persistants (moins de faux changements).
    var_floor : plancher de variance, en fraction de la variance totale de chaque variable (évite qu'un régime
                se réduise à quelques points).
    n_init    : nombre d'initialisations ; on garde la meilleure vraisemblance. La première regroupe les séances
                par quantile de volatilité locale, les suivantes par blocs aléatoires (graine fixe : reproductible).
    Les régimes sont triés du plus calme (0) au plus agité : par log-volatilité moyenne quand elle fait partie des
    variables observées (`vol_col`), sinon par variance de la 1re variable (le rendement).
    """

    def __init__(self, n_states: int = 2, sticky: float = 0.0, var_floor: float = 1e-3, n_iter: int = 300,
                 tol: float = 1e-6, n_init: int = 3, seed: int = 0):
        self.n_states, self.sticky, self.var_floor = int(n_states), float(sticky), float(var_floor)
        self.n_iter, self.tol, self.n_init, self.seed = int(n_iter), float(tol), int(n_init), int(seed)

    # --- initialisations ---------------------------------------------------
    def _from_labels(self, X, lab, floor):
        K = self.n_states
        mu = np.array([np.nanmean(X[lab == k], 0) if (lab == k).sum() > 2 else np.nanmean(X, 0) for k in range(K)])
        var = np.array([np.nanvar(X[lab == k], 0) if (lab == k).sum() > 2 else np.nanvar(X, 0) for k in range(K)])
        A = np.full((K, K), 0.02 / (K - 1))
        np.fill_diagonal(A, 0.98)
        return np.full(K, 1.0 / K), A, mu, np.maximum(var, floor)

    def _inits(self, X, floor):
        K, T = self.n_states, len(X)
        v = pd.Series(X[:, 0]).rolling(20, min_periods=5).std().bfill().to_numpy()
        lab = np.searchsorted(np.quantile(v, np.linspace(0, 1, K + 1)[1:-1]), v)
        out = [self._from_labels(X, lab, floor)]
        rng = np.random.default_rng(self.seed)
        for _ in range(self.n_init - 1):
            cuts = np.sort(rng.choice(np.arange(1, T), size=max(T // 60, K), replace=False))
            blocks = np.split(np.arange(T), cuts)
            lab = np.empty(T, dtype=int)
            for blk in blocks:
                lab[blk] = rng.integers(0, K)
            out.append(self._from_labels(X, lab, floor))
        return out

    # --- EM -------------------------------------------------------------------
    def _em(self, X, theta, floor):
        pi, A, mu, var = (np.array(a, dtype=float) for a in theta)
        K = self.n_states
        ok = ~np.isnan(X).any(1)
        Xo = X[ok]
        prev = -np.inf
        self.n_iter_ = 0
        for _ in range(self.n_iter):
            self.n_iter_ += 1
            g, xi, ll = _posteriors(_emission_logpdf(X, mu, var), pi, A)
            pi = g[0] + 1e-6
            pi /= pi.sum()
            counts = xi + self.sticky * np.eye(K) + 1e-6
            A = counts / counts.sum(1, keepdims=True)
            go = g[ok]
            w = go.sum(0) + 1e-12
            mu = (go.T @ Xo) / w[:, None]
            var = np.maximum((go.T @ Xo ** 2) / w[:, None] - mu ** 2, floor)
            if ll - prev < self.tol * abs(ll):
                break
            prev = ll
        ll = _forward(_emission_logpdf(X, mu, var), pi, A)[1]
        return (pi, A, mu, var), ll

    def fit(self, X, init=None, vol_col: int | None = None, init_only: bool = False):
        """init      : paramètres de départ (ex. ceux de l'estimation précédente). Ils remplacent les
                       initialisations aléatoires ; l'initialisation par quantiles de volatilité est aussi essayée,
                       sauf si init_only=True (ré-estimation rapide).
        vol_col   : colonne « log volatilité » servant à trier les régimes (None = variance de la colonne 0)."""
        X = np.asarray(X, dtype=float)
        floor = self.var_floor * np.nanvar(X, axis=0)
        if init is not None and init_only:
            cands = [init]
        else:
            cands = self._inits(X, floor)
            if init is not None:
                cands = cands[:1] + [init]
        best = max((self._em(X, th, floor) for th in cands), key=lambda r: r[1])
        (pi, A, mu, var), self.loglik_ = best
        order = np.argsort(var[:, 0] if vol_col is None else mu[:, vol_col])   # 0 = régime le plus calme
        self.startprob_, self.means_, self.vars_ = pi[order], mu[order], var[order]
        self.transmat_ = A[np.ix_(order, order)]
        self.n_obs_ = int((~np.isnan(X).any(1)).sum())
        return self

    @property
    def params(self):
        return self.startprob_, self.transmat_, self.means_, self.vars_

    def n_free_params(self) -> int:
        K, D = self.means_.shape
        return K * (K - 1) + (K - 1) + 2 * K * D

    def bic(self) -> float:
        return -2 * self.loglik_ + self.n_free_params() * np.log(self.n_obs_)

    def filter(self, X) -> np.ndarray:
        """Probabilités filtrées P(s_t | x_0..t), en partant de la distribution stationnaire."""
        logb = _emission_logpdf(np.asarray(X, dtype=float), self.means_, self.vars_)
        return _forward(logb, stationary(self.transmat_), self.transmat_)[0]

    def smooth(self, X) -> np.ndarray:
        """Probabilités LISSÉES P(s_t | toutes les données) : utilisent le futur. Pour la description seulement."""
        logb = _emission_logpdf(np.asarray(X, dtype=float), self.means_, self.vars_)
        return _posteriors(logb, stationary(self.transmat_), self.transmat_)[0]


# =============================================================================
# Lecture « temps de réalisation » d'un modèle estimé
# =============================================================================
def implied_timing(model: GaussianHMM, horizon_max: int = 2000) -> dict:
    """Durée moyenne de chaque régime (1 / (1 - a_kk)) et vitesse d'oubli du régime courant.
    T50 / T90 : séances nécessaires pour que 50 % / 90 % de l'écart de rendement attendu entre « être dans
    le régime calme » et « être dans le régime agité » soit réalisé, selon le modèle."""
    A, mu = model.transmat_, model.means_[:, 0]
    K = len(A)
    dur = 1.0 / np.maximum(1.0 - np.diag(A), 1e-12)
    lam = np.sort(np.abs(np.linalg.eigvals(A)))[::-1]
    lam2 = float(lam[1]) if K > 1 else 0.0
    d = np.zeros(K)
    d[0], d[-1] = 1.0, -1.0                                 # calme - agité
    cum, v = [], d.copy()
    for _ in range(horizon_max):
        v = v @ A
        cum.append(v @ mu)
    cum = np.cumsum(cum)
    tot = cum[-1]
    frac = cum / tot if abs(tot) > 1e-15 else np.full(len(cum), np.nan)

    def first(q):
        hit = np.flatnonzero(frac >= q)
        return int(hit[0] + 1) if len(hit) else None

    return {"durées": dur, "lambda2": lam2,
            "demi-vie": float(np.log(0.5) / np.log(lam2)) if 0 < lam2 < 1 else np.inf,
            "T50": first(0.5), "T90": first(0.9), "écart_total_%": float(tot * 100)}


# =============================================================================
# Variables observées et estimation glissante
# =============================================================================
def hmm_features(close: pd.Series, kind: str = "ret", vol_halflife: float = 10) -> pd.DataFrame:
    """'ret'     : rendement log quotidien (le HMM « textbook ») ;
       'ret_vol' : rendement + log de la volatilité réalisée (EWMA, demi-vie `vol_halflife`), régimes plus stables ;
       'trend_vol' : rendement moyen récent (EWMA) + log volatilité : sépare tendance et turbulence."""
    r = np.log(close.astype(float)).diff()
    if kind == "ret":
        return r.to_frame("ret")
    vol = r.ewm(halflife=vol_halflife, min_periods=20).std()
    if kind == "ret_vol":
        return pd.DataFrame({"ret": r, "logvol": np.log(vol)})
    if kind == "trend_vol":
        return pd.DataFrame({"trend": r.ewm(halflife=vol_halflife, min_periods=20).mean(), "logvol": np.log(vol)})
    raise ValueError(f"variables HMM inconnues : {kind}")


def hmm_walk_forward(X: pd.DataFrame, n_states: int = 2, min_train: int = 500, refit: int = 63,
                     window: int = 0, sticky: float = 0.0, seed: int = 0, full_every: int = 4) -> dict:
    """Estimation glissante, 100 % causale.

    À la clôture de la séance f, on estime le HMM sur les séances [f - window + 1, f] (ou depuis le début si
    window = 0), puis ce modèle filtre les séances f … f + refit - 1. La probabilité datée t n'utilise que les
    observations jusqu'à t et des paramètres estimés avant t.
    Chaque ré-estimation part du modèle précédent ; toutes les `full_every` ré-estimations, on essaie aussi les
    initialisations de départ et on garde la meilleure vraisemblance (évite de rester coincé dans un optimum local).
    Retourne {'probs': DataFrame (T, K), 'fits': liste de dicts (date, modèle, durées…)}."""
    Xv = X.to_numpy(dtype=float)
    vol_col = list(X.columns).index("logvol") if "logvol" in X.columns else None
    valid = np.flatnonzero(~np.isnan(Xv).any(axis=1))
    T = len(X)
    probs = np.full((T, n_states), np.nan)
    fits, prev = [], None
    if len(valid) <= min_train:
        return {"probs": pd.DataFrame(probs, index=X.index), "fits": fits}
    first = valid[0]
    f = valid[min_train - 1]
    i = 0
    while f < T:
        lo = first if not window else max(first, f + 1 - int(window))
        quick = prev is not None and full_every > 0 and i % full_every != 0
        model = GaussianHMM(n_states, sticky=sticky, seed=seed).fit(
            Xv[lo:f + 1], init=prev.params if prev is not None else None, vol_col=vol_col, init_only=quick)
        i += 1
        end = min(f + int(refit), T)
        probs[f:end] = model.filter(Xv[first:end])[f - first:]
        fits.append({"date": X.index[f], "jusqu'à": X.index[end - 1], "modèle": model, **implied_timing(model)})
        prev, f = model, end
    cols = [f"régime {k}" for k in range(n_states)]
    return {"probs": pd.DataFrame(probs, index=X.index, columns=cols), "fits": fits}


def expected_return_parts(wf: dict, horizon: int) -> pd.DataFrame:
    """Prévision du modèle : rendement cumulé attendu sur les `horizon` prochaines séances, divisé par la
    volatilité stationnaire du modèle sur ce même horizon (une sorte de ratio de Sharpe attendu).
    Décomposée en deux parts :
    * 'niveau moyen' : ce qu'attend le modèle sans rien savoir du régime actuel (distribution stationnaire).
      Il ne bouge qu'aux ré-estimations et suit en pratique le rendement moyen passé depuis le début ;
    * 'partie régime' : l'écart dû au régime actuel, la seule information propre au HMM."""
    probs = wf["probs"]
    out = pd.DataFrame(np.nan, index=probs.index, columns=["total", "niveau moyen", "partie régime"])
    for fit in wf["fits"]:
        m = fit["modèle"]
        A, mu = m.transmat_, m.means_[:, 0]
        S, Ak = np.zeros_like(A), np.eye(len(A))
        for _ in range(int(horizon)):
            Ak = Ak @ A
            S += Ak
        pi = stationary(A)
        sd = np.sqrt(pi @ (m.vars_[:, 0] + mu ** 2) - (pi @ mu) ** 2) * np.sqrt(horizon)
        seg = probs.loc[fit["date"]:fit["jusqu'à"]]
        P = seg.to_numpy()
        out.loc[seg.index, "niveau moyen"] = float(pi @ S @ mu / sd)
        out.loc[seg.index, "partie régime"] = (P - pi) @ S @ mu / sd
    out["total"] = out["niveau moyen"] + out["partie régime"]
    return out


def expected_return_fixed(wf: dict, horizon: int, fit: int = -1) -> pd.Series:
    """DIAGNOSTIC : la même prévision, mais avec les paramètres d'une seule estimation (la dernière par défaut)
    appliqués à toutes les dates. Seules les probabilités de régime varient encore. Si ce signal ne prédit rien,
    le pouvoir apparent du rendement attendu vient des ré-estimations successives, pas du régime.
    Avec fit=-1, il utilise des paramètres estimés après coup : ne jamais le backtester."""
    m = wf["fits"][fit]["modèle"]
    A, mu = m.transmat_, m.means_[:, 0]
    S, Ak = np.zeros_like(A), np.eye(len(A))
    for _ in range(int(horizon)):
        Ak = Ak @ A
        S += Ak
    return pd.Series(wf["probs"].to_numpy() @ S @ mu, index=wf["probs"].index)


def hmm_expected_return(wf: dict, horizon: int) -> pd.Series:
    """Rendement attendu par le modèle (voir expected_return_parts)."""
    return expected_return_parts(wf, horizon)["total"]

In [ ]:
# ─── notebooks/06_annexe_news_prix/nikkei_plots.py · 650 lignes ───

"""
nikkei_plots — graphiques et tableau de bord interactif du notebook.

Palette : instance de référence validée (clair ET sombre) — rôles fixes dans tout le notebook :
  stratégie = bleu · news = orange · prix = aqua · ML = jaune · buy & hold = gris.
Jamais de double axe : les séries d'échelles différentes vont dans des panneaux séparés.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.dates  # noqa: F401  (mpl.dates utilisé pour placer les étiquettes de fin)
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm, to_rgb
from matplotlib.ticker import FuncFormatter, PercentFormatter, FixedLocator

import nikkei_lab as nl

LIGHT = dict(surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9", axis="#c3c2b7",
             strategy="#2a78d6", news="#eb6834", price="#1baf7a", ml="#eda100", bench="#898781", bench2="#c3c2b7",
             div=["#a32b2b", "#e34948", "#f4b4b3", "#f0efec", "#b7d3f6", "#3987e5", "#184f95"],
             seq=["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
DARK = dict(surface="#1a1a19", ink="#f0efec", ink2="#c3c2b7", muted="#898781", grid="#2c2c2a", axis="#383835",
            strategy="#3987e5", news="#d95926", price="#199e70", ml="#c98500", bench="#898781", bench2="#5a5955",
            div=["#f2a7a6", "#e66767", "#7a2f2f", "#383835", "#1c4a80", "#3987e5", "#b7d3f6"],
            seq=["#13325a", "#184f95", "#256abf", "#3987e5", "#6da7ec", "#9ec5f4", "#cde2fb"])
T: dict = dict(LIGHT)


def set_theme(dark: bool = False):
    """Applique le thème (clair par défaut ; dark=True pour un notebook en thème sombre)."""
    T.clear()
    T.update(DARK if dark else LIGHT)
    mpl.rcParams.update({
        "figure.facecolor": T["surface"], "axes.facecolor": T["surface"], "savefig.facecolor": T["surface"],
        "axes.edgecolor": T["axis"], "axes.labelcolor": T["ink2"], "axes.titlecolor": T["ink"],
        "axes.grid": True, "grid.color": T["grid"], "grid.linewidth": 0.7, "grid.linestyle": "-",
        "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
        "axes.linewidth": 0.8, "xtick.color": T["axis"], "ytick.color": T["axis"],
        "xtick.labelcolor": T["ink2"], "ytick.labelcolor": T["ink2"], "text.color": T["ink"],
        "legend.frameon": False, "legend.labelcolor": T["ink2"], "legend.fontsize": 9,
        "lines.linewidth": 1.5, "lines.solid_capstyle": "round", "lines.solid_joinstyle": "round",
        "font.family": "sans-serif", "font.sans-serif": ["Helvetica Neue", "Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 9.5, "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "axes.titlepad": 8, "figure.dpi": 110, "figure.constrained_layout.use": True,
        "axes.prop_cycle": mpl.cycler(color=[T["strategy"], T["news"], T["price"], T["ml"]]),
    })


set_theme(False)


# ----------------------------------------------------------------------------- utilitaires
def div_cmap():
    return LinearSegmentedColormap.from_list("div", T["div"])


def seq_cmap():
    return LinearSegmentedColormap.from_list("seq", T["seq"])


def _zero(ax, y=0.0):
    ax.axhline(y, color=T["axis"], lw=0.9, zorder=1)


def _end_label(ax, x, y, text, color):
    ax.scatter([x], [y], s=34, color=color, edgecolor=T["surface"], linewidth=1.5, zorder=6, clip_on=False)
    ax.annotate(text, (x, y), xytext=(7, 0), textcoords="offset points", va="center", fontsize=8.5,
                color=T["ink2"], annotation_clip=False)


def _end_labels(ax, items, min_px=12):
    """items : [(x, y, texte, couleur)] par priorité décroissante. Un point de fin pour chacun ;
    le texte n'est posé que s'il ne chevauche pas une étiquette déjà placée (sinon : la légende suffit)."""
    ax.figure.canvas.draw()  # nécessaire pour convertir en pixels avec l'échelle finale
    placed = []
    for x, y, text, colr in items:
        ax.scatter([x], [y], s=34, color=colr, edgecolor=T["surface"], linewidth=1.5, zorder=6, clip_on=False)
        py = ax.transData.transform((mpl.dates.date2num(x) if isinstance(x, pd.Timestamp) else x, y))[1]
        if all(abs(py - q) >= min_px for q in placed):
            placed.append(py)
            ax.annotate(text, (x, y), xytext=(7, 0), textcoords="offset points", va="center", fontsize=8.5,
                        color=T["ink2"], annotation_clip=False)


def _log_multiple_axis(ax):
    ax.set_yscale("log")
    lo, hi = ax.get_ylim()
    cands = [0.25, 0.33, 0.5, 0.67, 0.75, 1, 1.25, 1.5, 2, 2.5, 3, 4, 5, 6, 8, 10, 15, 20]
    ticks = [t for t in cands if lo <= t <= hi]
    if len(ticks) > 7:
        ticks = ticks[::2] if 1 in ticks[::2] else ticks[1::2]
    ax.yaxis.set_major_locator(FixedLocator(ticks))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"×{x:g}"))
    ax.yaxis.set_minor_formatter(FuncFormatter(lambda *_: ""))
    ax.yaxis.set_minor_locator(FixedLocator([]))


def _k(x, _=None):
    return f"{x / 1000:.0f}K" if abs(x) >= 1000 else f"{x:.0f}"


def _log_price_axis(ax, lo, hi):
    """Axe log avec des graduations « rondes », de quelques milliers à 100 000 points."""
    ax.set_yscale("log")
    ticks = []
    for mults in ([1, 1.5, 2, 2.5, 3, 4, 5, 6, 7, 8], [1, 2, 3, 5, 7], [1, 2, 5], [1]):
        ticks = [m * 10 ** e for e in range(2, 6) for m in mults if lo * 0.95 <= m * 10 ** e <= hi * 1.05]
        if len(ticks) <= 8:
            break
    ax.yaxis.set_major_locator(FixedLocator(ticks))
    ax.yaxis.set_major_formatter(FuncFormatter(_k))
    ax.yaxis.set_minor_formatter(FuncFormatter(lambda *_: ""))


def _horizon_axis(ax, horizons):
    ax.set_xscale("log")
    ticks = [h for h in (1, 2, 5, 10, 20, 40, 60, 125, 250, 500) if min(horizons) <= h <= max(horizons)]
    ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.0f}"))
    ax.xaxis.set_minor_formatter(FuncFormatter(lambda *_: ""))
    ax.set_xlabel("horizon h (séances de bourse)")


def _ink_on(rgb) -> str:
    r, g, b = [c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4 for c in rgb[:3]]
    return "#ffffff" if 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.30 else "#0b0b0b"


# ----------------------------------------------------------------------------- données
def plot_overview(D: nl.Data, start=None):
    a = D.articles
    fig, axes = plt.subplots(3, 1, figsize=(11, 7.5), sharex=True, height_ratios=[1.4, 1, 1])
    c = D.close.loc[start:]
    ax = axes[0]
    ax.plot(c.index, c, color=T["ink2"], lw=1.3)
    _log_price_axis(ax, c.min(), c.max())
    ax.set_title("Nikkei 225 — clôture (échelle log)")
    m = a.groupby([pd.Grouper(key="session", freq="ME"), "theme"]).size().unstack(fill_value=0)
    s = a.groupby([pd.Grouper(key="session", freq="ME"), "theme"])["score"].mean().unstack()
    for col, lab, colr in (("boj", "BoJ", T["strategy"]), ("japan", "Japon", T["news"])):
        axes[1].plot(m.index, m[col], color=colr, lw=1.3, label=lab)
        axes[2].plot(s.index, s[col], color=colr, lw=1.3, label=lab)
    axes[1].set_title("Articles par mois et par thème")
    axes[1].legend(loc="upper left", ncols=2)
    axes[2].set_title("Sentiment moyen mensuel (FinBERT : p_pos − p_neg)")
    axes[2].legend(loc="lower left", ncols=2)
    _zero(axes[2])
    return fig


CRITICAL = "#d03b3b"  # couleur de statut « critique » (identique en clair et en sombre), toujours avec un libellé


def plot_collection(q: pd.DataFrame, start=None, end=None, confirm_z: float = 0.0,
                    title="Collecte des articles : trous et pics"):
    """Contrôle qualité : articles par séance, trous (zones grises), pics confirmés / non confirmés par GDELT."""
    d = q.loc[start:end]
    day = pd.Timedelta(days=1)
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6.2), sharex=True, height_ratios=[1.7, 1])
    normal = ~(d["burst"] | d["gap"])
    a1.bar(d.index[normal], d["n"][normal], width=1.0, color=T["bench2"], label="articles par séance")
    a1.bar(d.index[d["confirmed"]], d["n"][d["confirmed"]], width=1.0, color=T["strategy"],
           label="pic confirmé par GDELT : gardé, écrêté au seuil")
    a1.bar(d.index[d["suspect"]], d["n"][d["suspect"]], width=1.0, color=CRITICAL,
           label="pic non confirmé : ignoré")
    g = d["gap"].astype(int)
    runs = g.groupby((g != g.shift()).cumsum())
    first = True
    for _, r in runs:
        if r.iloc[0] == 1:
            a1.axvspan(r.index[0] - day * 0.6, r.index[-1] + day * 0.6, color=T["grid"], lw=0, zorder=0,
                       label="trou de collecte : ignoré" if first else None)
            first = False
    a1.plot(d.index, d["median_60"], color=T["ink2"], lw=1.1, label="médiane des 60 séances précédentes")
    if np.isfinite(d["cap"]).any():
        a1.plot(d.index, d["cap"], color=T["muted"], lw=0.9, label="seuil de pic")
    a1.set_ylabel("articles")
    a1.set_ylim(0, float(d["n"].max()) * 1.45 + 1)  # place pour la légende au-dessus des barres
    a1.legend(loc="upper left", fontsize=8, ncols=3)
    a1.set_title(title)
    a2.plot(d.index, d["gdelt_attention"], color=T["ink2"], lw=1.1)
    a2.axhline(confirm_z, color=T["muted"], lw=0.9)
    a2.annotate("seuil de confirmation", (1.0, confirm_z), xycoords=("axes fraction", "data"), xytext=(4, 0),
                textcoords="offset points", va="center", fontsize=8, color=T["ink2"], annotation_clip=False)
    for mask, colr, lab in ((d["confirmed"], T["strategy"], "pic confirmé"), (d["suspect"], CRITICAL, "pic non confirmé")):
        a2.scatter(d.index[mask], d["gdelt_attention"][mask].fillna(a2.get_ylim()[0]), s=30, color=colr,
                   edgecolor=T["surface"], linewidth=1.2, zorder=5, label=lab)
    _zero(a2)
    a2.set_ylabel("z")
    a2.legend(loc="upper left", fontsize=8.5, ncols=2)
    a2.set_title("Attention GDELT ce jour-là (max Japon / BoJ, mesurée sur tout le corpus GDELT)")
    return fig


# ----------------------------------------------------------------------------- signaux
def plot_signals(res: nl.StrategyResult, D: nl.Data, start=None, end=None, title=None):
    p = res.params
    fig, axes = plt.subplots(5, 1, figsize=(11, 10), sharex=True, height_ratios=[1.3, 1, 1, 1, 0.8])
    c = D.close.loc[start:end]
    axes[0].plot(c.index, c, color=T["ink2"], lw=1.3)
    _log_price_axis(axes[0], c.min(), c.max())
    axes[0].set_title(title or "Nikkei 225")
    n = res.news.loc[start:end]
    axes[1].plot(n.index, n, color=T["news"], lw=1.1)
    sens = "suivi" if p["news_direction"] > 0 else "contrarien"
    axes[1].set_title(f"Signal news — {p['news_source']} / {p['news_mode']}, demi-vie {p['news_halflife']} j, {sens} (z)")
    pr = res.price.loc[start:end]
    axes[2].plot(pr.index, pr, color=T["price"], lw=1.1)
    if p["price_kind"] == "hmm":
        sens = "suit le régime" if p["price_direction"] > 0 else "contrarien"
        axes[2].set_title(f"Signal prix — HMM {p['price_hmm_features']}, {p['price_hmm_states']} régimes, "
                          f"{p['price_hmm_signal']}, {sens} (z)")
    else:
        sens = "tendance" if p["price_direction"] > 0 else "retour à la moyenne"
        axes[2].set_title(f"Signal prix — {p['price_kind']} {p['price_lookback']} j, {sens} (z)")
    s = res.score.loc[start:end]
    axes[3].plot(s.index, s, color=T["strategy"], lw=1.1)
    axes[3].set_title(f"Score combiné — {p['combo']}, poids news {p['w_news']:.2f}")
    if p["mapping"] == "threshold":
        for lvl, lab in ((p["entry"], "entrée"), (p["exit"], "sortie")):
            axes[3].axhline(lvl, color=T["muted"], lw=0.8)
            axes[3].annotate(lab, (1.0, lvl), xycoords=("axes fraction", "data"), xytext=(4, 0),
                             textcoords="offset points", va="center", fontsize=8, color=T["ink2"], annotation_clip=False)
    for ax in axes[1:4]:
        _zero(ax)
    e = res.bt["expo"].loc[start:end]
    axes[4].fill_between(e.index, 0, e, color=T["strategy"], alpha=0.12, lw=0, step="post")
    axes[4].plot(e.index, e, color=T["strategy"], lw=1.1, drawstyle="steps-post")
    axes[4].yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    axes[4].set_title("Exposition (part du capital investie dans l'indice)")
    return fig


def plot_panels(D: nl.Data, panels: dict, start=None, end=None, title="Nikkei 225"):
    """Prix en haut puis un panneau par signal (petits multiples, axe x partagé, jamais de double axe).
    panels : {titre: (série, couleur)}"""
    n = len(panels)
    fig, axes = plt.subplots(n + 1, 1, figsize=(11, 2.0 * n + 2.6), sharex=True, height_ratios=[1.3] + [1] * n)
    c = D.close.loc[start:end]
    axes[0].plot(c.index, c, color=T["ink2"], lw=1.2)
    _log_price_axis(axes[0], c.min(), c.max())
    axes[0].set_title(title)
    for ax, (name, (s, colr)) in zip(axes[1:], panels.items()):
        s = s.loc[start:end]
        ax.plot(s.index, s, color=colr, lw=1.1)
        _zero(ax)
        ax.set_title(name)
    return fig


def plot_signal_vs_price(signal: pd.Series, D: nl.Data, color, title, start=None, end=None):
    fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True, height_ratios=[1.2, 1])
    c = D.close.loc[start:end]
    axes[0].plot(c.index, c, color=T["ink2"], lw=1.2)
    _log_price_axis(axes[0], c.min(), c.max())
    axes[0].set_title("Nikkei 225")
    s = signal.loc[start:end]
    axes[1].plot(s.index, s, color=color, lw=1.1)
    _zero(axes[1])
    axes[1].set_title(title)
    return fig


def plot_regimes(close: pd.Series, wf: dict, smoothed: pd.DataFrame | None = None, start=None, end=None,
                 title="Régimes HMM estimés sans regarder le futur"):
    """Haut : Nikkei (log) ; zones grises = séances où le régime le plus agité est le plus probable (filtré).
    Bas : probabilité filtrée du régime le plus agité (connue le jour même) et, en pointillé, la probabilité
    lissée d'un modèle estimé sur tout l'historique : elle connaît le futur et ne doit jamais servir à un backtest."""
    pr = wf["probs"].loc[start:end].dropna()
    K = pr.shape[1]
    c = close.loc[pr.index[0]:pr.index[-1]]
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6.4), sharex=True, height_ratios=[1.5, 1])
    a1.plot(c.index, c, color=T["ink2"], lw=1.2)
    _log_price_axis(a1, c.min(), c.max())
    top = pd.Series((pr.to_numpy().argmax(1) == K - 1).astype(int), index=pr.index)
    day, first = pd.Timedelta(days=1), True
    for _, g in top.groupby((top != top.shift()).cumsum()):
        if g.iloc[0] == 1:
            a1.axvspan(g.index[0] - day * 0.5, g.index[-1] + day * 0.5, color=T["grid"], lw=0, zorder=0,
                       label="régime agité le plus probable (filtré)" if first else None)
            first = False
    a1.legend(loc="upper left")
    a1.set_title(title)
    p_top = pr.iloc[:, -1]
    a2.fill_between(p_top.index, 0, p_top, color=T["price"], alpha=0.15, lw=0)
    a2.plot(p_top.index, p_top, color=T["price"], lw=1.1, label="filtrée : connue le jour même (utilisée par la stratégie)")
    if smoothed is not None:
        sm = smoothed.iloc[:, -1].loc[p_top.index[0]:p_top.index[-1]]
        a2.plot(sm.index, sm, color=T["ink2"], lw=0.9, ls=(0, (3, 2)),
                label="lissée sur tout l'historique : connaît le futur")
    a2.set_ylim(0, 1.32)
    a2.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    a2.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    a2.set_title(f"Probabilité du régime le plus agité ({K} régimes)")
    a2.legend(loc="upper left", ncols=2)
    return fig


# ----------------------------------------------------------------------------- heatmaps
def plot_heatmap(df: pd.DataFrame, title="", center=0.0, vmax=None, fmt="{:.2f}", kind="div", cbar_label="",
                 xlabel=None, ylabel=None, annotate=True, figsize=None, ax=None):
    data = df.to_numpy(dtype=float)
    nr, nc = data.shape
    if ax is None:
        figsize = figsize or (max(6.0, 0.62 * nc + 3.2), max(2.4, 0.34 * nr + 1.3))
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure
    if kind == "div":
        vmax = vmax or float(np.nanmax(np.abs(data - center))) or 1.0
        norm, cmap = TwoSlopeNorm(vcenter=center, vmin=center - vmax, vmax=center + vmax), div_cmap()
    else:
        norm, cmap = Normalize(np.nanmin(data), np.nanmax(data)), seq_cmap()
    mesh = ax.pcolormesh(np.ma.masked_invalid(data), cmap=cmap, norm=norm, edgecolors=T["surface"], linewidth=1.5)
    ax.set_xlim(0, nc)
    ax.set_ylim(nr, 0)
    ax.grid(False)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_xticks(np.arange(nc) + 0.5, [str(c) for c in df.columns], fontsize=8.5)
    ax.set_yticks(np.arange(nr) + 0.5, [str(i) for i in df.index], fontsize=8.5)
    ax.tick_params(length=0)
    if annotate and nr * nc <= 400:
        for i in range(nr):
            for j in range(nc):
                v = data[i, j]
                if np.isfinite(v):
                    ax.text(j + 0.5, i + 0.5, fmt.format(v), ha="center", va="center", fontsize=7.5,
                            color=_ink_on(cmap(norm(v))))
    cb = fig.colorbar(mesh, ax=ax, fraction=0.035, pad=0.015)
    cb.outline.set_visible(False)
    cb.ax.tick_params(length=0, labelsize=8)
    if cbar_label:
        cb.set_label(cbar_label, color=T["ink2"], fontsize=8.5)
    ax.set_title(title)
    ax.set_xlabel(xlabel if xlabel is not None else (df.columns.name or ""))
    ax.set_ylabel(ylabel if ylabel is not None else (df.index.name or ""))
    return fig


# ----------------------------------------------------------------------------- horizons
def plot_horizon(prof: pd.DataFrame, placebo: pd.DataFrame | None = None, summary: dict | None = None,
                 color=None, title=""):
    color = color or T["strategy"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
    h = prof.index.to_numpy()
    a1.fill_between(h, prof["lo95"], prof["hi95"], color=color, alpha=0.12, lw=0)
    a1.plot(h, prof["beta_%"], color=color, marker="o", ms=5.5, mec=T["surface"], mew=1.2)
    _zero(a1)
    _horizon_axis(a1, h)
    a1.set_ylabel("rendement cumulé moyen (%)")
    a1.set_title("Effet cumulé pour un signal à +1σ (bande : IC 95 % Newey-West)")
    if summary:
        txt = f"meilleur rapport signal/bruit à {summary['h_meilleur_t']} j (t = {summary['t_max']:.1f})"
        if summary.get("T50"):
            txt += (f"\neffet significatif jusqu'à {summary['h_signif_max']} j"
                    f"\n50 % de l'effet réalisé à {summary['T50']} j · 90 % à {summary['T90']} j")
        else:
            txt += "\naucun horizon significatif à 5 %"
        a1.text(0.02, 0.97, txt, transform=a1.transAxes, fontsize=8.5, color=T["ink2"], va="top",
                bbox=dict(facecolor=T["surface"], edgecolor="none", alpha=0.85, pad=2))
    if placebo is not None and len(placebo):
        a2.fill_between(placebo.index, placebo.iloc[:, 0], placebo.iloc[:, -1], color=T["grid"], lw=0,
                        label="bande du hasard 95 % (placebo)")
    a2.plot(h, prof["IC"], color=color, marker="o", ms=5.5, mec=T["surface"], mew=1.2, label="IC réel")
    _zero(a2)
    _horizon_axis(a2, h)
    a2.set_ylabel("corrélation de rang (IC)")
    a2.set_title("IC vs hasard (signal décalé aléatoirement)")
    a2.legend(loc="best")
    fig.suptitle(title, x=0.01, ha="left", fontsize=11.5, fontweight="bold", color=T["ink"])
    return fig


def plot_horizon_compare(profiles: dict, metric="t_hac", title="Rapport signal/bruit par horizon"):
    """profiles : {nom: (profil, couleur)}"""
    fig, ax = plt.subplots(figsize=(11, 4))
    for name, (prof, colr) in profiles.items():
        ax.plot(prof.index, prof[metric], color=colr, marker="o", ms=5, mec=T["surface"], mew=1.2, label=name)
    if metric == "t_hac":
        for y in (-1.96, 1.96):
            ax.axhline(y, color=T["muted"], lw=0.8)
        ax.annotate("seuil 5 %", (1.0, 1.96), xycoords=("axes fraction", "data"), xytext=(4, 0),
                    textcoords="offset points", va="center", fontsize=8, color=T["ink2"], annotation_clip=False)
        ax.set_ylabel("t-stat Newey-West")
    _zero(ax)
    _horizon_axis(ax, list(next(iter(profiles.values()))[0].index))
    ax.legend(loc="best")
    ax.set_title(title)
    return fig


def plot_event_study(es: dict, color=None, title=""):
    color = color or T["strategy"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2), width_ratios=[2, 1])
    k = es["k"]
    for row in es["paths_abn_%"]:
        a1.plot(k, row, color=T["muted"], lw=0.5, alpha=0.25)
    a1.fill_between(k, es["lo"], es["hi"], color=color, alpha=0.15, lw=0)
    a1.plot(k, es["mean_abn_%"], color=color, lw=2)
    _zero(a1)
    a1.set_xlabel("séances après l'exécution")
    a1.set_ylabel("rendement anormal cumulé (%)")
    a1.set_title("Trajectoire moyenne (bande : IC 95 % bootstrap ; gris : chaque événement)")
    a2.plot(k, es["hit_rate"], color=color, lw=1.5)
    a2.axhline(0.5, color=T["axis"], lw=0.9)
    a2.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    a2.set_ylim(0, 1)
    a2.set_xlabel("séances après l'exécution")
    a2.set_title("% d'événements au-dessus de la dérive")
    fig.suptitle(title, x=0.01, ha="left", fontsize=11.5, fontweight="bold", color=T["ink"])
    return fig


def plot_holding(curves: dict, metric="Sharpe", title="Performance selon la durée de détention"):
    """curves : {nom: (df holding_curve, couleur)}"""
    fig, ax = plt.subplots(figsize=(11, 3.8))
    for name, (df, colr) in curves.items():
        ax.plot(df.index, df[metric], color=colr, marker="o", ms=5, mec=T["surface"], mew=1.2, label=name)
    _zero(ax)
    _horizon_axis(ax, list(next(iter(curves.values()))[0].index))
    ax.set_xlabel("durée de détention H (séances)")
    ax.set_ylabel(metric)
    ax.legend(loc="best")
    ax.set_title(title)
    return fig


# ----------------------------------------------------------------------------- backtest
def plot_backtest(res: nl.StrategyResult, start=None, end=None, title="Stratégie vs Buy & Hold", label="Stratégie",
                  color=None):
    color = color or T["strategy"]
    b = res.bt.loc[start:end]
    e = float(b["expo"].mean())
    rets = {"Buy & Hold": (b["bench"], T["bench"]),
            f"B&H expo constante {e:.0%}": (b["rf"] + e * (b["bench"] - b["rf"]), T["bench2"]),
            label: (b["net"], color)}
    fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, height_ratios=[2, 1, 0.8])
    ax = axes[0]
    ends = []
    for name, (r, colr) in rets.items():
        eq = (1 + r).cumprod()
        ax.plot(eq.index, eq, color=colr, lw=1.9 if name == label else 1.3, label=f"{name} (×{eq.iloc[-1]:.2f})")
        ends.append((eq.index[-1], eq.iloc[-1], f"×{eq.iloc[-1]:.2f}", colr))
    _log_multiple_axis(ax)
    _end_labels(ax, ends[::-1])
    ax.legend(loc="upper left")
    ax.set_title(title + " — capital (échelle log)")
    for name, (r, colr) in list(rets.items())[::2]:
        dd = nl.drawdown_series(r)
        if name == label:
            axes[1].fill_between(dd.index, dd, 0, color=colr, alpha=0.12, lw=0)
        axes[1].plot(dd.index, dd, color=colr, lw=1.1, label=name)
    axes[1].yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    axes[1].legend(loc="lower left", ncols=2)
    axes[1].set_title("Drawdown")
    ex = b["expo"]
    axes[2].fill_between(ex.index, 0, ex, color=color, alpha=0.12, lw=0, step="post")
    axes[2].plot(ex.index, ex, color=color, lw=1.1, drawstyle="steps-post")
    axes[2].yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    axes[2].set_title(f"Exposition (moyenne {e:.0%})")
    return fig


def plot_equity_curves(curves: dict, title="Comparaison", log=True):
    """curves : {nom: (série de rendements, couleur)} — chaque courbe part de 1."""
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, height_ratios=[2, 1])
    ends = []
    for name, (r, colr) in curves.items():
        eq = (1 + r.fillna(0)).cumprod()
        axes[0].plot(eq.index, eq, color=colr, lw=1.5, label=f"{name} (×{eq.iloc[-1]:.2f})")
        ends.append((eq.index[-1], eq.iloc[-1], f"×{eq.iloc[-1]:.2f}", colr))
        axes[1].plot(eq.index, nl.drawdown_series(r), color=colr, lw=1.1, label=name)
    if log:
        _log_multiple_axis(axes[0])
    _end_labels(axes[0], ends[::-1])
    axes[0].legend(loc="upper left")
    axes[0].set_title(title + " — capital")
    axes[1].yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    axes[1].set_title("Drawdown")
    return fig


def plot_yearly(yearly: pd.DataFrame, label="Stratégie", color=None):
    color = color or T["strategy"]
    fig, ax = plt.subplots(figsize=(11, 3.6))
    x = np.arange(len(yearly))
    w, gap = 0.3, 0.03
    ax.bar(x - (w + gap) / 2, yearly[label] if label in yearly else yearly.iloc[:, 0], width=w, color=color, label=label)
    ax.bar(x + (w + gap) / 2, yearly["Buy & Hold"], width=w, color=T["bench"], label="Buy & Hold")
    _zero(ax)
    ax.set_xticks(x, [str(i) for i in yearly.index])
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.legend(loc="upper left", ncols=2)
    ax.set_title("Rendement par année civile")
    ax.grid(axis="x", visible=False)
    return fig


# ----------------------------------------------------------------------------- optimisation
def plot_is_oos(df: pd.DataFrame, bh_is=None, bh_oos=None, top=20, title="Sur-apprentissage : score in-sample vs out-of-sample"):
    fig, ax = plt.subplots(figsize=(7.5, 6))
    ax.scatter(df["IS"], df["OOS"], s=10, color=T["muted"], alpha=0.35, lw=0, label="toutes les configs")
    best = df.nlargest(top, "IS")
    ax.scatter(best["IS"], best["OOS"], s=40, color=T["strategy"], edgecolor=T["surface"], linewidth=1.5, zorder=5,
               label=f"{top} meilleures in-sample")
    if bh_is is not None:
        ax.axvline(bh_is, color=T["axis"], lw=0.9)
        ax.axhline(bh_oos, color=T["axis"], lw=0.9)
        ax.annotate("Buy & Hold", (bh_is, bh_oos), xytext=(5, 5), textcoords="offset points", fontsize=8.5, color=T["ink2"])
    rho = df[["IS", "OOS"]].corr(method="spearman").iloc[0, 1]
    ax.set_xlabel("score in-sample")
    ax.set_ylabel("score out-of-sample")
    ax.legend(loc="best")
    ax.set_title(f"{title}\ncorrélation de rang IS/OOS = {rho:.2f}")
    return fig


def param_pivot(configs: pd.DataFrame, scores: pd.Series, row: str, col: str, agg="mean") -> pd.DataFrame:
    d = configs.assign(_s=scores.to_numpy())
    return d.pivot_table(index=row, columns=col, values="_s", aggfunc=agg)


def plot_param_pair(configs: pd.DataFrame, scores: pd.Series, row: str, col: str, center: float, objective="score"):
    """Deux cartes côte à côte : score MOYEN sur les autres paramètres (robustesse) et MEILLEUR score (potentiel).
    Même échelle de couleurs, centrée sur le buy & hold."""
    mean = param_pivot(configs, scores, row, col, "mean")
    best = param_pivot(configs, scores, row, col, "max")
    allv = np.r_[mean.to_numpy().ravel(), best.to_numpy().ravel()]
    vmax = float(np.nanmax(np.abs(allv - center))) or 1.0
    nr, nc = mean.shape
    fig, axes = plt.subplots(1, 2, figsize=(max(9.5, 2 * (0.6 * nc + 2.4)), max(2.3, 0.42 * nr + 1.5)))
    plot_heatmap(mean, title="moyenne sur les autres paramètres", center=center, vmax=vmax, ax=axes[0])
    plot_heatmap(best, title="meilleure config de la case", center=center, vmax=vmax, ax=axes[1], cbar_label=objective)
    fig.suptitle(f"{objective} : {row} × {col} (gris = buy & hold, bleu = mieux)", x=0.01, ha="left",
                 fontsize=11.5, fontweight="bold", color=T["ink"])
    return fig


def plot_wfo(oos: pd.Series, bench: pd.Series, chosen: pd.DataFrame, title="Walk-forward : performance 100 % hors-échantillon"):
    curves = {"Buy & Hold": (bench.loc[oos.index], T["bench"]), "Walk-forward": (oos, T["strategy"])}
    fig = plot_equity_curves(curves, title=title)
    for d in pd.to_datetime(chosen["début test"]):
        for ax in fig.axes:
            ax.axvline(d, color=T["grid"], lw=0.9, zorder=0)
    return fig


def plot_coefs(coefs: pd.DataFrame, title="Coefficients du modèle au fil des ré-entraînements"):
    c = coefs.copy()
    c.index = [d.strftime("%Y-%m") for d in c.index]
    step = max(1, len(c) // 14)
    c = c.iloc[::step]
    return plot_heatmap(c.T, title=title, fmt="{:.2f}", annotate=False, figsize=(11, 4.2), xlabel="ré-entraînement",
                        ylabel="", cbar_label="coefficient (variables standardisées)")


# ----------------------------------------------------------------------------- tableau de bord
def style_stats(df: pd.DataFrame):
    pct = ["CAGR", "Volatilité", "Max drawdown", "% jours positifs", "Exposition moy.", "% temps investi", "Alpha annuel",
           "CAGR B&H même expo", "Gain de timing (CAGR)"]
    fmt = {}
    for r in df.index:
        fmt[r] = "{:.1%}" if r in pct else ("{:.0f}" if r in ("Nb transactions",) else "{:.2f}")

    def f(v, r):
        try:
            return fmt[r].format(v) if pd.notna(v) else "—"
        except (ValueError, TypeError):
            return str(v)
    return df.apply(lambda col: pd.Series([f(v, r) for r, v in col.items()], index=col.index))


def make_dashboard(D: nl.Data, base_params: dict, start=None, end=None):
    """Tableau de bord ipywidgets : chaque réglage relance le backtest (≈ 0,1 s) et redessine."""
    import ipywidgets as W
    from IPython.display import display

    p0 = nl.make_params(base_params)
    lay = W.Layout(width="330px")
    sty = {"description_width": "125px"}

    def dd(k, opts, desc):
        return W.Dropdown(options=opts, value=p0[k], description=desc, layout=lay, style=sty)

    def isl(k, a, b, s, desc):
        return W.IntSlider(value=int(p0[k]), min=a, max=b, step=s, description=desc, continuous_update=False, layout=lay, style=sty)

    def fsl(k, a, b, s, desc):
        return W.FloatSlider(value=float(p0[k]), min=a, max=b, step=s, description=desc, continuous_update=False,
                             readout_format=".2f", layout=lay, style=sty)

    w = {
        "news_source": dd("news_source", ["articles", "gdelt", "blend"], "Source news"),
        "news_mode": dd("news_mode", ["level", "momentum", "attention"], "Mesure news"),
        "news_score": dd("news_score", ["prob", "vote", "neg", "pos"], "Score article"),
        "news_halflife": isl("news_halflife", 2, 250, 1, "Demi-vie news (j)"),
        "news_z_window": isl("news_z_window", 60, 1000, 10, "Fenêtre norm. news"),
        "news_direction": dd("news_direction", [("suivre", 1), ("contrarien", -1)], "Sens news"),
        "price_kind": dd("price_kind", ["tsmom", "composite", "ma", "sma_dist", "breakout", "drawdown", "hmm"], "Signal prix"),
        "price_lookback": isl("price_lookback", 10, 500, 5, "Fenêtre prix (j)"),
        "price_fast": isl("price_fast", 5, 100, 1, "EMA rapide (ma)"),
        "price_slow": isl("price_slow", 20, 400, 5, "EMA lente (ma)"),
        "price_direction": dd("price_direction", [("tendance", 1), ("retour moyenne", -1)], "Sens prix"),
        "price_hmm_features": dd("price_hmm_features", ["ret", "ret_vol", "trend_vol"], "HMM : variables"),
        "price_hmm_states": dd("price_hmm_states", [2, 3], "HMM : régimes"),
        "price_hmm_signal": dd("price_hmm_signal", ["regime", "exp_ret"], "HMM : signal"),
        "price_hmm_sticky": fsl("price_hmm_sticky", 0, 500, 25, "HMM : persistance"),
        "combo": dd("combo", ["linear", "agree", "filter"], "Combinaison"),
        "w_news": fsl("w_news", 0, 1, 0.05, "Poids news"),
        "filter_threshold": fsl("filter_threshold", -3, 1, 0.1, "Seuil filtre news"),
        "mapping": dd("mapping", ["threshold", "linear", "sigmoid"], "Score → position"),
        "entry": fsl("entry", -2, 2, 0.05, "Seuil entrée"),
        "exit": fsl("exit", -3, 2, 0.05, "Seuil sortie"),
        "base": fsl("base", 0, 1, 0.05, "Expo de base"),
        "scale": fsl("scale", 0, 3, 0.05, "Pente"),
        "long_only": W.Checkbox(value=bool(p0["long_only"]), description="Long only", layout=lay, style=sty),
        "max_leverage": fsl("max_leverage", 0.5, 2, 0.05, "Levier max"),
        "rebalance": isl("rebalance", 1, 63, 1, "Rebalancement (j)"),
        "min_trade": fsl("min_trade", 0, 0.5, 0.05, "Bande no-trade"),
        "vol_target": fsl("vol_target", 0, 0.4, 0.01, "Vol cible (0=off)"),
        "risk_overlay": dd("risk_overlay", ["none", "hmm", "vol"], "Filtre de risque"),
        "overlay_k": fsl("overlay_k", 0, 1, 0.05, "Force du filtre"),
        "exec_lag": isl("exec_lag", 0, 5, 1, "Délai exécution (j)"),
        "cost_bps": fsl("cost_bps", 0, 30, 0.5, "Coût (bps)"),
    }
    groups = [("News", ["news_source", "news_mode", "news_score", "news_halflife", "news_z_window", "news_direction"]),
              ("Prix", ["price_kind", "price_lookback", "price_fast", "price_slow", "price_direction",
                        "price_hmm_features", "price_hmm_states", "price_hmm_signal", "price_hmm_sticky"]),
              ("Combinaison & position", ["combo", "w_news", "filter_threshold", "mapping", "entry", "exit", "base", "scale"]),
              ("Risque & exécution", ["long_only", "max_leverage", "rebalance", "min_trade", "vol_target",
                                      "risk_overlay", "overlay_k", "exec_lag", "cost_bps"])]
    cols = [W.VBox([W.HTML(f"<b>{g}</b>")] + [w[k] for k in ks]) for g, ks in groups]
    ui = W.VBox([W.HBox(cols[:2]), W.HBox(cols[2:])])

    def render(**kw):
        p = {**p0, **kw}
        res = nl.run_strategy(D, p)
        display(style_stats(res.compare(start, end)))
        fig = plot_backtest(res, start, end)
        display(fig)
        plt.close(fig)
        changed = {k: v for k, v in p.items() if nl.DEFAULT_PARAMS.get(k) != v}
        print("Paramètres (différences vs défauts) — à copier dans PARAMS :\n", changed)

    out = W.interactive_output(render, w)
    display(ui, out)
    return w

---
<a id="partie-7"></a>
## Partie 7 · La plateforme

### 7.1 · API, tableau de bord, site

| Fichier | Rôle |
|:--|:--|
| `app/api.py` | API FastAPI qui sert le signal, l'historique et le prédicteur (on règle les variables, l'API renvoie la probabilité et les contributions). |
| `app/dashboard.py` | Tableau de bord Streamlit. |
| `app/export_site.py` | Fabrique le site statique publié sur GitHub Pages : relit `runs/`, les notebooks et les figures, écrit `site/data/donnees.js`, convertit les notebooks en HTML et prépare les archives téléchargeables. |
| `app/export_statique.py`, `app/build_site.py` | Anciennes versions de l'export. |

> **Concepts** · service d'un modèle figé, séparation calcul / affichage, site statique reproductible.

In [ ]:
# ─── app/api.py · 254 lignes ───

"""API du projet Nikkei : le point d'accès qui sert la prédiction (« served prediction endpoint » de l'énoncé).

Lancer depuis la racine du dépôt : uvicorn app.api:app --port 8000
Documentation interactive : http://127.0.0.1:8000/docs
Le site de présentation (dossier web/, en PHP) interroge cette API.
"""
import json
from functools import lru_cache

import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException, Query

from src.config import COLONNES, FEATURES, RUNS
from src.data import charger_table
from src.decision import charger_decision
from src.logreg_scratch import constante_lipschitz, cout, descente, gradient  # annexe de Maths
from src.predict import contributions, historique, signal

app = FastAPI(title="Projet Nikkei", version="1.1",
              description="Signal mensuel acheter / neutre / vendre sur le Nikkei 225 pour un investisseur en euros.")
TABLE = charger_table()


@app.get("/sante")
def sante() -> dict:
    """État du service et de la décision figée."""
    dec = charger_decision()
    return {"statut": "ok", "modele": dec["modele_nikkei"], "decision_calculee_le": dec["date"],
            "dernier_mois_disponible": TABLE.index.max().strftime("%Y-%m")}


@app.get("/mois")
def mois_disponibles() -> list[dict]:
    """Mois pour lesquels un signal peut être demandé (validation, test, mois à prédire), du plus récent au plus ancien."""
    d = TABLE[TABLE["bloc"].isin(["validation", "test", "à prédire"])]
    return [{"mois": date.strftime("%Y-%m"), "bloc": bloc} for date, bloc in zip(d.index[::-1], d["bloc"][::-1])]


@app.get("/signal")
def signal_du_mois(mois: str | None = Query(None, pattern=r"^\d{4}-\d{2}$", description="mois des données, AAAA-MM")) -> dict:
    """Probabilité de hausse du mois suivant, score, action du modèle et recommandation finale."""
    try:
        return signal(mois, TABLE)
    except KeyError as erreur:
        raise HTTPException(status_code=404, detail=str(erreur))


@app.get("/explication")
def explication(mois: str | None = Query(None, pattern=r"^\d{4}-\d{2}$")) -> list[dict]:
    """Contributions SHAP de chaque variable au log-odds de la baisse, de la plus forte à la plus faible."""
    try:
        return contributions(mois, TABLE).round(4).to_dict(orient="records")
    except KeyError as erreur:
        raise HTTPException(status_code=404, detail=str(erreur))


@app.get("/historique")
def historique_des_decisions(bloc: str = Query("test", pattern="^(validation|test)$")) -> list[dict]:
    """Décisions mois par mois et rendements réalisés, sur la validation ou le test."""
    h = historique(TABLE, blocs=(bloc,)).round(6).reset_index()   # 6 décimales : les rendements sont composés côté site
    h["date"] = h["date"].dt.strftime("%Y-%m")
    return h.to_dict(orient="records")


@app.get("/evaluation")
def evaluation() -> dict:
    """Résultats de l'évaluation unique sur le test 2024-2026."""
    return json.loads((RUNS / "evaluation_test.json").read_text())


# Variables d'information que le prédicteur du site laisse régler (les autres gardent leur valeur du mois de base)
REGLABLES = {
    "news_attention_6m": ("Volume anormal d'articles", "ratio"),
    "news_sent_w": ("Sentiment des news (FinBERT)", "score"),
    "n225_mom_12m": ("Momentum du Nikkei sur 12 mois", "pourcent"),
    "n225_mom_3m": ("Momentum du Nikkei sur 3 mois", "pourcent"),
    "d_jgb10y_bp": ("Variation du taux japonais à 10 ans", "pb"),
    "carry_1m": ("Portage de la couverture", "annuel"),
}


def _ligne_de_base(mois: str | None):
    lignes = TABLE[TABLE["bloc"] == "à prédire"] if mois is None else TABLE[TABLE.index.strftime("%Y-%m") == mois]
    if lignes.empty:
        raise HTTPException(status_code=404, detail=f"aucune donnée pour le mois {mois}")
    return lignes.index[0]


@app.get("/variables")
def variables_reglables(mois: str | None = Query(None, pattern=r"^\d{4}-\d{2}$")) -> list[dict]:
    """Variables d'information réglables : libellé, bornes observées sur l'entraînement et valeur du mois de base."""
    date = _ligne_de_base(mois)
    train = TABLE[TABLE["bloc"] == "entraînement"]
    return [{"nom": nom, "libelle": libelle, "affichage": affichage,
             "min": round(float(train[nom].quantile(0.01)), 5), "max": round(float(train[nom].quantile(0.99)), 5),
             "valeur": round(float(TABLE.loc[date, nom]), 5)} for nom, (libelle, affichage) in REGLABLES.items()]


@app.get("/prediction")
def prediction_selon_l_information(
        mois: str | None = Query(None, pattern=r"^\d{4}-\d{2}$", description="mois de base, AAAA-MM"),
        news_attention_6m: float | None = None, news_sent_w: float | None = None, n225_mom_12m: float | None = None,
        n225_mom_3m: float | None = None, d_jgb10y_bp: float | None = None, carry_1m: float | None = None) -> dict:
    """Prédiction du modèle pour une information choisie : on part d'un mois réel et on remplace certaines variables.

    C'est une simulation « et si » : elle montre comment le modèle réagit, pas ce que le marché fera.
    """
    date = _ligne_de_base(mois)
    table = TABLE.copy()
    reglages = {"news_attention_6m": news_attention_6m, "news_sent_w": news_sent_w, "n225_mom_12m": n225_mom_12m,
                "n225_mom_3m": n225_mom_3m, "d_jgb10y_bp": d_jgb10y_bp, "carry_1m": carry_1m}
    modifiees = {nom: valeur for nom, valeur in reglages.items() if valeur is not None}
    for nom, valeur in modifiees.items():
        table.loc[date, nom] = valeur
    cle = date.strftime("%Y-%m")
    resultat = signal(cle, table)
    resultat["variables_modifiees"] = modifiees
    resultat["contributions"] = contributions(cle, table).round(4).to_dict(orient="records")
    resultat.pop("rendement_realise_mois_suivant", None)
    return resultat


@app.get("/decision")
def decision_figee() -> dict:
    """Paramètres de la décision figée avant le test : coûts mesurés sur l'entraînement, marge et seuils."""
    dec = charger_decision()
    return {"couts": dec["couts_entrainement_exces_du_cash"], "marge": dec["marge_m"], "seuils": dec["seuils"],
            "platt": dec["platt"][dec["modele_nikkei"]], "couverture": dec["couverture"]}


@app.get("/serie")
def serie_du_nikkei() -> list[dict]:
    """Clôture mensuelle du Nikkei 225, bloc du split, classe du mois suivant et présence de news (scènes 3D)."""
    d = TABLE[["n225_close", "bloc", "y_baisse", "news_dispo"]]
    return [{"mois": date.strftime("%Y-%m"), "cloture": round(float(c), 1), "bloc": b,
             "baisse": None if pd.isna(y) else int(y), "news": int(n)}
            for date, c, b, y, n in zip(d.index, d["n225_close"], d["bloc"], d["y_baisse"], d["news_dispo"])]


@app.get("/essais")
def essais_optuna() -> list[dict]:
    """Journal des 200 essais Optuna : cible, modèle, numéro de l'essai, log-loss de validation croisée."""
    journal = pd.read_csv(RUNS / "trials_log.csv")
    return [{"cible": c, "modele": m, "essai": int(e), "logloss": round(float(v), 5), "retenu": bool(r)}
            for c, m, e, v, r in zip(journal["cible"], journal["modele"], journal["essai"], journal["cv_logloss"], journal["retenu"])]


@app.get("/volatilite")
def volatilite() -> dict:
    """Second volet (ajouté après le test) : volatilité prévue et réalisée mois par mois, et chiffres de synthèse."""
    bilan = json.loads((RUNS / "exploration_volatilite.json").read_text())
    serie = pd.read_csv(RUNS / "exploration_volatilite_predictions.csv", parse_dates=["date"])
    nul = lambda x: None if pd.isna(x) else round(float(x), 4)
    points = [{"mois": d.strftime("%Y-%m"), "bloc": b, "reelle": nul(r), "prevue": nul(p), "moyenne": nul(m),
               "p_agite": nul(pa), "agite": None if pd.isna(a) else int(a), "rendement": nul(g)}
              for d, b, r, p, m, pa, a, g in zip(serie["date"], serie["bloc"], serie["vol_reelle"], serie["vol_prevue"],
                                                 serie["vol_moyenne_passee"], serie["p_agite"], serie["agite"], serie["rendement_suivant"])]
    return {"avertissement": bilan["avertissement"], "methode": bilan["methode"], "serie": points,
            "prevision": bilan["prevision_de_la_volatilite"], "agitation": bilan["mois_agite_ou_non"],
            "usage": bilan["usage"], "mois_prochain": bilan["mois_prochain"]}


@app.get("/signal_risque")
def signal_risque() -> dict:
    """Signal d'achat fondé sur le risque prévu (après coup, règle figée sur 2000-2015) : synthèse et série hebdomadaire."""
    bilan = json.loads((RUNS / "signal_risque.json").read_text())
    d = pd.read_csv(RUNS / "signal_risque_journalier.csv", index_col=0, parse_dates=True)
    garder, regle, mois = "valeur_acheter et garder", "valeur_signal risque, chaque jour (règle figée)", "valeur_signal risque, seulement en fin de mois"
    semaines = d.resample("W-FRI").last().dropna(subset=[garder])
    nul = lambda x: None if pd.isna(x) else round(float(x), 4)
    serie = [{"date": t.strftime("%Y-%m-%d"), "vol": nul(v), "signal": nul(s), "garder": nul(g), "regle": nul(r), "mois": nul(m)}
             for t, v, s, g, r, m in zip(semaines.index, semaines["volatilite_prevue"], semaines["signal"], semaines[garder], semaines[regle], semaines[mois])]
    episodes = {}
    for nom, (a, b) in {"crise de 2008": ("2008-09-01", "2009-03-31"), "Covid, 2020": ("2020-02-17", "2020-06-30"),
                        "août 2024": ("2024-07-29", "2024-09-30")}.items():
        x = d.loc[a:b]
        episodes[nom] = {"garder": nul(x[garder].iloc[-1] / x[garder].iloc[0] - 1), "regle": nul(x[regle].iloc[-1] / x[regle].iloc[0] - 1)}
    return {"avertissement": bilan["avertissement"], "regle": bilan["regle"], "prevision": bilan["prevision_r2_log_volatilite"],
            "strategies": bilan["strategies"]["couvert"], "comparaisons": bilan["comparaisons"]["couvert"],
            "aujourd_hui": bilan["aujourd_hui"], "episodes": episodes, "serie": serie}


@app.get("/verifications")
def verifications_apres_coup() -> dict:
    """Résumé des trois vérifications faites après le test : vingt modèles, base journalière, historique long."""
    modeles = json.loads((RUNS / "exploration_apres_coup.json").read_text())
    jours = json.loads((RUNS / "exploration_journaliere.json").read_text())
    long = json.loads((RUNS / "exploration_historique_long.json").read_text())
    classement = sorted(({"modele": r["modele"], "fenetre": r["fenetre"], "log_loss": round(r["test_2024_2026"]["log_loss"], 4),
                          "ecart": round(r["test_2024_2026"]["ecart_vs_constante"], 4),
                          "erreur_type": round(r["test_2024_2026"]["erreur_type_de_l_ecart"], 4),
                          "cumul_regle": round(r["test_2024_2026"]["cumul_regle"], 4)} for r in modeles["resultats"]),
                        key=lambda r: r["log_loss"])
    par_jour = {nom: {"observations": e["test_2024_2026"]["observations"],
                      **min(({"modele": m, "ecart": round(v["ecart_vs_constante"], 4), "erreur_type": round(v["erreur_type"], 4),
                              "auc": round(v["auc"], 3)} for m, v in e["test_2024_2026"]["modeles"].items() if m != "constante"),
                            key=lambda v: v["ecart"])} for nom, e in jours["sens_du_marche"].items()}
    departs = {depart: {m: round(v["log_loss"], 4) for m, v in e["test_2024_2026"]["modeles"].items()}
               for depart, e in long["resultats_par_depart"].items()}
    return {"avertissement": modeles["avertissement"], "vingt_modeles": {"configurations": modeles["configurations"],
            "battent_la_constante": modeles["battent_la_constante"], "choix_sur_2016_2023": modeles["choix_sur_2016_2023"]["modele"],
            "classement_sur_le_test": classement}, "base_journaliere": par_jour, "historique_long": departs}


@lru_cache(maxsize=1)
def _surface() -> dict:
    """Surface de coût d'un modèle réduit à deux variables, et trajectoires de descente de gradient.

    Modèle réduit : volume anormal d'articles et momentum 12 mois, standardisés par la pipeline du modèle final ;
    même λ que le modèle final ; l'intercept est fixé à son optimum pour que la surface soit en deux dimensions.
    """
    dec = charger_decision()
    pipe = joblib.load(RUNS / "models" / "nikkei_logit.joblib")
    train = TABLE[TABLE["bloc"] == "entraînement"]
    Z = pipe[:-1].transform(train[COLONNES])
    X = Z[:, [FEATURES.index("news_attention_6m"), FEATURES.index("n225_mom_12m")]]
    y = train["y_baisse"].to_numpy()
    lam = 1 / (2 * len(y) * pipe.named_steps["model"].C)
    w_opt, b_opt, _ = descente(X, y, 1 / constante_lipschitz(X, lam), lam, 20000)
    # b est fixé : la borne et la limite locale se calculent sur w seulement
    L = 0.25 * np.linalg.eigvalsh(X.T @ X / len(y)).max() + 2 * lam
    p_opt = 1 / (1 + np.exp(-(X @ w_opt + b_opt)))
    hessienne = X.T @ (X * (p_opt * (1 - p_opt))[:, None]) / len(y) + 2 * lam * np.eye(2)
    limite = 2 / np.linalg.eigvalsh(hessienne).max()

    def cout_w(w):
        return cout(np.asarray(w, dtype=float), b_opt, X, y, lam)

    axe = np.linspace(-1.6, 1.6, 41)
    grille = [[round(float(cout_w([w1, w2])), 5) for w1 in axe] for w2 in axe]
    depart = np.array([-1.35, 1.25])
    pas = {"trop petit": 0.05, "classique (1/L)": 1 / L, "près de la limite": 0.93 * limite, "au-delà de la limite": 1.12 * limite}
    trajectoires = {}
    for nom, alpha in pas.items():
        w, points = depart.copy(), []
        for _ in range(60):
            points.append([round(float(w[0]), 4), round(float(w[1]), 4), round(float(cout_w(w)), 5)])
            w = w - alpha * gradient(w, b_opt, X, y, lam)[0]
            if np.abs(w).max() > 1.6:      # sortie de la grille : la descente diverge
                break
        trajectoires[nom] = {"alpha": round(float(alpha), 4), "points": points,
                             "converge": bool(np.abs(w - w_opt).max() < 1e-2)}
    return {"axe": [round(float(a), 3) for a in axe], "cout": grille, "lambda": round(float(lam), 4),
            "L": round(float(L), 4), "limite_locale": round(float(limite), 3), "optimum": [round(float(w_opt[0]), 4), round(float(w_opt[1]), 4), round(float(cout_w(w_opt)), 5)],
            "variables": ["volume anormal d'articles", "momentum 12 mois"], "trajectoires": trajectoires,
            "modele": dec["modele_nikkei"]}


@app.get("/surface")
def surface_de_cout() -> dict:
    """Surface J(w1, w2) d'une logistique réduite à deux variables et descentes de gradient pour quatre pas."""
    return _surface()

In [ ]:
# ─── app/dashboard.py · 116 lignes ───

"""Tableau de bord du projet Nikkei (démo de l'oral).

Lancer depuis la racine du dépôt : streamlit run app/dashboard.py
"""
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from src.config import FIG, RUNS  # noqa: E402
from src.data import charger_table  # noqa: E402
from src.decision import charger_decision  # noqa: E402
from src.predict import contributions, historique, signal  # noqa: E402

st.set_page_config(page_title="Projet Nikkei", layout="wide")
COULEURS = {"acheter": "#548235", "neutre": "#9e9e9e", "vendre": "#c00000"}


@st.cache_data
def donnees():
    table = charger_table()
    return table, historique(table), json.loads((RUNS / "evaluation_test.json").read_text())


TABLE, HIST, EVAL = donnees()
DEC = charger_decision()
MOIS = [d.strftime("%Y-%m") for d in TABLE.index[TABLE["bloc"].isin(["validation", "test", "à prédire"])]]

st.title("Nikkei 225 : acheter, rester neutre ou vendre ?")
st.caption("Signal mensuel pour un investisseur en euros · régression logistique calibrée · décision figée avant le test")

onglets = st.tabs(["Signal du mois", "Pourquoi ce signal", "Historique", "Le modèle a-t-il marché ?", "Méthode"])

with onglets[0]:
    mois = st.selectbox("Données de fin de mois", MOIS[::-1], index=0)
    s = signal(mois, TABLE)
    st.error(f"**Recommandation finale : {s['recommandation_finale']}.** "
             "Sur le test 2024-2026, le modèle ne bat pas le modèle constant : sa règle est montrée à titre d'analyse.")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Probabilité de hausse le mois suivant", f"{s['p_hausse']:.0%}")
    c2.metric("Score S = 2p − 1", f"{s['score']:+.2f}")
    c3.metric("Action de la règle du modèle", s["action_du_modele"])
    c4.metric("Couverture si achat", "couvert")

    fig, ax = plt.subplots(figsize=(9, 1.3))
    tv, ta = s["seuils"]["theta_v"], s["seuils"]["theta_a"]
    ax.axvspan(0, tv, color=COULEURS["vendre"], alpha=0.25)
    ax.axvspan(tv, ta, color=COULEURS["neutre"], alpha=0.25)
    ax.axvspan(ta, 1, color=COULEURS["acheter"], alpha=0.25)
    ax.axvline(s["p_hausse"], color="black", lw=3)
    for x, texte in [(tv / 2, "vendre"), ((tv + ta) / 2, "neutre"), ((ta + 1) / 2, "acheter")]:
        ax.text(x, 0.5, texte, ha="center", va="center", fontsize=10)
    ax.set_xlim(0, 1); ax.set_yticks([]); ax.set_xlabel("probabilité calibrée de hausse")
    st.pyplot(fig)
    st.caption(f"Seuils tirés de l'espérance de gain en euros : θ_v = {tv:.3f}, θ_a = {ta:.3f} (marge de 0,5 %/mois).")
    if s["rendement_realise_mois_suivant"] is None:
        st.info("Le mois suivant n'est pas encore connu. Attention : la collecte des news de fin 2026 est incomplète.")
    else:
        st.write(f"Ce qui s'est passé ensuite : Nikkei **{100 * s['rendement_realise_mois_suivant']:+.1f} %** le mois suivant.")

with onglets[1]:
    mois_x = st.selectbox("Mois à expliquer", MOIS[::-1], index=0, key="explication")
    c = contributions(mois_x, TABLE).head(8).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(c["variable"], c["contribution"], color=np.where(c["contribution"] > 0, "#c55a11", "#1f4e79"))
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("contribution au log-odds de la baisse (> 0 : pousse vers « baisse »)")
    st.pyplot(fig)
    st.caption("SHAP d'un modèle linéaire : poids × écart de la variable standardisée à sa moyenne d'entraînement. "
               "SHAP décrit le modèle, pas l'économie : il n'établit aucune causalité (S9).")
    st.dataframe(contributions(mois_x, TABLE).round(3), use_container_width=True)

with onglets[2]:
    bloc = st.radio("Période", ["test", "validation"], horizontal=True)
    h = HIST[HIST["bloc"] == bloc]
    cumul = (1 + h[["rendement_regle", "rendement_achat_couvert", "rendement_cash"]]).cumprod()
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(cumul.index, cumul["rendement_achat_couvert"], color="#c55a11", label="toujours acheter, couvert")
    ax.plot(cumul.index, cumul["rendement_regle"], color="#1f4e79", lw=2, label="règle du modèle")
    ax.plot(cumul.index, cumul["rendement_cash"], color="grey", ls="--", label="cash")
    ax.scatter(cumul.index, cumul["rendement_regle"], c=[COULEURS[a] for a in h["action"]], s=22, zorder=3)
    ax.set_ylabel("valeur de 1 €"); ax.legend(frameon=False)
    st.pyplot(fig)
    croise = pd.crosstab(h["action"], h["hausse_realisee"].map({1: "hausse réalisée", 0: "baisse réalisée"}))
    st.write("Actions et réalité du mois suivant :")
    st.dataframe(croise, use_container_width=True)

with onglets[3]:
    m = pd.DataFrame(EVAL["metriques"]).T[["log-loss", "Brier", "ECE", "PR-AUC"]]
    p = pd.DataFrame(EVAL["performance_euros"]).T
    c1, c2 = st.columns(2)
    c1.write("**Probabilités sur le test 2024-2026** (plus bas = mieux, sauf PR-AUC)")
    c1.dataframe(m.round(3), use_container_width=True)
    c2.write("**Rendement en euros sur le test**")
    c2.dataframe(p.round(3), use_container_width=True)
    st.write(f"Plancher sans compétence battu : **{'oui' if EVAL['plancher_battu'] else 'non'}** · "
             f"recommandation : **{EVAL['recommandation']}**.")
    for f in ["03b_test_probabilites.png", "04_inversion_signal_news.png"]:
        if (FIG / f).exists():
            st.image(str(FIG / f))

with onglets[4]:
    st.markdown(f"""
- **Données** : 200 mois (février 2010 → septembre 2026), news GDELT notées par FinBERT à partir de 2017.
- **Split temporel** : entraînement 2010-2021, validation 2022-2023, test 2024-2026 évalué une seule fois.
- **Modèle** : régression logistique L2 (choisie par la règle de baseline de S1 face au Random Forest et à LightGBM),
  calibrée par Platt sur les prédictions hors fold.
- **Décision** : espérance de gain en euros avec des coûts mesurés sur l'entraînement et une marge fixée à l'avance
  ({100 * DEC['marge_m']:.1f} %/mois) ; couverture : {DEC['couverture']}.
- **Traçabilité** : décision commitée (`75712cf`) avant l'évaluation du test (`6efd12c`).
""")

In [ ]:
# ─── app/export_site.py · 489 lignes ───

"""Prépare le site « Visite et Lecture » (dossier site/) : données, figures, notebooks et PDF.

Le site est statique : il ne demande ni PHP ni API vivante. Ce script interroge l'API en mémoire (sans serveur),
écrit toutes les réponses dans site/data/donnees.js, puis copie à côté les figures, les notebooks (en HTML pour les lire,
en .ipynb et en .zip pour les télécharger) et les deux PDF. À relancer après tout changement des résultats ou des notebooks :

    python -m app.export_site

Puis, pour regarder le site : python -m http.server 8095 --directory site  (adresse http://localhost:8095).
"""
import json
import re
import shutil
import subprocess
import warnings
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
SITE = ROOT / "site"

ROUTES = {
    "sante": "/sante",
    "evaluation": "/evaluation",
    "decision": "/decision",
    "serie": "/serie",
    "essais": "/essais",
    "surface": "/surface",
    "volatilite": "/volatilite",
    "signal_risque": "/signal_risque",
    "verifications": "/verifications",
    "signal": "/signal",
    "explication": "/explication",
    "historique_test": "/historique?bloc=test",
    "historique_validation": "/historique?bloc=validation",
}

NOTEBOOKS = [
    ("00a_collecte_V1", "Collecte, première version : prix, taux, change, news"),
    ("00b_collecte_V2", "Collecte, seconde version : 1999-2026, nouvelles sources"),
    ("01_cadrage_eda_variables", "Cadrage, audit des fuites, exploration, variables, découpage"),
    ("02_pipeline_modeles_optuna", "Pipeline, trois modèles, Optuna, journal des essais"),
    ("03_calibration_seuils", "Calibration, choix du modèle, règle de décision figée"),
    ("03b_test_final", "Évaluation unique sur le test 2024-2026"),
    ("04_explicabilite_diagnostics", "Tranches, permutation, SHAP, dépendances partielles"),
    ("05_annexe_maths_logistique", "Annexe de Maths : la régression logistique à la main"),
    ("06_annexe_news_prix", "Annexe : signal news × signal prix, temps de réalisation, backtests, HMM"),
    ("07_seconde_version_bloomberg", "Seconde version : 1999-2026, Bloomberg, résultats et explicabilité"),
    ("08_ameliorations_sans_surapprentissage", "Améliorations sans surapprentissage : protocole figé, jugé sur 2010-2023"),
    ("09_code_et_concepts", "Tout le code du projet, module par module, avec les concepts expliqués"),
]
# Un notebook rangé dans son propre dossier (avec ses scripts et ses figures) se télécharge avec tout son dossier.
DOSSIERS_NB = {"06_annexe_news_prix": "notebooks/06_annexe_news_prix"}
ARCHIVE_NOTEBOOKS = "notebooks_nikkei_225.zip"   # le dossier notebooks/ au complet
ARCHIVE_PROJET = "projet_nikkei_225.zip"         # tout ce qu'il faut pour refaire tourner le projet
# hors de l'archive du projet : le site lui-même, l'ancien site, les données brutes (lourdes, ou sous licence comme Bloomberg)
HORS_PROJET = ("site/", "site_statique/", "web/", "data/1_brutes/", "data/archives/", ".claude/", "lancer.php", "Ouvrir le site.html",
               "Lancer le projet.command")
DATE_FIXE = (2026, 1, 1, 0, 0, 0)               # archives identiques d'un export à l'autre si rien n'a changé
WIDGET = "application/vnd.jupyter.widget-view+json"
NOTE_WIDGET = ('<div style="padding:12px 14px;border:1px solid #d9d2c5;border-radius:4px;background:#f7f3ec;color:#1f1d1a;'
               'font:14px/1.5 system-ui,sans-serif"><b>Tableau de bord interactif (ipywidgets).</b> Ses curseurs ont besoin de Python : '
               'ils ne fonctionnent pas dans cette page. Téléchargez le dossier du notebook depuis le site, puis ouvrez-le dans Jupyter.</div>')

LISEZMOI_PROJET = """# Projet Nikkei 225 : version finale

Cette archive contient tout le projet, sauf le site de présentation et les données brutes :
- `notebooks/` : tous les notebooks, exécutés (le 06 avec ses scripts, ses figures et son README) ;
- `src/`, `app/`, `tests/` : le code, l'API et les tests ;
- `data/2_construites/`, `data/3_modele/` : les tables mensuelles et les tables de modélisation ;
- `runs/` : journaux des essais, modèles, décisions figées, évaluations et résultats des trois étapes ;
- `figures/`, `report/` : les figures et le rapport.

Les données brutes (`data/1_brutes/`) ne sont pas incluses : elles se retéléchargent avec les notebooks 00a et 00b, et les
exports Bloomberg ne peuvent pas être redistribués. Les notebooks se lisent tels quels, sorties comprises ; pour les
réexécuter, voir le README.md (installation, puis python -m pytest -q).
"""


def donnees_api() -> dict:
    warnings.filterwarnings("ignore")
    from fastapi.testclient import TestClient

    from app.api import app

    client = TestClient(app)
    sortie = {}
    for cle, route in ROUTES.items():
        reponse = client.get(route)
        reponse.raise_for_status()
        sortie[cle] = reponse.json()
    # l'explication de chaque décision du test, mois par mois (première version : contributions de la logistique)
    sortie["explications_test"] = {}
    for mois in [m["date"] for m in sortie["historique_test"]] + [sortie["signal"]["mois_des_donnees"]]:
        reponse = client.get(f"/explication?mois={mois}")
        if reponse.status_code == 200:
            sortie["explications_test"][mois] = reponse.json()
    return sortie


LIBELLES = {
    "news_attention_6m": "volume anormal d'articles", "news_sent_w": "tonalité des news", "news_dispo": "news disponibles",
    "n225_mom_12m": "momentum 12 mois", "n225_mom_3m": "momentum 3 mois", "n225_vol_ewma": "volatilité lissée",
    "d_jgb10y_bp": "variation du taux à 10 ans", "tankan_large_manuf": "Tankan", "carry_1m": "portage à un mois",
    "eurjpy_rvol_1m": "volatilité de l'euro-yen", "kmeans0": "distance au régime 1", "kmeans1": "distance au régime 2",
}


def modele_final() -> dict:
    """Coefficients de la logistique figée (parcours Maths : le prior gaussien et les poids appris)."""
    import joblib

    pipe = joblib.load(ROOT / "runs" / "models" / "nikkei_logit.joblib")
    reg = pipe.named_steps["model"]
    noms = [n.split("__")[-1] for n in pipe[:-1].get_feature_names_out()]
    return {"variables": noms, "libelles": [LIBELLES.get(n, n) for n in noms],
            "coefficients": [round(float(c), 5) for c in reg.coef_[0]], "intercept": round(float(reg.intercept_[0]), 5),
            "C": round(float(reg.C), 6), "solveur": reg.solver}


def versionner() -> None:
    """Chaque page appelle ses scripts et ses styles avec une empreinte de leur contenu (?v=…) : en ligne, un navigateur
    qui garde une ancienne copie en cache recharge le fichier dès qu'il change, sans jamais mélanger deux versions."""
    import hashlib

    for p in [SITE / page for page in ("index.html", "lecture.html", "cabinet.html")] + [SITE / "en" / page for page in ("index.html", "lecture.html", "cabinet.html")]:
        if not p.exists():
            continue

        def empreinte(m: re.Match, dossier: Path = p.parent) -> str:
            chemin = m.group(2)
            cible = dossier / chemin
            if not cible.exists():
                return m.group(0)
            return f'{m.group(1)}="{chemin}?v={hashlib.sha256(cible.read_bytes()).hexdigest()[:10]}"'
        html = p.read_text(encoding="utf-8")
        html = re.sub(r'(src|href)="((?:\.\./)?(?:assets|data)/[^"?]+\.(?:js|css))(?:\?v=[0-9a-f]+)?"', empreinte, html)
        p.write_text(html, encoding="utf-8")


def maths_cours() -> dict:
    """Parcours Maths : la log-loss d'entraînement du modèle figé, et l'option L2 de l'annexe (notebook 05, section 7) :
    les équations normales résolues à la main pour prédire le rendement du mois suivant, et le R² qu'elles donnent."""
    import joblib
    import numpy as np
    import pandas as pd
    import yaml
    from sklearn.linear_model import LinearRegression

    cfg = yaml.safe_load(open(ROOT / "config.yaml"))
    table = pd.read_csv(ROOT / cfg["data"]["model_table"], parse_dates=["date"]).set_index("date")
    features = [c for g in ["news", "price", "macro", "fx"] for c in cfg["features"][g]]
    colonnes = features + [c for c in cfg["features"]["regime"] if c not in features]
    train, valid = table[table["bloc"] == "entraînement"], table[table["bloc"] == "validation"]
    pipe = joblib.load(ROOT / "runs" / "models" / "nikkei_logit.joblib")
    prep = pipe[:-1]
    X, Xv = prep.transform(train[colonnes]), prep.transform(valid[colonnes])
    y = train["y_baisse"].astype(int).to_numpy()
    p = np.clip(pipe.predict_proba(train[colonnes])[:, 1], 1e-12, 1 - 1e-12)
    w = pipe.named_steps["model"].coef_[0]
    lam = 1 / (2 * len(y) * pipe.named_steps["model"].C)
    ll = float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
    Xt, Xvt = np.c_[X, np.ones(len(X))], np.c_[Xv, np.ones(len(Xv))]
    r, rv = train["target_rn_next"].to_numpy(), valid["target_rn_next"].to_numpy()
    beta = np.linalg.solve(Xt.T @ Xt, Xt.T @ r)
    ols = LinearRegression().fit(X, r)
    r2 = lambda vrai, prevu: float(1 - ((vrai - prevu) ** 2).sum() / ((vrai - vrai.mean()) ** 2).sum())
    pv = Xvt @ beta
    u = X @ w + pipe.named_steps["model"].intercept_[0]          # score wᵀx + b de chaque mois d'entraînement
    return {"logloss_entrainement": ll, "cout_penalise_entrainement": ll + lam * float(w @ w), "lambda": lam, "mois_entrainement": int(len(y)),
            "scores_entrainement": [[d.strftime("%Y-%m"), round(float(a), 4), int(b)] for d, a, b in zip(train.index, u, y)],
            "ols": {"r2_entrainement": r2(r, Xt @ beta), "r2_validation": r2(rv, pv),
                    "ecart_scikit_learn": float(np.abs(beta - np.r_[ols.coef_, ols.intercept_]).max()),
                    "variables": int(X.shape[1]), "mois_validation": int(len(rv)),
                    "entrainement": [[d.strftime("%Y-%m"), round(float(a), 5), round(float(b), 5)] for d, a, b in zip(train.index, Xt @ beta, r)],
                    "validation": [[d.strftime("%Y-%m"), round(float(a), 5), round(float(b), 5)] for d, a, b in zip(valid.index, pv, rv)]}}


# ---------- paquets : un script classique par page, pour ouvrir le site d'un double-clic ----------
# Les navigateurs refusent les modules JavaScript ouverts depuis le disque (file://). On assemble donc, pour
# chaque page, Three.js et nos modules en un seul script classique : chaque module devient une fonction qui
# renvoie ses exports, et chaque import devient une lecture de ces exports.
PAQUETS = {
    "visite": ["contenu", "commun", "oeuvres", "visite"],
    "lecture": ["contenu", "commun", "lecture"],
    "cabinet": ["contenu", "commun", "cabinet"],
}


def _three() -> str:
    code = (SITE / "vendor" / "three.module.min.js").read_text(encoding="utf-8")
    export = re.search(r"export\s*\{([^}]*)\}\s*;?\s*$", code)
    paires = [p.strip().split(" as ") for p in export.group(1).split(",") if p.strip()]
    retour = ",".join(f"{(p[1] if len(p) == 2 else p[0]).strip()}:{p[0].strip()}" for p in paires)
    return f"const __three = (() => {{\n\"use strict\";\n{code[:export.start()]}\nreturn {{{retour}}};\n}})();\n"


def _module(nom: str, dossier: Path = SITE / "assets") -> str:
    source = dossier / f"{nom}.js"
    if not source.exists():                      # version anglaise : un module sans texte (oeuvres.js) reste commun
        source = SITE / "assets" / f"{nom}.js"
    code = source.read_text(encoding="utf-8")
    exports = []

    def importer(m: re.Match) -> str:
        quoi, source = m.group(1).strip(), Path(m.group(2)).stem
        if source.startswith("three"):
            return f"const {quoi.split(' as ')[1].strip()} = __three;"
        return f"const {quoi} = __{source};"

    code = re.sub(r'^import\s+(.+?)\s+from\s+"([^"]+)";\s*$', importer, code, flags=re.M)

    def exporter(m: re.Match) -> str:
        exports.append(m.group(2))
        return f"{m.group(1)} {m.group(2)}"

    code = re.sub(r"^export\s+(const|let|function|class)\s+([A-Za-z_$][\w$]*)", exporter, code, flags=re.M)
    if re.search(r"^\s*(import|export)\b", code, flags=re.M):
        raise SystemExit(f"assets/{nom}.js : import ou export non pris en charge par l'assemblage")
    return f"const __{nom} = (() => {{\n\"use strict\";\n{code}\nreturn {{ {', '.join(exports)} }};\n}})();\n"


def paquets() -> None:
    # le site français (site/assets) et sa copie anglaise (site/en/assets, mêmes données, textes traduits)
    for dossier in (SITE / "assets", SITE / "en" / "assets"):
        if not dossier.exists():
            continue
        for page, modules in PAQUETS.items():
            morceaux = [f"// Fichier écrit par python -m app.export_site à partir de {dossier.relative_to(SITE)}/*.js : ne pas modifier à la main.\n"]
            if any(m in ("oeuvres", "visite") for m in modules):
                morceaux.append(_three())
            morceaux += [_module(m, dossier) for m in modules]
            sortie = dossier / f"{page}.bundle.js"
            sortie.write_text("".join(morceaux), encoding="utf-8")
            print(f"site/{sortie.relative_to(SITE)} : {sortie.stat().st_size / 1e3:.0f} ko")


# ---------- seconde version (Bloomberg, 1999-2026) et description du dataset ----------
def version_2() -> dict:
    """Résultats de src.v2_bloomberg et table 1999-2026, s'ils existent."""
    import pandas as pd

    dossier = ROOT / "runs" / "v2_bloomberg"
    sortie = {}
    for nom in ["evaluation_test", "decision_finale", "controle_bloomberg", "shap_test"]:
        f = dossier / f"{nom}.json"
        if f.exists():
            sortie[nom] = json.loads(f.read_text())
    table = ROOT / "data" / "3_modele" / "dataset_model2.csv"
    if table.exists():
        t = pd.read_csv(table, parse_dates=["date"])
        sortie["serie"] = [{"mois": d.strftime("%Y-%m"), "cloture": round(float(c), 1), "bloc": b}
                           for d, c, b in zip(t["date"], t["n225_close"], t["bloc"])]
    journal = dossier / "trials_log.csv"
    if journal.exists():
        j = pd.read_csv(journal)
        j = j[j["cible"] == "nikkei"]
        sortie["essais"] = [{"modele": m, "essai": int(e), "logloss": float(l), "retenu": bool(r)}
                            for m, e, l, r in zip(j["modele"], j["essai"], j["cv_logloss"], j["retenu"])]
    return sortie


def version_3() -> dict:
    """Résultats de src.ameliorations (protocole figé avant le calcul), s'ils existent."""
    dossier = ROOT / "runs" / "v3_ameliorations"
    if not (dossier / "resultats.json").exists():
        return {}
    r = json.loads((dossier / "resultats.json").read_text())
    fige = json.loads((dossier / "protocole.json").read_text())
    garder = ["protocole", "premiere_prevision", "mois_par_periode", "tests_probabilites", "tests_euros", "retenus",
              "assemblage_final", "mois_prochain"]
    sortie = {k: r[k] for k in garder}
    sortie["probabilites"] = {per: {v: {k: b[k] for k in ["log_loss", "ecart_vs_constante", "erreur_type", "auc"]} for v, b in d.items()}
                              for per, d in r["probabilites"].items()}
    sortie["euros"] = {per: {s: {k: b[k] for k in ["sharpe", "cumul", "volatilite", "perte_maximale"]} for s, b in d.items()}
                       for per, d in r["euros"].items()}
    import pandas as pd

    p = pd.read_csv(dossier / "probabilites.csv", parse_dates=["date"]).set_index("date").loc["2010":"2023"]
    sortie["actions_2010_2023"] = {c.split(" · ", 1)[1]: {k: int(v) for k, v in p[c].value_counts().items()}
                                   for c in p.columns if c.startswith("action · ")}
    sortie["descriptions"] = fige["protocole"]["probabilites"] | {k: v for k, v in fige["protocole"]["leviers_euros"].items()
                                                                  if not k.startswith("règle")}
    return sortie


def dataset() -> dict:
    """Ce que contient la table 1999-2026 : taille, sources, décalages de publication."""
    import pandas as pd

    dico = ROOT / "data" / "2_construites" / "dictionnaire_variables2.csv"
    mensuel = ROOT / "data" / "2_construites" / "dataset_monthly2.csv"
    if not (dico.exists() and mensuel.exists()):
        return {}
    d = pd.read_csv(dico)
    m = d[d["fichier"] == "dataset_monthly2.csv"]
    def famille(source: str, colonne: str) -> str:
        if colonne.startswith(("news_", "gdelt_")):
            return "News (GDELT, FinBERT)"
        if colonne.startswith("cpi_"):
            return "Inflation (OCDE, e-Stat)"
        if colonne.startswith("ff_"):
            return "Facteurs Fama-French"
        if colonne == "nikkei_vi":
            return "Volatilité implicite (Nikkei VI)"
        for cle, nom in [("Yahoo", "Yahoo Finance"), ("MoF", "Ministère des Finances"), ("BoJ", "Banque du Japon"), ("OCDE", "OCDE"),
                         ("BCE", "BCE et Bundesbank"), ("Bundesbank", "BCE et Bundesbank"), ("policy", "Incertitude et géopolitique"),
                         ("Caldara", "Incertitude et géopolitique")]:
            if cle in source:
                return nom
        return "Calendrier et dérivées"
    tab = pd.read_csv(mensuel, usecols=["date"], parse_dates=["date"])
    return {"colonnes": int(len(m)), "mois": int(len(tab)), "premier_mois": tab["date"].min().strftime("%Y-%m"),
            "dernier_mois": tab["date"].max().strftime("%Y-%m"),
            "variables_journalieres": int((d["fichier"] == "dataset_daily2.csv").sum()),
            "sources": pd.Series([famille(src, col) for src, col in zip(m["source"].fillna(""), m["colonne"])]).value_counts().to_dict(),
            "decalages": m["decalage"].fillna("").value_counts().to_dict()}


def images_integrees() -> None:
    """Les estampes de la rotonde, en data URI : la 3D les lit même quand le site est ouvert depuis le disque."""
    import base64

    images = {f.stem: "data:image/jpeg;base64," + base64.b64encode(f.read_bytes()).decode()
              for f in sorted((SITE / "images").glob("*.jpg"))}
    (SITE / "data" / "images.js").write_text("// Fichier écrit par python -m app.export_site : ne pas modifier à la main.\n"
                                             f"window.NIKKEI_IMAGES = {json.dumps(images)};\n", encoding="utf-8")
    print(f"site/data/images.js : {len(images)} images")


def ipynb(nom: str) -> Path:
    dossier = DOSSIERS_NB.get(nom)
    return ROOT / (dossier or "notebooks") / f"{nom}.ipynb"


def fichiers_du_depot(prefixe: str = "") -> list[str]:
    """Fichiers suivis par Git, ou nouveaux et non ignorés (le .gitignore écarte caches et données lourdes)."""
    sortie = subprocess.run(["git", "ls-files", "--cached", "--others", "--exclude-standard", "-z", "--", prefixe or "."],
                            cwd=ROOT, capture_output=True, check=True)
    noms = {n for n in sortie.stdout.decode().split("\0") if n}
    return sorted(n for n in noms if (ROOT / n).is_file() and not n.endswith(".DS_Store"))


def ecrire_zip(cible: Path, fichiers: list[tuple[str, Path | str]]) -> int:
    """Archive déterministe : (nom dans l'archive, fichier ou texte), dates fixées, ordre trié."""
    cible.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(cible, "w", zipfile.ZIP_DEFLATED) as z:
        for nom, contenu in sorted(fichiers, key=lambda f: f[0]):
            info = zipfile.ZipInfo(nom, date_time=DATE_FIXE)
            info.compress_type, info.external_attr = zipfile.ZIP_DEFLATED, 0o644 << 16
            z.writestr(info, contenu.read_bytes() if isinstance(contenu, Path) else contenu.encode("utf-8"))
    return len(fichiers)


# avertissements de bibliothèques sans rapport avec les résultats (tqdm, torch, barres de chargement)
BRUIT = re.compile(r"TqdmWarning|IProgress not found|autonotebook import tqdm|Redirects are currently not supported|it/s\]|^\s*$")


def pour_affichage(nb) -> None:
    """Version HTML seulement (le .ipynb reste intact) : un tableau de bord ipywidgets, qui ne tourne pas dans une page
    statique, est remplacé par une note ; les avertissements techniques des bibliothèques sont retirés."""
    import nbformat

    for cellule in nb.cells:
        if "outputs" not in cellule:
            continue
        sorties, note = [], False
        for o in cellule.outputs:
            if o.get("output_type") == "stream" and o.get("name") == "stderr" and all(BRUIT.search(l) for l in o.get("text", "").splitlines() or [""]):
                continue
            if WIDGET in o.get("data", {}):
                if not note:
                    sorties.append(nbformat.v4.new_output("display_data", data={"text/html": NOTE_WIDGET,
                                                                                "text/plain": "Tableau de bord interactif (Jupyter)"}))
                    note = True
                continue
            sorties.append(o)
        cellule.outputs = sorties
    nb.metadata.pop("widgets", None)


def notebook_html(nom: str) -> Path:
    """La page HTML d'un notebook : celle de web/nb, ou une conversion faite ici si elle manque."""
    source = ROOT / "web" / "nb" / f"{nom}.html"
    if source.exists():
        return source
    import nbformat
    from nbconvert import HTMLExporter

    exporteur = HTMLExporter(template_name="lab")
    exporteur.embed_images = True
    nb = nbformat.read(ipynb(nom), as_version=4)
    pour_affichage(nb)
    titre = dict(NOTEBOOKS)[nom]
    html, _ = exporteur.from_notebook_node(nb, resources={"metadata": {"name": f"Notebook {nom.split('_')[0]} · {titre}"}})
    cible = SITE / "nb" / f"{nom}.html"
    cible.parent.mkdir(parents=True, exist_ok=True)
    cible.write_text(html, encoding="utf-8")
    return cible


def alleger(page: Path) -> None:
    """Les pages de notebooks ne doivent pas attendre un CDN pour s'afficher : require.js (inutile ici, aucun widget ni
    sortie JavaScript) est retiré, et MathJax (les formules) se charge en différé, après le texte."""
    html = page.read_text(encoding="utf-8")
    html = re.sub(r'<script[^>]*src="[^"]*require(?:\.min)?\.js"[^>]*>\s*</script>\s*', "", html)
    html = re.sub(r'<script(?![^>]*\bdefer\b)([^>]*src="[^"]*mathjax[^"]*")', r'<script defer\1', html)
    page.write_text(html, encoding="utf-8")


def copier(source: Path, destination: Path) -> int:
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    return destination.stat().st_size


def telechargements() -> dict:
    """Les .ipynb, le dossier du 06, et les deux archives : écrits dans site/nb, décrits pour le site."""
    nb = SITE / "nb"
    sortie = {"notebooks": {}, "archives": []}
    for nom, _ in NOTEBOOKS:
        if nom in DOSSIERS_NB:
            dossier = DOSSIERS_NB[nom]
            n = ecrire_zip(nb / f"{nom}.zip", [(f[len("notebooks/"):], ROOT / f) for f in fichiers_du_depot(dossier + "/")])
            (nb / f"{nom}.ipynb").unlink(missing_ok=True)
            sortie["notebooks"][nom] = {"href": f"nb/{nom}.zip", "fichier": f"{nom}.zip", "dossier": True, "fichiers": n}
        else:
            copier(ipynb(nom), nb / f"{nom}.ipynb")
            sortie["notebooks"][nom] = {"href": f"nb/{nom}.ipynb", "fichier": f"{nom}.ipynb", "dossier": False, "fichiers": 1}
    tous = fichiers_du_depot("notebooks/")
    projet = [f for f in fichiers_du_depot() if not f.startswith(HORS_PROJET)]
    for cle, nom, titre, fichiers in [
            ("notebooks", ARCHIVE_NOTEBOOKS, "Tous les notebooks", [(f, ROOT / f) for f in tous]),
            ("projet", ARCHIVE_PROJET, "Le projet complet", [(f"projet_nikkei_225/{f}", ROOT / f) for f in projet]
             + [("projet_nikkei_225/LISEZMOI.md", LISEZMOI_PROJET)])]:
        n = ecrire_zip(nb / nom, fichiers)
        sortie["archives"].append({"cle": cle, "href": f"nb/{nom}", "fichier": nom, "titre": titre, "fichiers": n,
                                   "octets": (nb / nom).stat().st_size})
    return sortie


def main() -> None:
    donnees = donnees_api()
    donnees["notebooks"] = [{"nom": nom, "titre": titre} for nom, titre in NOTEBOOKS]
    donnees["figures"] = sorted(p.name for p in (ROOT / "figures").glob("*.png"))
    donnees["modele"] = modele_final()
    donnees["v2"] = version_2()
    donnees["v3"] = version_3()
    donnees["maths"] = maths_cours()
    donnees["telechargements"] = telechargements()
    donnees["dataset"] = dataset()

    (SITE / "data").mkdir(parents=True, exist_ok=True)
    texte = json.dumps(donnees, ensure_ascii=False, separators=(",", ":"))
    (SITE / "data" / "donnees.js").write_text(
        "// Fichier écrit par python -m app.export_site : ne pas modifier à la main.\n"
        f"window.NIKKEI = {texte};\n", encoding="utf-8")
    print(f"site/data/donnees.js : {len(texte) / 1e3:.0f} ko")

    total = 0
    for png in (ROOT / "figures").glob("*.png"):
        total += copier(png, SITE / "figures" / png.name)
    print(f"site/figures : {len(donnees['figures'])} figures, {total / 1e6:.1f} Mo")

    total = 0
    for nom, _ in NOTEBOOKS:
        source = notebook_html(nom)
        cible = SITE / "nb" / f"{nom}.html"
        if source != cible:
            copier(source, cible)
        alleger(cible)
        total += cible.stat().st_size
    print(f"site/nb : {len(NOTEBOOKS)} notebooks en HTML ({total / 1e6:.1f} Mo), à télécharger en .ipynb ou en dossier ; "
          + " ; ".join(f"{a['fichier']} {a['fichiers']} fichiers, {a['octets'] / 1e6:.1f} Mo" for a in donnees["telechargements"]["archives"]))

    for pdf in ("rapport.pdf", "annexe_maths.pdf"):
        copier(ROOT / "report" / pdf, SITE / "docs" / pdf)
    print("site/docs : rapport.pdf, annexe_maths.pdf")

    vendeur = SITE / "vendor" / "three.module.min.js"
    if not vendeur.exists():
        copier(ROOT / "web" / "assets" / "three.module.min.js", vendeur)
    images_integrees()
    paquets()
    versionner()


if __name__ == "__main__":
    main()

In [ ]:
# ─── app/export_statique.py · 137 lignes ───

"""Fabrique une copie statique du site, publiable sans serveur (Python et PHP doivent tourner pendant l'export).

Les pages sont rendues par le serveur PHP, les réponses de l'API sont enregistrées en fichiers JSON, et un petit script
injecté en tête de chaque page renvoie les appels « api.php?... » vers ces fichiers. Seules les simulations « et si »
du prédicteur (réglages libres) ont besoin de l'API vivante : dans la copie, elles affichent un message.

Usage : python -m app.export_statique [--site http://localhost:8080] [--sortie site_statique]
"""
import argparse
import json
import re
import shutil
import urllib.parse
import urllib.request
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
WEB = ROOT / "web"
NOTEBOOKS = ["01_cadrage_eda_variables", "02_pipeline_modeles_optuna", "03_calibration_seuils", "03b_test_final",
             "04_explicabilite_diagnostics", "05_annexe_maths_logistique"]
SIMPLES = ["sante", "mois", "evaluation", "decision", "serie", "essais", "surface", "volatilite", "verifications", "signal_risque",
           "signal", "explication", "variables", "prediction"]
PAR_MOIS = ["signal", "explication", "variables", "prediction"]

# Redirige les appels à l'API vers les fichiers JSON de la copie
AIGUILLAGE = """<script>
(() => {
  const nomFichier = (u) => {
    const p = new URLSearchParams(u.split("?")[1] || ""), route = p.get("route"); p.delete("route");
    const cles = [...p.keys()].sort();
    if (cles.some((k) => !["mois", "bloc"].includes(k))) return null;              // simulation libre : pas de fichier
    return route + cles.map((k) => "__" + k + "-" + p.get(k)).join("");
  };
  const fetchOriginal = window.fetch.bind(window);
  let paquet = null;                                  // toutes les réponses de l'API, dans un seul fichier chargé une fois
  const donnees = () => (paquet ??= fetchOriginal("donnees.json").then((r) => r.json()));
  const reponse = (corps, statut) => new Response(JSON.stringify(corps), { status: statut, headers: { "Content-Type": "application/json" } });
  window.fetch = (entree, options) => {
    const u = typeof entree === "string" ? entree : entree.url;
    if (u && u.includes("api.php?")) {
      const f = nomFichier(u);
      if (!f) return Promise.resolve(reponse({ detail: "Simulation disponible seulement dans la version locale du site" }, 503));
      return donnees().then((d) => (f in d ? reponse(d[f], 200) : reponse({ detail: "donnée absente de la copie" }, 404)));
    }
    return fetchOriginal(entree, options);
  };
  for (const nom of ["replaceState", "pushState"]) {
    const original = history[nom].bind(history);
    history[nom] = (...a) => { try { original(...a); } catch (e) {} };
  }
})();
</script>
"""


def lire(url: str) -> bytes:
    with urllib.request.urlopen(url, timeout=60) as r:
        return r.read()


def nom_json(route: str, params: dict) -> str:
    return route + "".join(f"__{k}-{params[k]}" for k in sorted(params)) + ".json"


def relier(html: str) -> str:
    """Rend les liens absolus du site relatifs à la copie."""
    html = re.sub(r'href="/notebooks\?nb=([a-z0-9_]+)"', r'href="notebooks-\1.html"', html)
    html = html.replace('href="/notebooks"', 'href="notebooks.html"')
    html = re.sub(r'href="/recit([?#][^"]*)?"', lambda m: f'href="recit.html{m.group(1) or ""}"', html)
    html = html.replace('href="/"', 'href="accueil.html"')
    html = re.sub(r'(href|src)="/(assets|figures|nb)/', r'\1="\2/', html)
    html = re.sub(r'href="/(rapport|annexe_maths)\.pdf"', r'href="\1.pdf"', html)
    html = re.sub(r'<a href="/ipynb/[^"]*"[^>]*>[^<]*</a>', "", html)                # pas de .ipynb dans la copie
    html = re.sub(r'href="http://127\.0\.0\.1:\d+/docs"', 'href="recit.html#ressources"', html)
    return html


def sans_enveloppe(html: str) -> str:
    """Page principale d'un Artifact : sans doctype, html, head ni body (la plateforme les ajoute)."""
    html = re.sub(r"<!doctype[^>]*>", "", html, flags=re.I)
    html = re.sub(r"</?(html|head)[^>]*>", "", html, flags=re.I)
    html = re.sub(r'<meta (charset|name="viewport")[^>]*>', "", html, flags=re.I)
    corps = re.search(r"<body([^>]*)>", html, flags=re.I)
    if corps and corps.group(1).strip():
        attributs = corps.group(1)
        html = html.replace(corps.group(0), "") + f"<script>(() => {{ const t = document.createElement('template'); t.innerHTML = '<i{attributs}></i>'; for (const a of t.content.firstChild.attributes) document.body.setAttribute(a.name, a.value); }})();</script>"
    else:
        html = re.sub(r"<body[^>]*>", "", html, flags=re.I)
    return re.sub(r"</body>", "", html, flags=re.I)


def avec_aiguillage(html: str) -> str:
    return re.sub(r"(<head[^>]*>)", r"\1" + AIGUILLAGE.replace("\\", "\\\\"), html, count=1, flags=re.I) if re.search(r"<head", html, re.I) else AIGUILLAGE + html


def main(site: str, sortie: Path) -> None:
    if sortie.exists():
        shutil.rmtree(sortie)
    sortie.mkdir(parents=True)
    paquet = {}

    # 1. Les réponses de l'API
    mois = [m["mois"] for m in json.loads(lire(f"{site}/api.php?route=mois"))]
    demandes = [(r, {}) for r in SIMPLES] + [("historique", {"bloc": b}) for b in ["test", "validation"]]
    demandes += [(r, {"mois": m}) for r in PAR_MOIS for m in mois]
    for route, params in demandes:
        corps = lire(f"{site}/api.php?" + urllib.parse.urlencode({"route": route, **params}))
        paquet[nom_json(route, params)[:-5]] = json.loads(corps)
    (sortie / "donnees.json").write_text(json.dumps(paquet, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")

    # 2. Les pages
    pages = {"accueil.html": "/", "recit.html": "/recit", "notebooks.html": "/notebooks"}
    pages |= {f"notebooks-{nb}.html": f"/notebooks?nb={nb}" for nb in NOTEBOOKS}
    for fichier, chemin in pages.items():
        html = avec_aiguillage(relier(lire(site + chemin).decode("utf-8")))
        (sortie / fichier).write_text(html, encoding="utf-8")
    accueil = (sortie / "accueil.html").read_text(encoding="utf-8")
    (sortie / "index.html").write_text(sans_enveloppe(accueil), encoding="utf-8")

    # 3. Les fichiers fixes
    shutil.copytree(WEB / "assets", sortie / "assets")
    shutil.copytree(WEB / "nb", sortie / "nb")
    shutil.copytree(ROOT / "figures", sortie / "figures")
    for pdf in ["rapport.pdf", "annexe_maths.pdf"]:
        shutil.copy(ROOT / "report" / pdf, sortie / pdf)

    n = sum(1 for _ in sortie.rglob("*") if _.is_file())
    taille = sum(f.stat().st_size for f in sortie.rglob("*") if f.is_file()) / 1e6
    print(f"{sortie} : {n} fichiers, {taille:.1f} Mo ({len(demandes)} réponses de l'API)")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--site", default="http://localhost:8080")
    parser.add_argument("--sortie", default=str(ROOT / "site_statique"))
    args = parser.parse_args()
    main(args.site.rstrip("/"), Path(args.sortie))

In [ ]:
# ─── app/build_site.py · 35 lignes ───

"""Convertit les notebooks en pages HTML pour le site (web/nb/).

À relancer après toute modification d'un notebook : python -m app.build_site
"""
from pathlib import Path

import nbformat
from nbconvert import HTMLExporter

ROOT = Path(__file__).resolve().parents[1]
SORTIE = ROOT / "web" / "nb"

NOTEBOOKS = [
    ("01_cadrage_eda_variables", "Cadrage, audit des fuites, EDA, variables, split"),
    ("02_pipeline_modeles_optuna", "Pipeline, trois modèles, Optuna, journal des essais"),
    ("03_calibration_seuils", "Calibration, choix du modèle, règle de décision"),
    ("03b_test_final", "Évaluation unique sur le test 2024-2026"),
    ("04_explicabilite_diagnostics", "Slices, permutation, SHAP, PDP"),
    ("05_annexe_maths_logistique", "Annexe de Maths : la logistique à la main"),
]


def main() -> None:
    SORTIE.mkdir(parents=True, exist_ok=True)
    exporteur = HTMLExporter(template_name="lab")
    exporteur.embed_images = True
    for nom, _ in NOTEBOOKS:
        nb = nbformat.read(ROOT / "notebooks" / f"{nom}.ipynb", as_version=4)
        html, _ = exporteur.from_notebook_node(nb, resources={"metadata": {"name": nom}})
        (SORTIE / f"{nom}.html").write_text(html, encoding="utf-8")
        print(f"{nom}.html : {len(html) / 1e6:.1f} Mo")


if __name__ == "__main__":
    main()

---
## Ce qu'il faut retenir

> **1. Une seule idée tient tout le projet : ne jamais laisser le futur entrer dans le passé.**
> Décalages de publication, audit des fuites, découpage chronologique, pipeline, décision figée, test ouvert une fois, protocole figé pour la V3.
>
> **2. Le meilleur modèle est une logistique régularisée et calibrée.**
> Simple, explicable par SHAP, et pas battue par les arbres sur si peu de mois.
>
> **3. Le sens du Nikkei se prévoit à peine ; sa volatilité, nettement mieux.**